In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'issue_worker.py': '"""GS-Issue worker: one local LLM finds the issues of each claim and proposes edits (same prompt for every model).\n\n    INPUTS=layer_a.jsonl,d_edit.jsonl python -u issue_worker.py --job \'{"tag": "q3-4b", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1}\'\n\nRuns inside a Kaggle notebook built by tools/kaggle/build_nb.py with llm.py and multirun.py of experiments/AG-1.\nGreedy, thinking off. Input rows: {id, claim, evidence}. Output: out/<tag>__<input stem>.jsonl, one row per input row:\n{id, raw, parsed (or null)}; out/<tag>_summary.json. Env: LIMIT=n (first n rows of each input, smoke), MAX_TOKENS (900).\nThe prompt is the D-EDIT labelling guide (benchmark/gs_eval_v2/LABELING_GUIDE.md section 2), shortened; the taxonomy\nis benchmark/gs_issue/taxonomy.py (embedded next to this file).\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom pathlib import Path\n\nfrom taxonomy import CODES, KINDS\n\nSYSTEM = "Bạn là kiểm toán viên soát xét tuyên bố môi trường trong báo cáo của doanh nghiệp Việt Nam. Bạn chỉ trả lời bằng JSON."\nPROMPT = """Đọc TUYÊN BỐ (và BẰNG CHỨNG nếu có). Tìm các vấn đề khiến tuyên bố có thể gây hiểu nhầm, rồi đề xuất sửa.\n\nMã vấn đề (chỉ dùng các mã này):\n{codes}\n\nBốn loại sửa được phép:\n{kinds}\n\nQuy tắc:\n- Chỉ nêu vấn đề có thật trong câu; câu đã đủ chỉ số, số, kỳ, năm gốc, phạm vi, nguồn và không nói quá thì "issues": [].\n- Câu không phải tuyên bố môi trường (tiêu đề, mục lục, câu tài chính, định nghĩa) thì "is_green_claim": false và không sửa.\n- BẰNG CHỨNG chỉ để hiểu bối cảnh (ví dụ: tổng phát thải thật ra tăng, số của công ty mẹ khác số hợp nhất, mục tiêu chưa đạt).\n- Không thêm bất kỳ con số nào không có trong tuyên bố. Thông tin chưa biết thì viết chỗ trống "[bổ sung: ...]".\n- Không đổi nghĩa, không biến mục tiêu thành thành tích, không xoá thông tin đúng.\n- "span" và "target_span" chép NGUYÊN VĂN một đoạn của tuyên bố.\n\nTUYÊN BỐ: {claim}\nBẰNG CHỨNG: {evidence}\n\nTrả lời đúng một đối tượng JSON:\n{{"is_green_claim": true, "issues": [{{"code": "...", "span": "..."}}], "edits": [{{"kind": "...", "rule_code": "...", "target_span": "...", "replacement": "..."}}], "rewritten_claim": "tuyên bố sau khi áp mọi sửa"}}"""\n\n\ndef parse(text):\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(text)):\n            ch = text[i]\n            if in_str:\n                esc = (ch == "\\\\" and not esc)\n                if ch == \'"\' and not esc:\n                    in_str = False\n                continue\n            if ch == \'"\':\n                in_str = True\n            depth += {"{": 1, "}": -1}.get(ch, 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    inputs = [p for p in os.environ.get("INPUTS", "").split(",") if p]\n    rows = []\n    for path in inputs:\n        part = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]\n        if os.environ.get("LIMIT"):\n            part = part[: int(os.environ["LIMIT"])]\n        rows += [(Path(path).stem, r) for r in part]\n    print(f"[{tag}] {len(rows)} rows from {inputs}, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    codes = "\\n".join(f"- {k}: {v}" for k, v in CODES.items())\n    kinds = "\\n".join(f"- {k}: {v}" for k, v in KINDS.items())\n    prompts = [[{"role": "system", "content": SYSTEM},\n                {"role": "user", "content": PROMPT.format(codes=codes, kinds=kinds, claim=r["claim"],\n                                                          evidence=(r.get("evidence") or "(không có)")[:900])}] for _, r in rows]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "900")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)} ({time.time() - t1:.0f}s)", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    bad, files = 0, {}\n    for (stem, r), text in zip(rows, raw):\n        obj = parse(text)\n        bad += obj is None\n        files.setdefault(stem, []).append({"id": r["id"], "raw": text, "parsed": obj})\n    for stem, out in files.items():\n        with open(f"out/{tag}__{stem}.jsonl", "w", encoding="utf-8") as fh:\n            for o in out:\n                fh.write(json.dumps(o, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "rows": len(rows), "unparsed": bad, "load_s": round(load_s), "gen_s": round(gen_s),\n               "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None), "overflow": getattr(m, "overflow", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'taxonomy.py': '"""GS-Issue taxonomy: issue codes, families, edit kinds, and the fix oracle (shared by generator, scorer and worker).\n\nThe codes are those of benchmark/gs_eval_v2/LABELING_GUIDE.md section 2 (D-EDIT), unchanged.\nA family groups codes the guide itself lets overlap (MISSING_SCOPE says "công ty mẹ hay tập đoàn",\nENTITY_SCOPE_AMBIGUOUS says the same): scoring reports strict codes and families.\n"""\nimport re\nimport unicodedata\n\nCODES = {\n    "MISSING_PERIOD": "Thiếu kỳ báo cáo / năm của số liệu",\n    "MISSING_BASELINE": "Tuyên bố thay đổi (giảm, tăng, tiết kiệm) không có năm gốc",\n    "MISSING_SCOPE": "Thiếu ranh giới (nhà máy nào, công ty mẹ hay tập đoàn, phạm vi phát thải 1/2/3)",\n    "NO_FIGURE": "Tuyên bố hiệu quả môi trường mà không có số liệu",\n    "VAGUE_TERM": "Từ mơ hồ hoặc quảng cáo: xanh, thân thiện môi trường, bền vững, không ngừng",\n    "ABSOLUTE_CLAIM": "Tuyệt đối hoá: 100 %, hoàn toàn, không phát thải, tốt nhất",\n    "OFFSET_NEUTRALITY": "Trung hòa / net zero dựa vào bù trừ, không nêu tỷ lệ bù trừ",\n    "FUTURE_AS_ACHIEVED": "Cam kết / mục tiêu diễn đạt như đã đạt",\n    "UNVERIFIED_LABEL_OR_CERT": "Nhắc nhãn, chứng nhận, giải thưởng mà không nêu tổ chức cấp, phạm vi, hiệu lực",\n    "NO_EVIDENCE_REF": "Tuyên bố định lượng không dẫn nguồn / phương pháp",\n    "HIDDEN_TRADEOFF": "Nêu một mặt tốt, che mặt xấu lớn hơn liên quan trực tiếp",\n    "IRRELEVANT_CLAIM": "Khoe điều hiển nhiên hoặc bắt buộc theo luật như một thành tích",\n    "INTENSITY_VS_ABSOLUTE": "Dùng cường độ giảm để ngụ ý tổng phát thải giảm",\n    "ENTITY_SCOPE_AMBIGUOUS": "Không rõ số của công ty mẹ hay tập đoàn",\n}\nFAMILY = {"MISSING_SCOPE": "SCOPE", "ENTITY_SCOPE_AMBIGUOUS": "SCOPE",\n          "VAGUE_TERM": "VAGUE", "NO_FIGURE": "VAGUE"}\n\n\ndef family(code: str) -> str:\n    return FAMILY.get(code, code)\n\n\nKINDS = {\n    "add_scope": "thêm kỳ, năm gốc, ranh giới, phạm vi còn thiếu, dạng chỗ trống [bổ sung: ...]",\n    "placeholder": "thay cụm mơ hồ / câu không có số bằng chỗ trống [bổ sung: chỉ số và giá trị]",\n    "evidence_ref": "thêm dẫn chiếu nguồn: [bổ sung: nguồn xác minh]",\n    "soften": "bỏ hoặc làm nhẹ lời nói quá, nói lại đúng mức",\n}\n# The kind the guide prescribes for each code (several are acceptable for some).\nKIND_OK = {\n    "MISSING_PERIOD": {"add_scope"}, "MISSING_BASELINE": {"add_scope"}, "MISSING_SCOPE": {"add_scope"},\n    "ENTITY_SCOPE_AMBIGUOUS": {"add_scope"}, "NO_FIGURE": {"placeholder"}, "VAGUE_TERM": {"placeholder", "soften"},\n    "ABSOLUTE_CLAIM": {"soften"}, "OFFSET_NEUTRALITY": {"soften"}, "FUTURE_AS_ACHIEVED": {"soften"},\n    "UNVERIFIED_LABEL_OR_CERT": {"add_scope", "evidence_ref"}, "NO_EVIDENCE_REF": {"evidence_ref"},\n    "HIDDEN_TRADEOFF": {"add_scope", "evidence_ref", "soften"}, "IRRELEVANT_CLAIM": {"soften"},\n    "INTENSITY_VS_ABSOLUTE": {"soften", "add_scope"},\n}\n\nPH = re.compile(r"[\\[(][^\\])]*(?:bổ sung|cần nêu|cần bổ sung|cần ghi)[^\\])]*[\\])]", re.I)\n# What a placeholder must mention to fill each missing slot.\nSLOT_WORDS = {\n    "period": r"kỳ|năm(?! gốc)|thời gian|giai đoạn",\n    "baseline": r"gốc|so sánh|mốc",\n    "scope": r"phạm vi|ranh giới|nhà máy|cơ sở|công ty mẹ|hợp nhất|tập đoàn|đơn vị",\n    "figure": r"số|giá trị|chỉ số|mức|định lượng",\n    "source": r"nguồn|trang|tài liệu|báo cáo|phương pháp|xác minh|kiểm toán|bảo đảm",\n    "issuer": r"tổ chức|cấp|hiệu lực|phạm vi|đơn vị",\n    "context": r"tổng|toàn bộ|diễn biến|bối cảnh|cùng kỳ|tuyệt đối|phát thải",\n    "offset": r"tỷ lệ|bù trừ|tín chỉ|sổ đăng ký",\n}\n\n\ndef fold(s: str) -> str:\n    s = unicodedata.normalize("NFC", s or "").lower()\n    return re.sub(r"\\s+", " ", s).strip()\n\n\nNUM = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef numbers(s: str) -> set[str]:\n    return {re.sub(r"[.,]", "", x) for x in NUM.findall(s or "")}\n\n\ndef placeholders(text: str) -> list[str]:\n    return [fold(m.group(0)) for m in PH.finditer(text or "")]\n\n\ndef slot_filled(before: str, after: str, slot: str) -> bool:\n    """A new placeholder (not in the text before) names the slot."""\n    old = placeholders(before)\n    new = [p for p in placeholders(after) if p not in old]\n    return any(re.search(SLOT_WORDS[slot], p) for p in new)\n\n\ndef oracle(case: dict, before: str, after: str) -> dict:\n    """Did the edited text fix this case\'s planted issue, and keep everything else?\n\n    resolves: the slot is asked for (drop mutations) or the planted phrase is gone or softened (inject mutations).\n    keeps_numbers: every number of the text before is still there. new_numbers: numbers the edit invented.\n    """\n    o = case["oracle"]\n    a = fold(after)\n    if o["type"] == "all":  # several planted issues: every one must be fixed\n        parts = [oracle(dict(case, oracle=p), before, after) for p in o["parts"]]\n        return {"resolves": all(p["resolves"] for p in parts), "keeps_numbers": all(p["keeps_numbers"] for p in parts),\n                "new_numbers": sorted(set().union(*(p["new_numbers"] for p in parts)))}\n    if o["type"] == "slot":\n        res = slot_filled(before, after, o["slot"])\n    elif o["type"] == "phrase":\n        gone = fold(o["phrase"]) not in a\n        asked = any(re.search(SLOT_WORDS[o.get("slot", "context")], p) for p in placeholders(after) if p not in placeholders(before))\n        res = gone or asked\n        if o.get("forbid"):  # e.g. FUTURE_AS_ACHIEVED: "đã giảm" must not remain\n            res = res and not re.search(o["forbid"], a)\n        if o.get("require"):\n            res = res and bool(re.search(o["require"], a))\n        if o.get("alt"):  # another acceptable fix, e.g. "theo quy định" for IRRELEVANT_CLAIM\n            res = res or bool(re.search(o["alt"], a))\n    else:\n        res = False\n    before_n, after_n = numbers(before), numbers(after)\n    droppable = numbers(o.get("phrase", ""))  # softening "100 %" may drop the 100\n    return {"resolves": bool(res), "keeps_numbers": (before_n - droppable) <= after_n,\n            "new_numbers": sorted(after_n - before_n)}\n',
    'jobs_32b.json': '[\n {\n  "tag": "q3-32b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-32B-AWQ"\n   }\n  ]\n }\n]',
    'gsi_layer_b2.jsonl': '{"id": "B-A-ffb744fc6840c4db", "claim": "cho phụ nữ, đồng thời đặt nền móng cho sự tiến bộ bền vững hơn trong các lĩnh", "evidence": "& Nhân viên 38,66 Sự khác biệt đáng kể về thời lượng đào tạo bình quân giữa lao động nam và nữ chủ yếu xuất phát từ các công ty thành viên trong lĩnh vực sản xuất thép, nơi lực lượng lao động phần lớn là nam giới và yêu cầu đào tạo cao do tính chất công việc mang tính kỹ thuật, vận hành thường xuyên và an toàn nghiêm ngặt. Ngược lại, lao động nữ thường tập trung nhiều hơn ở các bộ phận hành chính và hỗ trợ, được đào tạo tập trung về quản trị chung, nhóm chương trình có thời lượng ngắn hơn đào tạo kỹ thuật. 62 ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 3.1 Tự chủ 90% nhu cầu điện tại Khu liên hợp Gang thép Hòa Phát Dung Quất Hoàn thành kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho t"}\n{"id": "B-A-47013ddfc2b54248", "claim": "bảo tính kế thừa, ổn định và phát triển bền vững của đội ngũ quản lý;", "evidence": "hỏe và an toàn lao động. Thông qua các chương trình bảo hiểm và phúc lợi này, Tập đoàn Hòa Phát khẳng định cam kết xây dựng môi trường làm việc hỗ trợ và lành mạnh, đóng góp vào cả sức khỏe thể chất lẫn tinh thần của đội ngũ nhân viên. 55 CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi cam kết duy trì môi trường làm việc an toàn, tôn trọng và hòa nhập. Hòa Phát chú trọng phát triển và nuôi dưỡng đội ngũ nhân sự sáng tạo, sẵn sàng thích ứng và đổi mới, để xây dựng một doanh nghiệp nơi mọi nhân viên đều được trâ"}\n{"id": "B-A-69fc4ef8e3c4d1cb", "claim": "Đào tạo hướng dẫn sử dụng vaccine tai xanh, cúm gia cầm.", "evidence": "QUẢN TRỊ NGUỒN NHÂN LỰC 6.2 STT NỘI DUNG SỐ LƯỢNG CHƯƠNG TRÌNH ĐÀO TẠO ĐỐI TƯỢNG THAM DỰ Các chương trình Phát triển Năng lực Lãnh đạo Quản lý: 1. Quản trị tinh gọn LEAN 2. Ứng dụng Gen AI trong Quản trị và Lãnh đạo. 3. Nâng cao hiệu suất công việc với Gen AI. 4. Cập nhật các thay đổi về chính sách Thuế 2025. 5. Quy chế Quản lý tiền lương và thu nhập người lao động. 6. Giảng viên nội bộ. ... BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CBLĐ, CBQL toàn Tập đoàn 22 1 PHẦN 6 PHÁT TRIỂN CON NGƯỜI Đào tạo bồi dưỡng nâng cao năng lực chuyên môn và năng lực chung: 1. Hướng dẫn vận hành thiết bị ép phế liệu. 2. Hướng dẫn kiểm tra thiết bị thân lò trung tần. 3. Hướng dẫn lấy mẫu than cám. 4. Đào tạo vận hành trạ"}\n{"id": "B-A-24a656f27f173cad", "claim": "gắn bó và đóng góp lâu dài cho sự phát triển bền vững của Tập đoàn.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 4. GẮN KẾT CỘNG ĐỒNG Chúng tôi tạo cơ hội việc làm tại địa phương và đóng góp cho sự phát triển xã hội, môi trường của cộng đồng nơi Tập đoàn hoạt động. 1. NGUỒN CUNG ỨNG CÓ TRÁCH NHIỆM Chúng tôi ưu tiên hợp tác với các nhà cung cấp trong nước, đảm bảo chất lượng và thúc đẩy các thực hành bền vững ngay từ khâu đầu vào. 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT 7 1 5. KHÁCH HÀNG LÀ TRUNG TÂM, CHẤT LƯỢNG LÀ ƯU TIÊN HÀNG ĐẦU Chúng tôi đồng hành cùng khách hàng để tạo ra những giải pháp bền vững, sáng tạo và lâu dài, đáp ứng kỳ vọng ngày càng cao của xã hội. Do đó chất lượng sản phẩm luôn là ưu tiên hàng đầu của Tập đoàn"}\n{"id": "B-A-12774bf83df00fdf", "claim": "nguồn nhân lực, nhằm đảm bảo sự phát triển bền vững.", "evidence": "chính sách nhân sự, tập trung vào các lĩnh vực trọng điểm như hệ thống đánh giá nhân viên và hoạch định nguồn nhân lực, nhằm đảm bảo sự phát triển bền vững. Phát triển cộng đồng Hành trình số hóa quản trị nhân sự cũng sẽ được đẩy mạnh, với việc mở rộng thêm các phân hệ mới, hướng tới mục tiêu hình thành hệ sinh thái quản lý nhân sự số hóa toàn diện trên phạm vi toàn Tập đoàn. Trong năm 2025, giai đoạn tiếp theo sẽ tập trung vào số hóa các quy trình tuyển dụng, đào tạo và tích hợp trả lương, sau khi đã chuẩn hóa thống nhất tại các công ty thành viên. Bên cạnh đó, dựa trên kết quả tích cực đạt được, mô hình Quản trị tinh gọn dự kiến sẽ được mở rộng áp dụng sang các công ty thành viên khác tron"}\n{"id": "B-A-2bbfe88313fd08a3", "claim": "các phân hệ mới, hướng tới mục tiêu hình thành hệ sinh thái quản lý nhân sự số", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP"}\n{"id": "B-A-226368a0fa8f6b7c", "claim": "Trên hành trình phát triển bền vững, Tập đoàn Hòa Phát luôn coi việc tôn trọng và", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 4. GẮN KẾT CỘNG ĐỒNG Chúng tôi tạo cơ hội việc làm tại địa phương và đóng góp cho sự phát triển xã hội, môi trường của cộng đồng nơi Tập đoàn hoạt động. 1. NGUỒN CUNG ỨNG CÓ TRÁCH NHIỆM Chúng tôi ưu tiên hợp tác với các nhà cung cấp trong nước, đảm bảo chất lượng và thúc đẩy các thực hành bền vững ngay từ khâu đầu vào. 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT 7 1 5. KHÁCH HÀNG LÀ TRUNG TÂM, CHẤT LƯỢNG LÀ ƯU TIÊN HÀNG ĐẦU Chúng tôi đồng hành cùng khách hàng để tạo ra những giải pháp bền vững, sáng tạo và lâu dài, đáp ứng kỳ vọng ngày càng cao của xã hội. Do đó chất lượng sản phẩm luôn là ưu tiên hàng đầu của Tập đoàn"}\n{"id": "B-A-88843a87b36e27ee", "claim": "đến tăng trưởng bền vững và tạo giá trị chung cho các bên", "evidence": ", tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người 6. QUẢN TRỊ VỮNG MẠNH Với nền tảng quản trị minh bạch và trách nhiệm, Hòa Phát duy trì cơ chế giám sát và giải quyết khiếu nại hiệu quả, củng cố niềm tin của các bên liên quan. Giải thưởng và vinh danh năm 2025 lao động và đồng hành cùng 3 5 cộng đồng. 3. HIỆU QUẢ VÀ QUẢN LÝ TÀI NGUYÊN, THỰC HÀNH TIẾT KIỆM Chúng tôi nỗ lực hoạt động hiệu quả, tiết kiệm và có trách nhiệm, góp phần ứng phó biến đổi khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài nguyên chung, Biến chất thải sản xuất thành các nguyên liệu có giá trị cho các ngành khác, góp phần giảm khai thác tài nguyên thô. Thành viên của các Hiệp hội ngành nghề 4 Hoạt động kinh doa"}\n{"id": "B-A-66292b04de49db9e", "claim": "liên tục và tạo ra tác động bền vững, đặc biệt trong các lĩnh vực như chăm sóc trẻ", "evidence": "SỨC KHỎE VÀ AN TOÀN LAO ĐỘNG 6.1 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 6 PHÁT TRIỂN CON NGƯỜI CHƯƠNG TRÌNH CHĂM SÓC SỨC KHỎE VÀ PHÚC LỢI NGƯỜI LAO ĐỘNG ĐÀO TẠO VỀ SỨC KHỎE VÀ AN TOÀN LAO ĐỘNG Sức khoẻ và an toàn lao động Ở các công ty thành viên, đào tạo về sức khỏe và an toàn lao động là nền tảng quan trọng trong công tác phòng ngừa rủi ro và gắn kết người lao động. Các chương trình bao gồm: Quản trị nguồn nhân lực Chúng tôi nhận thức rằng phúc lợi của nhân viên không chỉ dừng lại ở vấn đề an toàn tại nơi làm việc. Do đó, Tập đoàn đã đầu tư vào nhiều chương trình bảo vệ sức khỏe và chăm sóc toàn diện tại các công ty thành viên. Ngoài các chế độ bảo hiểm xã hội bắt buộc, nhiều đơn vị còn tri"}\n{"id": "B-A-93772120a4625b66", "claim": "Hòa Phát nhận định lĩnh vực y tế và sức khỏe cộng đồng là nền tảng của một xã hội bền vững.", "evidence": "SỨC KHỎE VÀ AN TOÀN LAO ĐỘNG 6.1 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 6 PHÁT TRIỂN CON NGƯỜI CHƯƠNG TRÌNH CHĂM SÓC SỨC KHỎE VÀ PHÚC LỢI NGƯỜI LAO ĐỘNG ĐÀO TẠO VỀ SỨC KHỎE VÀ AN TOÀN LAO ĐỘNG Sức khoẻ và an toàn lao động Ở các công ty thành viên, đào tạo về sức khỏe và an toàn lao động là nền tảng quan trọng trong công tác phòng ngừa rủi ro và gắn kết người lao động. Các chương trình bao gồm: Quản trị nguồn nhân lực Chúng tôi nhận thức rằng phúc lợi của nhân viên không chỉ dừng lại ở vấn đề an toàn tại nơi làm việc. Do đó, Tập đoàn đã đầu tư vào nhiều chương trình bảo vệ sức khỏe và chăm sóc toàn diện tại các công ty thành viên. Ngoài các chế độ bảo hiểm xã hội bắt buộc, nhiều đơn vị còn tri"}\n{"id": "B-A-d68db1561c36c745", "claim": "đã trở thành một hoạt động thường niên bền vững, liên tục mở rộng phạm vi, tích cực tiếp cận và mang lại cơ hội sống khỏe", "evidence": "PHỤ LỤC A BẢNG CHỈ MỤC NỘI DUNG GRI TIÊU CHUẨN CHUNG NỘI DUNG CÓ NÊU TRONG BÁO MÃ CÔNG BỐ CHỦ ĐỀ TIÊU ĐỀ CÔNG BỐ GRI CÁO PHÁT TRIỂN PHẦN THAM CHIẾU/ LÝ DO LOẠI TRỪ GRI BỀN VỮNG NĂM 2025 GRI 2 – CÔNG BỐ THÔNG TIN THEO TIÊU CHUẨN CHUNG Tổ chức và các phương thức báo cáo của tổ chức 2-1 Thông tin chi tiết về Tổ chức Hoạt động kinh doanh tại tập đoàn Hòa Phát 2-2 Các Đơn vị được bao gồm trong Báo cáo Phát triển bền vững Phạm vi. thời gian và giới hạn báo cáo 2-3 Kỳ báo cáo. tần suất và thông tin liên hệ Phạm vi. thời gian và giới hạn báo cáo 2-4 Trình bày lại thông tin Đây là báo cáo công bố lần đầu tiên 2-5 Đảm bảo số liệu từ Đơn vị bên ngoài Số liệu về phát thải KNK của toàn tập đoàn đã được x"}\n{"id": "B-A-046459352b64380c", "claim": "Hòa Phát trong hành trình phát triển bền vững.", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-aee99997026a3a4f", "claim": "Hòa Phát luôn xem phát triển con người là nền tảng của phát triển bền vững.", "evidence": "n ở cấp độ Tập đoàn do Hội đồng Quản trị chỉ đạo trực tiếp. Công tác nghiên cứu chiều sâu được thực hiện tại cấp độ công ty con và phụ trách bởi Ban Giám đốc các công ty và các phòng ban liên quan, tập trung vào việc nghiên cứu phát triển sản phẩm, thường xuyên cải tiến công nghệ, kỹ thuật, nâng cao chất lượng, mở rộng các dải mác sản phẩm nhằm đáp ứng nhu cầu ngày càng đa dạng và khắt khe của khách hàng trong nước và quốc tế. ESG Hòa Phát tích hợp các mục tiêu phát triển bền vững (SDGs) vào chiến lược kinh doanh, phù hợp với định hướng phát triển xanh của Chính phủ Việt Nam và cam kết trung hòa carbon đến năm 2050. Chúng tôi tin rằng với tinh thần sáng tạo, quyết tâm và trách nhiệm, Hòa Phá"}\n{"id": "B-A-e42bafa88fdf1bf7", "claim": "Thông qua những chương trình trọng điểm, bền vững và giàu tính nhân văn, Hòa Phát", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 GRI 204, 411, 413] Với triết lý “Hòa hợp cùng phát triển”, Hòa Phát tích hợp mục tiêu phát triển kinh tế với trách nhiệm xã hội, hướng đến tăng trưởng bền vững và tạo giá trị chung cho các bên liên quan. Mọi hoạt động đều mang đến mục tiêu tạo ra giá trị lâu dài cho xã hội, lan tỏa tinh thần sẻ chia và đóng góp thiết thực cho đời sống người dân tại những địa phương nơi Hòa Phát hoạt động. CÁC YẾU TỐ CHÍNH TRONG PHƯƠNG PHÁP TIẾP CẬN CỦA HÒA PHÁT BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 HỖ TRỢ CÓ TRỌNG ĐIỂM: Các chương trình được thiết kế để tập trung giúp đỡ những nhóm đối tượng yếu thế, dễ bị tổn thương trong xã hội để nguồn lực được sử dụng hiệu quả và tạo ra tác động rõ rệ"}\n{"id": "B-A-5be6ce00cc9103e4", "claim": "Các khoản đầu tư xã hội năm 2025 thể hiện cam kết lâu dài của Hòa Phát trong việc tạo ra giá trị bền vững cho", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 4. GẮN KẾT CỘNG ĐỒNG Chúng tôi tạo cơ hội việc làm tại địa phương và đóng góp cho sự phát triển xã hội, môi trường của cộng đồng nơi Tập đoàn hoạt động. 1. NGUỒN CUNG ỨNG CÓ TRÁCH NHIỆM Chúng tôi ưu tiên hợp tác với các nhà cung cấp trong nước, đảm bảo chất lượng và thúc đẩy các thực hành bền vững ngay từ khâu đầu vào. 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT 7 1 5. KHÁCH HÀNG LÀ TRUNG TÂM, CHẤT LƯỢNG LÀ ƯU TIÊN HÀNG ĐẦU Chúng tôi đồng hành cùng khách hàng để tạo ra những giải pháp bền vững, sáng tạo và lâu dài, đáp ứng kỳ vọng ngày càng cao của xã hội. Do đó chất lượng sản phẩm luôn là ưu tiên hàng đầu của Tập đoàn"}\n{"id": "B-A-921f6477a47ae745", "claim": "nền tảng của sự phát triển bền vững, mà còn là yếu tố cốt lõi để củng cố niềm tin", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 TẦM NHÌN Trở thành Tập đoàn sản xuất công nghiệp với chất lượng dẫn đầu, trong đó Thép là lĩnh vực cốt lõi. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT SỨ MỆNH Cung cấp sản phẩm dẫn đầu, góp phần nâng cao chất lượng cuộc sống, đạt được sự tin yêu của khách hàng. Điểm nổi bật về phát triển bền vững năm 2025 GIÁ TRỊ CỐT LÕI Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công nhân viên, giữa Tập đoàn và đối tác, đại lý, cổ đông và cộng đồng xã hội, đảm bảo hài hòa lợi ích của các bên liên quan trên cùng một con thuyền, hướng tới sự phát triển "}\n{"id": "B-A-890dd19778e001f7", "claim": "tảng của quản trị bền vững, đóng vai trò then chốt trong việc kiểm soát rủi ro, bảo vệ", "evidence": "độc lập về công tác quản trị doanh nghiệp và kiểm soát rủi ro. • Chuẩn hóa quy trình và chính sách nhằm tăng cường kiểm soát nội bộ và quản lý rủi ro tại các công ty thành viên. • Tạo điều kiện để Ban Kiểm soát hoạt động hiệu quả, phù hợp với chức năng và nhiệm vụ được giao. • Tham mưu, đề xuất việc lựa chọn đơn vị kiểm toán có năng lực và uy tín cho Tập đoàn và các công ty con. BAN KIỂM SOÁT Tập đoàn Hòa Phát đã thành lập Ban Kiểm soát – cơ quan của Đại hội đồng cổ đông, đóng vai trò quan trọng trong việc đảm bảo tính minh bạch, trách nhiệm giải trình và quản trị lành mạnh trong toàn Tập đoàn. Các nhiệm vụ chính của Ban Kiểm soát bao gồm: 75 oàn xuất để tối ưu hóa vòng đời thứ cấp. Quá trìn"}\n{"id": "B-A-ad0817a9a19015b2", "claim": "uy tín doanh nghiệp và bảo đảm tuân thủ các tiêu chuẩn ESG trong bối cảnh ngành", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 CHUYỂN ĐỔI SỐ TRONG SẢN XUẤT VÀ QUẢN TRỊ DOANH NGHIỆP PHƯƠNG PHÁP QUẢN LÝ PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI E S G MÔI TRƯỜNG: Ứng dụng công nghệ số giúp giám sát và tối ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; đồng thời hỗ trợ cảnh báo sớm khi các chỉ số môi trường vượt ngưỡng và nâng cao tính minh bạch, tuân thủ trong quản lý phát thải. PHƯƠNG PHÁP QUẢN LÝ BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt kh"}\n{"id": "B-A-1907b4592ac7ecde", "claim": "đánh giá tổng thể các rủi ro ảnh hưởng đến hoạt động và phát triển bền vững", "evidence": "i ro – Tất cả các cấp quản lý và nhân viên đều đóng vai trò là chủ sở hữu rủi ro, có trách nhiệm xác định, đánh giá và đề xuất biện pháp ứng phó trong phạm vi công việc của mình. Các phòng ban định kỳ rà soát rủi ro tiềm ẩn, đánh giá theo hai yếu tố chính: - Khả năng xảy ra, và - Mức độ ảnh hưởng đến hoạt động kinh doanh cũng như các bên liên quan. Thông qua quá trình này, các đơn vị có thể ưu tiên nguồn lực để xử lý những rủi ro có tác động lớn nhất, từ đó bảo vệ hiệu quả hoạt động và uy tín của Tập đoàn. Những rủi ro được xếp loại “cao” hoặc “rất cao” liên quan đến chiến lược và kế hoạch kinh doanh sẽ được báo cáo lên Ban Tổng Giám đốc và Hội đồng Quản trị để xem xét và xử lý. Đối với các "}\n{"id": "B-A-022d32f19e6f7e97", "claim": "Rủi ro ESG: phát thải khí nhà kính, quy định CBAM (Cơ chế điều chỉnh biên giới", "evidence": "m, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám"}\n{"id": "B-A-d21a2502430cec3b", "claim": "Quản lý chất lượng là trọng tâm cho sự phát triển bền vững", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 [GRI 2-12, 2-24, 3-1, 3-2, 3-3] TRỞ THÀNH TẬP ĐOÀN SẢN XUẤT CÔNG NGHIỆP VỚI CHẤT LƯỢNG DẪN ĐẦU, TRONG ĐÓ THÉP LÀ LĨNH VỰC CỐT LÕI Tầm nhìn BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CUNG CẤP SẢN PHẨM DẪN ĐẦU, GÓP PHẦN NÂNG CAO CHẤT LƯỢNG CUỘC SỐNG, Sứ mệnh PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và th"}\n{"id": "B-A-8b204159630e229f", "claim": "lượng cao, an toàn, góp phần vào sự bền vững của các công trình xây dựng và cơ sở hạ tầng quốc gia.", "evidence": "PHÁT TRIỂN CỘNG ĐỒNG 6.4 HÒA PHÁT CÙNG EM ĐẾN TRƯỜNG. Được bắt đầu triển khai từ năm 2020, với mục tiêu tạo ra môi trường học tập tốt hơn cho trẻ em, đặc biệt là trẻ em vùng cao, vùng có điều kiện khó khăn, đây là hoạt động thường niên tập trung vào việc xây dựng, cải tạo cơ sở hạ tầng trường học, khu vui chơi, mua sắm trang thiết bị giáo dục, thiết bị bán trú... góp phần tạo dựng môi trường học tập khang trang, an toàn, chất lượng cho học sinh, sinh viên tại các địa phương. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Năm 2025, Hòa Phát khánh thành công trình trường tiểu học Bình Đông, thuộc xã Vạn Tường, tỉnh Quảng Ngãi. Công trình do Hòa Phát tài trợ cho địa phương theo hình thức chìa khóa trao tay v"}\n{"id": "B-A-32b55b9a38837e37", "claim": "Đối với bất động sản khu công nghiệp, chất lượng được đảm bảo qua việc phát triển hạ tầng kỹ thuật đồng bộ, hiện đại (điện, nước, xử lý chất thải) tạo môi trường", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 ĐẢM BẢO CHẤT LƯỢNG & AN TOÀN SẢN PHẨM LĨNH VỰC THÉP BÁO CÁO PHÁT TRIỂN BỀN VỮNG Việc đảm bảo chất lượng và an toàn sản phẩm là nguyên tắc cốt lõi, là nền tảng cho sự phát triển bền vững và uy tín của Tập đoàn Hòa Phát trên thị trường thép trong nước và quốc tế. Tất cả các sản phẩm nhóm ngành thép của Hòa Phát, từ thép thô đến các sản phẩm thép trung nguồn và hạ nguồn được Tập đoàn Hòa Phát thực hiện thông qua các khía cạnh: 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI #1: Đầu tư dây chuyền máy móc thiết bị hiện đại, chuyển giao công nghệ và đào tạo chuyên sâu Tập đoàn tập trung vào việc đầu tư các dây chuyền máy móc, thiết bị đồng bộ, hiện đại từ các đối tác hàng đầu thế giới, đảm "}\n{"id": "B-A-f940b322221cab8f", "claim": "mô hình khép kín với hệ thống xử lý chất thải đồng bộ và truy xuất nguồn gốc", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ KIỂM SOÁT CHẤT LƯỢNG KHÔNG KHÍ TRONG SẢN XUẤT THÉP Thực hiện theo tiêu chuẩn hệ thống quản lý môi trường ISO 14001, Tập đoàn Hòa Phát đặt mục tiêu giảm thiểu tác động không khí thông qua ứng dụng các công nghệ kiểm soát tiên tiến. Tại các nhà máy thép, hệ thống xử lý khí thải khép kín được đầu tư đồng bộ để kiểm soát chất lượng không khí từ đầu vào đến đầu ra của quá trình sản xuất, đảm bảo kiểm soát chặt chẽ các thông số về bụi, khí SOx, NOx và CO trước khi thải ra môi trường. Khí thải sau xử lý luôn được quan trắc tự động liên tục (CEMS) và truyền dữ liệu trực tiếp về Sở Nông nghiệp và Môi trường theo quy định pháp luật. Việc thu gom và"}\n{"id": "B-A-6e3c00c047801fd6", "claim": "kín, đảm bảo vệ sinh và có hệ thống xử lý chất thải đồng bộ.", "evidence": "CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải Bùn thải từ các trạm xử lý nước thải Phế liệu Xỉ thép Xỉ hạt lò cao CHẤT THẢI RẮN chất thải nguy hại, tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải. Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống tuần hoàn khép kín, tái chế nước và xử lý triệt để để tuân thủ quy chuẩn xả thải."}\n{"id": "B-A-5ee8205afb0534e1", "claim": "cải tiến chất lượng sản phẩm liên tục, góp phần phát triển bền vững và nâng", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi cam kết duy trì môi trường làm việc an toàn, tôn trọng và hòa nhập. Hòa Phát chú trọng phát triển và nuôi dưỡng đội ngũ nhân sự sáng tạo, sẵn sàng thích ứng và đổi mới, để xây dựng một doanh nghiệp nơi mọi nhân viên đều được trân trọng, ghi nhận và truyền cảm hứng. QUẢN TRỊ NGUỒN PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT NHÂN LỰC Chúng tôi cam kết đảm bảo minh bạch, trách nhiệm giải trình và liêm chính trong mọi cấp độ tổ chức. Hệ thống quản trị được thiết"}\n{"id": "B-A-863af66a7373e1f5", "claim": "khung gầm máy xúc/xe tải, tháp điện gió, kết cấu nhà xưởng khẩu độ lớn.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THU HỒI NHIỆT ĐỂ PHÁT ĐIỆN Từ năm 2025, Tập đoàn Hòa Phát chính thức đẩy mạnh lộ trình “xanh hóa” sản xuất thông qua việc triển khai hệ thống điện mặt trời áp mái tại các khu liên hợp và nhà máy trên toàn quốc, với tổng công suất mục tiêu lên đến hơn 130 MWp. Dự án có sự tham gia sâu rộng của các đơn vị nòng cốt như Khu liên hợp Gang thép Dung Quất và Hải Dương, hệ thống nhà máy Ống thép, Tôn Hòa Phát, các nhà máy sản xuất thức ăn chăn nuôi, điện lạnh và hệ thống trang trại chăn nuôi. Để hiện thực hóa mục tiêu này, hàng triệu tấm pin quang điện hiệu suất cao sẽ được lắp đặt trực tiếp trên hệ thống khung giàn thép kiên cố tại mái nhà "}\n{"id": "B-A-c04a9f6302590b2d", "claim": "trong việc cung cấp các giải pháp vật liệu bền vững.", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-2ee0a6abf989daf2", "claim": "năng lượng và thân thiện môi trường, nổi bật với công nghệ làm lạnh Block sử", "evidence": "hi xảy ra sự cố, đảm bảo môi trường làm việc an toàn cho toàn bộ người lao động. PHẦN 5 XANH HÓA SẢN XUẤT • TUÂN THỦ VÀ ĐẠO ĐỨC: Thực hiện nghiêm túc các quy định của pháp luật, quy chế nội bộ và quy tắc đạo đức kinh doanh của Công ty; đề cao trách nhiệm doanh nghiệp và nghĩa vụ xã hội. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường • CHẤT LƯỢNG HÀNG ĐẦU: Cung cấp sản phẩm có chất lượng cao, đáng tin cậy, đáp ứng kỳ vọng của khách hàng, đồng thời liên tục cải tiến chất lượng và năng lực cạnh tranh. • MÔI TRƯỜNG XANH VÀ HIỆU QUẢ NĂNG LƯỢNG: Giảm phát thải, tiết kiệm năng lượng, thúc đẩy tái chế và tái sử dụng thông qua việc áp dụng công nghệ sạch, thân thiện môi trư"}\n{"id": "B-A-c0c114329268a56f", "claim": "phát thải gián tiếp thông qua dữ liệu vận hành;", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "B-A-034c6bc00715b607", "claim": "cao tính minh bạch, tuân thủ trong quản lý phát thải.", "evidence": "hình thức hối lộ hoặc tác động không chính đáng giữa nhà thầu và nhân viên công ty nhằm mục đích trục lợi. Đảm bảo việc tuân thủ pháp luật, điều lệ công ty và TUÂN THỦ & ĐẢM BẢO các chính sách nội bộ thông qua hoạt động kiểm toán có hệ thống. Quản trị Doanh nghiệp và Quản lý rủi ro GIẢM THIỂU Xác định các điểm yếu trong hệ thống kiểm soát, giám sát tiến độ dự án và hoạt động mua sắm, đồng thời chủ RỦI RO Trong thời gian tới, Tập đoàn Hòa Phát đặt mục tiêu mở rộng việc áp dụng các thông lệ tốt này sang các công ty thành viên khác, qua đó củng cố cam kết về đạo đức kinh doanh, cạnh tranh lành mạnh và tuân thủ pháp luật, góp phần xây dựng môi trường kinh doanh minh bạch và có trách nhiệm. động "}\n{"id": "B-A-9e43e3777c07c6e9", "claim": "thải, tiêu chuẩn ESG ngày càng khắt khe và áp lực cạnh tranh", "evidence": "nhà xưởng. Năm 2025, tổng công suất đã lắp đặt đạt gần 20MWp, phần còn lại sẽ được tiếp tục triển khai mạnh mẽ trong năm 2026 và 2027. Sau khi đi vào vận hành ổn định, hệ thống dự kiến sẽ cung cấp khoảng 170 triệu kWh/năm, giúp đáp ứng 13% lượng điện mua ngoài hiện tại của Tập đoàn. Điểm ưu việt của mô hình này nằm ở việc năng lượng được sản xuất và tiêu thụ tại chỗ, loại bỏ hoàn toàn hao hụt trong quá trình truyền tải xa và tạo ra một lớp “lá chắn nhiệt” tự nhiên giúp hạ nhiệt độ mái tôn đáng kể, từ đó giảm điện năng tiêu thụ cho các thiết bị làm mát bên dưới. Việc tối ưu hóa diện tích mái xưởng sẵn có không chỉ giúp Hòa Phát chủ động nguồn điện, giảm tải cho lưới điện quốc gia mà còn là mi"}\n{"id": "B-A-ad36052045cc60c0", "claim": "trợ thực hiện các mục tiêu phát triển bền vững.", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-aa5c457105ad2856", "claim": "Giảm tiêu hao tài nguyên và phát thải gián tiếp", "evidence": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao. Đây là nền tảng cho lộ trình chuyển đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch. GIẢM TIÊU HAO THAN COKE Ở LÒ CAO ĐƠN VỊ THỰC HIỆN: Khu liên hợp Gang thép Hòa Phát Hải Dương và Dung Quất Giải pháp này bao gồm các hoạt động giúp tối ưu vận hành như cải thiện chất lượng phối liệu, kiểm soát kích cỡ than cốc, cải thiện phân bố liệu, tối ưu tốc độ gió nóng và"}\n{"id": "B-A-b28e4f0ce3cfb9a3", "claim": "Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG:", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 [GRI 2-12, 2-24, 3-1, 3-2, 3-3] TRỞ THÀNH TẬP ĐOÀN SẢN XUẤT CÔNG NGHIỆP VỚI CHẤT LƯỢNG DẪN ĐẦU, TRONG ĐÓ THÉP LÀ LĨNH VỰC CỐT LÕI Tầm nhìn BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CUNG CẤP SẢN PHẨM DẪN ĐẦU, GÓP PHẦN NÂNG CAO CHẤT LƯỢNG CUỘC SỐNG, Sứ mệnh PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và th"}\n{"id": "B-A-b05cda7f2061c49c", "claim": "nghiệm người dùng và từng bước hoàn thiện hệ sinh thái số tại công ty.", "evidence": "PHÁT TRIỂN ĐỔI MỚI SÁNG TẠO 7.3 KẾT QUẢ THỰC HIỆN NĂM 2025 • Phần mềm thống kê số liệu sản xuất: Trong năm 2025, phần mềm thống kê số liệu sản xuất đã được kết nối và thu thập dữ liệu trực tiếp từ các hệ thống khác, giúp số hóa toàn bộ biểu mẫu, đảm bảo dữ liệu sản xuất gang thép được cập nhật liên tục, phục vụ hiệu quả công tác theo dõi và báo cáo sản xuất hằng ngày của Công ty. Trong thời gian tới, dự án hướng tới thu thập dữ liệu sản xuất tự động hoàn toàn từ nhà máy và tổng hợp báo cáo sản xuất tập trung. Năm 2025 đánh dấu giai đoạn mở rộng triển khai và nâng cao hiệu quả sử dụng các hệ thống đã đầu tư, đồng thời từng bước hình thành nền tảng quản trị số toàn Tập đoàn. BÁO CÁO PHÁT TRIỂN"}\n{"id": "B-A-1ee97d497e1157f0", "claim": "Các Đơn vị được bao gồm trong Báo cáo Phát triển bền vững", "evidence": "1 GIỚI THIỆU BÁO CÁO PHÁT TRIỂN BỀN VỮNG KHUNG BÁO CÁO [GRI 1] Báo cáo Phát triển Bền vững năm 2025 của Tập đoàn Hòa Phát được xây dựng phù hợp với Bộ Tiêu chuẩn Báo cáo Phát triển Bền vững 2021 của Sáng kiến Báo cáo Toàn cầu (GRI), áp dụng cho giai đoạn từ ngày 01/01/2025 đến ngày 31/12/2025, với các thông tin công bố được tham chiếu trong bảng chỉ mục nội dung GRI. Báo cáo cũng phản ánh các tiêu chuẩn của Hội đồng Chuẩn mực Kế toán Bền vững (SASB) dành cho các nhà sản xuất sắt và thép. Báo cáo này được công bố định kỳ hằng năm. PHẠM VI VÀ GIỚI HẠN CỦA BÁO CÁO [GRI 2-2, 2-3] Hiệu quả phát triển bền vững của Tập đoàn Hòa Phát được trình bày trong báo cáo này bao quát toàn bộ hoạt động của Tậ"}\n{"id": "B-A-4c58c33a17f68064", "claim": "Số liệu về phát thải KNK của toàn tập đoàn đã được xác nhận bởi BSI.", "evidence": "ểm kê KNK đầy đủ cho các cơ quan chức năng từ năm 2024. PHẦN 5 XANH HÓA SẢN XUẤT PHƯƠNG PHÁP QUẢN LÝ Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên hợp Gang thép Hòa Phát Hải Dương và Hòa Phát Dung Quất — chiếm khoảng 94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải. • Phục vụ quản lý nội bộ: Ngoài đáp ứng yêu cầu của phá"}\n{"id": "B-A-512fa98dc262a42f", "claim": "Chuỗi Giá trị bền vững trong kinh doanh của chúng tôi", "evidence": ", tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người 6. QUẢN TRỊ VỮNG MẠNH Với nền tảng quản trị minh bạch và trách nhiệm, Hòa Phát duy trì cơ chế giám sát và giải quyết khiếu nại hiệu quả, củng cố niềm tin của các bên liên quan. Giải thưởng và vinh danh năm 2025 lao động và đồng hành cùng 3 5 cộng đồng. 3. HIỆU QUẢ VÀ QUẢN LÝ TÀI NGUYÊN, THỰC HÀNH TIẾT KIỆM Chúng tôi nỗ lực hoạt động hiệu quả, tiết kiệm và có trách nhiệm, góp phần ứng phó biến đổi khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài nguyên chung, Biến chất thải sản xuất thành các nguyên liệu có giá trị cho các ngành khác, góp phần giảm khai thác tài nguyên thô. Thành viên của các Hiệp hội ngành nghề 4 Hoạt động kinh doa"}\n{"id": "B-A-e3447bc7deb7035c", "claim": "Tuyên bố về chiến lược phát triển bền vững", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG • Định hướng chiến lược về Phát triển bền vững. • Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách nhiệm giám sát định hướng chiến lược cho các sáng kiến phát triển bền vững, bao gồm các vấn đề về môi trường, xã hội và quản trị (ESG). Việc triển khai và chịu trách nhiệm hàng ngày được thực hiện bởi Ban Tổng Giám đốc Tập đoàn và Ban Tổng Giám đốc "}\n{"id": "B-A-8fc04f3b82c85c0c", "claim": "Được trình bày theo các chủ đề trọng yếu về phát triển bền vững trong ba trụ cột chính", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 [GRI 2-12, 2-24, 3-1, 3-2, 3-3] TRỞ THÀNH TẬP ĐOÀN SẢN XUẤT CÔNG NGHIỆP VỚI CHẤT LƯỢNG DẪN ĐẦU, TRONG ĐÓ THÉP LÀ LĨNH VỰC CỐT LÕI Tầm nhìn BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 CUNG CẤP SẢN PHẨM DẪN ĐẦU, GÓP PHẦN NÂNG CAO CHẤT LƯỢNG CUỘC SỐNG, Sứ mệnh PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT ĐẠT ĐƯỢC SỰ TIN YÊU CỦA KHÁCH HÀNG Trụ cột XANH HÓA SẢN XUẤT PHÁT TRIỂN CON NGƯỜI QUẢN TRỊ VÀ ĐỔI MỚI Chuyển đổi quy trình sản xuất thép theo hướng ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển tích hợp ESG vào chiến lược, và th"}\n{"id": "B-A-9003a6587fbbd80b", "claim": "Mục tiêu và tiến độ giảm phát thải khí nhà kính", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-70c21b118f68e4fc", "claim": "Đây là báo cáo năm đầu tiên của chúng tôi, lộ trình giảm phát thải chi tiết vẫn đang được", "evidence": "ện năng tiêu thụ so với phương pháp truyền thống. • Thu hồi nhiệt từ quá trình thiêu kết, sử dụng nhiệt dư để phát điện, góp phần tối ưu hóa hiệu quả sử dụng năng lượng tổng thể. • Công nghệ đúc – cán liên tục cho phép nạp trực tiếp phôi nóng (750–900°C) vào dây chuyền cán. giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép. • Tận dụng các nguồn hơi nước từ hệ thống làm mát lò thổi, lò nung phôi để tận dụng phát điện. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu t"}\n{"id": "B-A-5e9c9e5ad48fb887", "claim": "Kết quả kiểm kê phát thải khí nhà kính năm 2025", "evidence": "quả kiểm kê phát thải khí nhà kính năm 2025 GRI 102 – Biến đổi khí hậu 2025 102-6 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025 102-8 Cường độ phát thải Do các nhà máy sử dụng công nghệ sản xuất thép khác nhau và có nhiều loại sản phẩm khác nhau, chúng tôi đang sắp xếp nguồn lực phù hợp để thu thập và tính toán chỉ tiêu này cho các kỳ báo cáo sau. HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607. Hoạt động kiểm kê KNK được thực hiện với 3 mục tiêu chính: [GRI 302-1, 302-2, 305-1, 30"}\n{"id": "B-A-41c8fffea1f74e48", "claim": "quả kiểm kê phát thải khí nhà kính năm 2025", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607. Hoạt động kiểm kê KNK được thực hiện với 3 mục tiêu chính: [GRI 302-1, 302-2, 305-1, 305-2, 305-4] Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero 2050 của Việt Nam tại COP26, Tập đoàn Hòa Phát xác định “Giảm thiểu và Thích ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng. Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm "}\n{"id": "B-A-0ce85756a109c9cf", "claim": "Phát sinh chất thải và các tác động đáng kể liên quan đến chất thải", "evidence": "g giảm thiểu và thích ứng với biến đổi khí hậu Điện máy gia dụng 104,6 57,8 55% Cộng 7.085 6.982 99% Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghìn tấn, trong đó chất thải nguy hại chiếm 3%, chất thải không nguy hại chiếm 97%. Quản lý môi trường Trong tổng số hơn 7 triệu tấn chất thải rắn phát sinh, có đến gần 6,98 triệu tấn đã được thu hồi và đưa quay trở lại chu trình sản xuất hoặc chuyển đổi mục đích sử dụng. Con số này cho thấy chất thải không còn là gánh nặng môi trường mà đã thực sự trở thành nguồn tài nguyên đầu vào quan trọng, giúp Tập đoàn tối ưu hóa chi phí và giảm thiểu tác động tiêu cực đến hệ sinh thái. 51 QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CH"}\n{"id": "B-A-2f630f4bf3c2fdaa", "claim": "Quản lý các tác động đáng kể liên quan đến chất thải", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn. Ngành công nghiệp thép, đặc biệt là tại các khu liên hợp sản xuất tích hợp từ quặng sắt là một trong những ngành phát sinh lượng chất thải rắn lớn. Tuy nhiên. với công nghệ hiện đại, phương thức quản lý chất thải rắn đã được thay đổi hoàn toàn. Chất thải r"}\n{"id": "B-A-cbbae25086fdbd92", "claim": "Chất thải được chuyển giao khỏi quy trình xử lý", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-02ae996f5eba2805", "claim": "BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-c0af9691869a3962", "claim": "Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e)", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-6535eec04a47e278", "claim": "305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1)", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-1c0c1f7d6fa6f02e", "claim": "Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e)", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-52b0a6670fa61122", "claim": "305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-4c690de9172eb8ad", "claim": "Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-61a41d15f82461ec", "claim": "Năng lượng tiêu thụ là năng lượng tái tạo tại các", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-5b7c9e406f15f044", "claim": "Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1)", "evidence": "PHỤ LỤC B BẢNG DỮ LIỆU ĐÁNH GIÁ HIỆU SUẤT ESG DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% N"}\n{"id": "B-A-c193ec133f6bf040", "claim": "Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương", "evidence": "PHỤ LỤC B BẢNG DỮ LIỆU ĐÁNH GIÁ HIỆU SUẤT ESG DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% N"}\n{"id": "B-A-48d94688ad50f1f8", "claim": "Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479", "evidence": "PHỤ LỤC B BẢNG DỮ LIỆU ĐÁNH GIÁ HIỆU SUẤT ESG DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% N"}\n{"id": "B-A-188432884cfe8b40", "claim": "Năng lượng tiêu thụ là năng lượng tái tạo tại các % 0,03%", "evidence": "GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo tại các GRI 3 – Chủ đề trọng yếu 2021 103-1 Chính sách và cam kết năng lượng Phương pháp quản lý 103-2 Tiêu thụ năng lượng và năng lượng tự tạo ra trong tổ chức Kết quả tiêu thụ năng lượng năm 2025 GRI 103 – Năng"}\n{"id": "B-A-27bf989a836cbfe8", "claim": "Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay", "evidence": "liệu nền đường. Quản lý môi trường LƯỢNG NƯỚC KHAI THÁC, HOÀN TRẢ VÀ SỬ DỤNG NĂM 2025 CHỈ TIÊU NĂM 2025 Năm 2025, Tập đoàn Hòa Phát đầu tư hạng mục xử lý bụi và bùn chứa kẽm với công nghệ lò đáy quay do CISDI Thermal & Environmental Engineering Co,. Ltd. (thành viên của MCC-China) cung cấp. Hệ thống có năng lực xử lý toàn bộ lượng bụi và bùn chứa kẽm phát sinh tại Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép và bột oxit kẽm là sản phẩm phụ có giá trị cao, có thể bán cho các nhà máy luyện kẽm, ứng dụng mạ kẽm hoặc ngành hóa chất. Hệ thống đồng"}\n{"id": "B-A-625548c32da203ff", "claim": "306-4 Chất thải được chuyển giao khỏi quy trình xử lý", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-de2b809fc9443aa4", "claim": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-f44dc77ca3c1a13a", "claim": "Chất thải tái sử dụng (chất thải nguy hại)", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-84ad5c908fe8829d", "claim": "Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích", "evidence": "hỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào "}\n{"id": "B-A-0199d05bebfd827c", "claim": "Chất thải tái sử dụng (chất thải không nguy hại)", "evidence": "Tổng lượng nước khai thác từ các bên thứ ba Triệu lít 1.275 303-3 Khai thác nước Tổng lượng nước khai thác từ tất cả các khu vực có áp lực về nước Triệu lít N/A 303-3 Khai thác nước Tổng lượng nước sử dụng từ tất cả các khu vực Triệu lít 41.334 303-5 Tiêu thụ nước Tổng lượng nước xả thải ra các khu vực Triệu lít 4.994 303-4 Xả thải nước Tổng lượng nước ngọt tiêu thụ tại các khu vực thiếu nước Triệu lít N/A Tổng lượng nước ngọt tiêu thụ tại các khu vực không thiếu nước Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát si"}\n{"id": "B-A-40baeca3249cbec3", "claim": "306-5 Chất thải được đưa vào quy trình xử lý", "evidence": "QUẢN LÝ MÔI TRƯỜNG GRI 103 – Năng lượng 2025 3-3 Quản lý chủ đề trọng yếu Phương pháp quản lý QUẢN LÝ CHẤT LƯỢNG KHÔNG KHÍ Kiểm soát chất lượng không khí Oxit nitơ (NOx). oxit lưu huỳnh (SOx) và các khí thải không khí đáng GRI 305- Phát thải 2016 305-7 kể khác QUẢN LÝ NƯỚC GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý nước 303-2 Quản lý tác động liên quan đến xả thải nước Quản lý nước GRI 303 – Nước và Nước thải 2018 303-3 Lượng nước khai thác Quản lý nước 303-4 Lượng nước thải Quản lý nước 303-5 Lượng nước tiêu thụ Quản lý nước QUẢN LÝ CHẤT THẢI RẮN GRI 3 – Chủ đề trọng yếu 2021 3-3 Quản lý chủ đề trọng yếu Quản lý chất thải rắn GRI 306: Waste 2020 306-1 Phát sinh chất "}\n{"id": "B-A-a68ffd62d93f9699", "claim": "Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ", "evidence": "Tổng lượng nước khai thác từ các bên thứ ba Triệu lít 1.275 303-3 Khai thác nước Tổng lượng nước khai thác từ tất cả các khu vực có áp lực về nước Triệu lít N/A 303-3 Khai thác nước Tổng lượng nước sử dụng từ tất cả các khu vực Triệu lít 41.334 303-5 Tiêu thụ nước Tổng lượng nước xả thải ra các khu vực Triệu lít 4.994 303-4 Xả thải nước Tổng lượng nước ngọt tiêu thụ tại các khu vực thiếu nước Triệu lít N/A Tổng lượng nước ngọt tiêu thụ tại các khu vực không thiếu nước Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát si"}\n{"id": "B-A-0ccdecc667e6c783", "claim": "Tổng khối lượng chất thải KHÔNG NGUY HẠI được xử lý", "evidence": "Tổng lượng nước khai thác từ các bên thứ ba Triệu lít 1.275 303-3 Khai thác nước Tổng lượng nước khai thác từ tất cả các khu vực có áp lực về nước Triệu lít N/A 303-3 Khai thác nước Tổng lượng nước sử dụng từ tất cả các khu vực Triệu lít 41.334 303-5 Tiêu thụ nước Tổng lượng nước xả thải ra các khu vực Triệu lít 4.994 303-4 Xả thải nước Tổng lượng nước ngọt tiêu thụ tại các khu vực thiếu nước Triệu lít N/A Tổng lượng nước ngọt tiêu thụ tại các khu vực không thiếu nước Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát si"}\n{"id": "B-A-6a429a8ba09f25a8", "claim": "Nhu cầu tăng trưởng, áp lực cạnh tranh và sự thiếu hụt nhận thức, kinh nghiệm, năng lực để thực hiện phát triển có trách nhiệm sẽ khiến bền vững vẫn chỉ mới là mong muốn hoặc là lời kêu gọi.", "evidence": "TABLE HỆ THỐNG VẬN HÀNH VÀ VĂN HÓA DOANH NGHIỆP TẠI BÌNH ĐẲNG VỀ CƠ HỘI, Bà Nguyễn Trình Thùy Trang PHÓ TỔNG GIÁM ĐỐC, KHỐI VẬN HÀNH KHÔNG BÌNH ĐẲNG VỀ Để khích lệ khả năng tự quyết và tinh thần sáng tạo, Công ty cũng chấp nhận những sai sót hoặc thiệt hại nhất định khi một bộ phận THÀNH TỰU nào đó chưa quen với hệ thống mới. Đây là cơ sở để mọi người thấy an tâm khi thực hiện sự đổi mới hoặc dám làm những điều chưa từng có. Khác với các mô hình kinh doanh Coteccons xác định “human capital” - con người là tài sản Một tiêu chí quan trọng khác trong cuộc tái khác, Coteccons chưa có nhà máy và nguồn vốn trọng tâm của Công ty - để từ đó, đầu tư về cấu trúc lần hai là xóa bỏ tập quán cào bằng. co"}\n{"id": "B-A-760ef7d9238cbf4e", "claim": "Bằng nhận thức đó, Coteccons lựa chọn triết lý phát triển dựa trên chiến lược ESG có tính cam kết sâu, rộng và tầm nhìn dài hạn.", "evidence": "Trong phạm trù rộng của ESG, ưu tiên trọng tâm của Coteccons Trong toàn bộ chiến lược phát triển, triết lý xuyên suốt và mục tiêu cao nhất của Coteccons là phát triển vì con người, phục vụ cho nhân sinh và sự bền vững của cộng đồng - xã hội. Do vậy, con người luôn luôn là trọng tâm trong mọi lựa chọn của chúng tôi. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons HOẠCH ĐỊNH CHIẾN LƯỢC ĐỂ THỰC HIỆN TẦM NHÌN ĐẾN 2029 Trong định hướng phát triển doanh nghiệp với tầm nhìn dài hạn và nhất quán,"}\n{"id": "B-A-e28a5ba26d14a261", "claim": "Trong đó, Coteccons đón nhận thách thức, khảo nghiệm và nguy cơ trên vai trò là người tiên phong nỗ lực thúc đẩy các tiêu chuẩn ESG của toàn ngành, góp phần giải quyết các vấn đề của ngành, đóng góp những giá trị có chiều sâu cho sự phát triển chung.", "evidence": "Chúng tôi nhận thức sâu sắc vai trò và khả năng ảnh hưởng của mình. Do đó, Coteccons muốn là và được công nhận là industry leader - thủ lĩnh trong ngành với mục tiêu thúc đẩy nâng cao tiêu chuẩn của toàn ngành và đóng góp tích cực hơn cho kinh tế - xã hội. Để hiện thực hóa sự công nhận này, trước tiên, chúng tôi cần trở thành tác nhân tạo ra xu hướng trong ngành xây dựng. MỘT DOANH NGHIỆP CHỈ THỰC SỰ VĨ ĐẠI KHI PHÁT TRIỂN VÌ CON NGƯỜI Các hoạt động xã hội của Coteccons mà công chúng thấy được trên truyền thông là phần đại diện cho chiến lược phát triển bền vững và yêu cầu thực hành ESG có chiều sâu mà Coteccons đã triển khai. ESG là một trong những phương diện mà Coteccons tiên phong lan tỏa"}\n{"id": "B-A-e88b564be82f11bf", "claim": "Qua đó, Công ty mong muốn cùng các bên thiết lập con đường rõ ràng hơn hướng đến tương lai xanh hơn cho cả cộng đồng và chính mình.", "evidence": "Tất cả báo cáo đều được công bố công khai đến các cơ quan hữu quan, truyền thông và công chúng trên website chính thức của Công ty tại www.coteccons.vn. Nhằm giúp gia tăng giá trị thông tin và tính tham khảo cho chính Coteccons cùng các bên liên quan, Công ty mong nhận được góp ý đóng góp hoặc phản hồi cho các nội dung đã được lập trong Báo cáo. 8 Tầm quan trọng đối với các bên liên quan Nam 4 Coteccons xin trân trọng cảm ơn sự đồng hành của tất cả cán bộ - nhân viên, lực lượng thi công, đối tác và các bên liên quan khác để cùng thực thi ESG từ những hoạt động thường ngày đến các chiến dịch lớn. Chúng ta hãy tiếp tục hành động cùng nhau vì tương lai xanh và bền vững cho chính mình và thế hệ "}\n{"id": "B-A-f2177591a2b07fca", "claim": "Bằng cam kết mạnh mẽ và hành động thực tế, Coteccons quyết tâm biến xanh và bền vững không còn chỉ là những mỹ từ mang tính ngoại giao.", "evidence": "TABLE TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS Coteccons xác định đa dạng là yếu tố quyết định, không chỉ dừng lại ở sự khác biệt về văn hóa, sắc tộc, hay tuổi tác, mà còn bao gồm cả tư duy, sở thích, sở trường, trình độ và phong cách làm việc. Công ty tin rằng một đội ngũ nhân sự đa dạng sẽ mang đến nhiều lợi ích, giúp tối ưu hóa quy trình làm việc và nâng cao hiệu quả công việc. Việc xây dựng một đội ngũ đa dạng không chỉ là mở rộng các tiêu chí tuyển dụng mà còn là tạo ra môi trường làm việc mà mọi nhân viên đều cảm thấy thuộc về. Một minh chứng cho sự cam kết này là hình ảnh nhận diện thương hiệu mới của logo Coteccons với 6 pantone màu chủ đạo. Đây chính là sự khẳng"}\n{"id": "B-A-83bda5d970664dee", "claim": "Tổng quan Chiến lược phát triển 2025 – 2029 28 - 33 34 - 35 Giảm phát thải - hướng tới Net Zero Kinh tế tuần hoàn Hợp tác cùng phát triển", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons SDG 13. GIẢM PHÁT THẢI - HƯỚNG TỚI NET ZERO 01 TUÂN THỦ CÁC TIÊU CHUẨN VỀ TRỌNG TÂM CHIẾN LƯỢC COTECCONS 2025 HOẠCH ĐỊNH CHIẾN LƯỢC ĐỂ THỰC HIỆN TẦM NHÌN ĐẾN 2029 Với mục tiêu đạt tỷ lệ tăng trưởng kép hàng năm (CAGR) 20-30% trong 5 năm tới, hướng tới phát triển bền vững trên vai trò industry leader ngành xây dựng tại Việt"}\n{"id": "B-A-6f486b1b9317249d", "claim": "Phát triển con người Cộng đồng bền vững 44 45 92 - 109 112 - 123 126 - 135 136 - 159 160 - 175 HĐQT: Hội đồng quản trị QTCT: Quản trị công ty", "evidence": "137 136 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS 109 108 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-27a5052986e273d4", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Dưới góc nhìn của tôi, nếu kết quả kinh doanh không mang lại lợi ích thực sự cho con người thì đó chỉ là một thành tựu vô nghĩa.", "evidence": "Trong quan niệm của tôi, nếu kết quả kinh doanh không phục vụ cho lợi ích thực sự của con người thì đó là một thành tựu vô nghĩa. Một doanh nghiệp theo đuổi sự tăng trưởng mà không tạo được giá trị thực cho cộng đồng thì dù lớn mạnh đến đâu cũng không thể trở thành một doanh nghiệp vĩ đại được. phục vụ cho nhân sinh và sự bền vững của cộng đồng - xã hội. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons NHỮNG CON SỐ TIÊU BIỂU TRONG NĂM Về thực thi ESG Về phát triển kinh doanh"}\n{"id": "B-A-e47c2109d6039490", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Tại Coteccons, tất yếu chúng tôi đang thay đổi để tốt hơn.", "evidence": "TABLE Phần I Phần II Phần III Phần IV Phần V Phần VI Con đường của Tổng quan Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG tương lai xanh về Coteccons về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons Tại Coteccons, tất yếu chúng tôi đang thay đổi để tốt hơn. Sau hàng ngàn giờ suy nghĩ, chúng tôi không hề nghi ngờ gì về sự thành công của con đường mình đã chọn. Bây giờ chỉ là vấn đề thực thi, chậm hay nhanh chúng tôi sẽ thực hiện được. Chúng tôi đã tập trung vào các nền tảng và đã đạt được sự ổn định. Bây giờ chúng tôi đang chuyển sang giai đoạn tiếp theo của cuộc hành trình, nơi chúng tôi có thể chấp nhận rủi ro để hướng tới kết quả cao hơn. | | | Tại Coteccon"}\n{"id": "B-A-7a33a7c28c9754b8", "claim": "Nỗ lực giảm phát thải, tôn trọng và phát triển con người dựa trên nền tảng quản trị minh bạch là những ưu tiên hàng đầu của Coteccons.", "evidence": "Trong đó, để hiệu quả, Công ty xác định các vấn đề trọng yếu và nhiệm vụ ưu tiên thiết thực. Nỗ lực giảm phát thải, tôn trọng và phát triển con người trên nền tảng quản trị công ty minh bạch là ưu tiên hàng đầu của Coteccons. Để trở thành industry leader, Coteccons tự đặt cho mình 3 nhiệm vụ cụ thể. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG Minh bạch là một trong những tiêu chí trọng tâm mà Coteccons kiên định theo đuổi từ khi bắt đầu thực hiện tiến trình tái cấu trúc doanh nghiệp."}\n{"id": "B-A-42d701d3ac36948e", "claim": "2021, khi tôi đề cập tới 3 chữ (sustainable development growth), không ai hiểu và thấm thía tôi nói gì.", "evidence": "United Nations. (2015). Transforming our world: the 2030 Agenda for Sustainable Development Sáng kiến Báo cáo Toàn cầu GRI Global Reporting Initiative. (2020). GRI Standards Nguyên tắc Hướng dẫn của Liên Hợp Quốc về Kinh doanh và Nhân quyền (United Nations. Chúng tôi có cập nhật và điều chỉnh để phù hợp với sự biến đổi liên tục của thị trường. Tuy nhiên, các chỉ tiêu lớn vẫn không đổi. Nói đúng ra, chúng tôi ngày càng cụ thể hóa lộ trình hành động của mình."}\n{"id": "B-A-517d52e4beef3d30", "claim": "Theo ước tính, ngành này tiêu thụ khoảng 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và sử dụng một lượng lao động khổng lồ.", "evidence": "Theo ước lượng, ngành xây dựng sử dụng đến 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và liên quan đến lượng lao động rất lớn. Đặc biệt, sản phẩm mà ngành xây dựng tạo ra có vai trò sâu rộng và lâu dài với xã hội - cộng đồng. Như vậy, ngành xây dựng và doanh nghiệp xây dựng chắc chắn phải có trách nhiệm, phải nghiêm khắc với tiêu chuẩn môi trường và xã hội trong từng hoạt động của mình, ở mọi giai đoạn phát triển. Các đặc thù về văn hóa, xã hội, địa lí luôn được quan tâm trong quá trình lập chính sách về môi trường, các qui định về an toàn và tuân thủ về bảo vệ môi trường trong quá trình thực hiện thi công các công trình nhằm mục tiêu đảm bảo"}\n{"id": "B-A-d20f21ccb166d076", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Khi chúng tôi xây dựng một công trình, chúng tôi hiểu rằng Coteccons đang cùng với khách hàng kiến tạo những giá trị trường tồn, không chỉ giới hạn ở lợi ích kinh tế mà còn là biểu tượng của bảo vệ môi trường và phát triển cộng đồng bền vững.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Start Tại Coteccons, chúng tôi tin rằng sự bền vững bắt đầu từ chính trong mỗi người “Tài sản trường tồn mà chúng ta “Cùng đồng hành” Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons NHỮNG CON SỐ TIÊU BIỂU TRONG NĂM Về thực thi ESG Về phát triển kinh doanh"}\n{"id": "B-A-c48d69fa93679e1e", "claim": "Chúng tôi tin rằng không nhất thiết phải lựa chọn giữa việc tăng trưởng kinh doanh, tìm kiếm lợi nhuận với phát triển xanh hơn, bền vững hơn.", "evidence": "Đây mới là kết quả quan trọng nhất, bởi nó là nền tảng vững chắc cho những triển vọng cao hơn trong tương lai. Chúng tôi nỗ lực tìm kiếm để định hình được con đường cần phải đi và nên đi. Trong đó, chúng tôi nhận thấy rằng đã đến lúc, thực hành ESG và phát triển kinh doanh bền vững phải là cốt lõi cho sinh mệnh và tương lai của doanh nghiệp. - chất xúc tác - mắt xích quan trọng trong việc thúc đẩy sự phát triển bền vững của ngành xây dựng nói chung. Chúng tôi phải đóng vai trò kết nối giữa các đối tác, nhà thầu phụ, khách hàng, kỹ sư và công nhân xây dựng... trong thực hành ESG. Chúng tôi tin rằng, cùng với các bên liên quan, chúng ta có thể tìm ra những giải pháp bền vững, hiệu quả và đem l"}\n{"id": "B-A-90d1ef7270a7edb5", "claim": "Các mục tiêu này đều có thể đạt được bằng con đường thực hành ESG.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhà máy Suntory PepsiCo do Coteccons thi công tại Long An Để đảm bảo rằng đang đạt được tiến bộ hướng tới các mục tiêu bảo tồn nước, Công ty đo lường lượng nước tiêu thụ trong các hoạt động xây dựng. Điều này giúp Công ty có thể xác định các lĩnh vực có thể cải thiện và theo dõi tiến trình hướng tới các mục tiêu bền vững. Được xếp vào nhóm này là sự công nhận cho cam kết của Coteccons hướng về việc đặt con người vào trung tâm của hoạt động kinh doanh gắn liền với các mục tiêu ESG. HIỆN DIỆN LẦN THỨ 6 LIÊN TIẾP TRONG TOP 50 DOANH"}\n{"id": "B-A-a88d4ba35e9f997e", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Start Tại Coteccons, chúng tôi tin rằng sự bền vững bắt đầu từ chính trong mỗi người “Tài sản trường tồn mà chúng ta “Cùng đồng hành”", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Khi chúng tôi xây dựng một công trình, chúng tôi hiểu rằng Coteccons đang cùng với khách hàng kiến tạo những giá trị trường tồn, không chỉ giới hạn ở lợi ích kinh tế mà còn là biểu tượng của bảo vệ môi trường và phát triển cộng đồng bền vững. Với vai trò là tổng thầu xây dựng hàng đầu Việt Nam, Coteccons xác định phải trở thành trung tâm Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-f4ad520e8e242f26", "claim": "Nhận thức Tầm nhìn có thể có được” Mọi người Mọi bên liên quan Tương lai xanh hơn Mọi cộng đồng Tương lai tốt đẹp hơn Mọi dự án Tăng trưởng bền vững", "evidence": "Electric tổ chức. Tại đây, Coteccons chia sẻ tầm nhìn trong việc định hình lại hệ sinh thái xây dựng bền vững và thúc đẩy ngành xây dựng xanh, tạo ra những không gian an toàn hơn, thoải mái hơn, ý nghĩa hơn trong toàn bộ chuỗi giá trị bằng những cam kết với các tiêu chuẩn ESG và HSE. Co- teccons cũng truyền đi thông điệp sẵn sàng đồng hành cùng các đối tác có cùng tầm nhìn và giá trị, nhằm lan tỏa giá trị ESG hiệu quả và thiết thực nhất đến mọi khía cạnh của ngành. 127 126 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Bằng nhận thức đó, Coteccons lựa chọn triết lý phát triển dựa trên chiến lược ESG có tính cam kết sâu, rộng và tầm nhìn dài hạn. Trong đó, Coteccons đón nhận thách thức, khảo nghiệm và"}\n{"id": "B-A-305bb6cc44fcd286", "claim": "Phát triển bền vững là kết quả của một lối tư duy hướng về tương lai, đặt con người và môi trường lên hàng đầu.", "evidence": "Nếu tư duy chỉ tập trung vào ngắn hạn và lợi nhuận trước mắt, sự phát triển sẽ không thể bền vững. Con đường của tương lai được mở ra từ tư duy đúng đắn về phát triển bền vững trong mỗi cá nhân và từng doanh nghiệp. 19 18 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Coteccons trở thành một thương hiệu hướng tới giá trị con người, nâng tầm tiêu chuẩn ngành xây dựng về môi trường, phục vụ cho lợi ích hài hòa của tất cả bên liên quan, phát triển bền vững vì cộng đồng và tương lai của xã hội. Coteccons xem đây là đạo đức nền tảng để phát triển một doanh nghiệp chân chính. Năng lực, lợi thế đã có và chiến lược thực hành ESG về chiều sâu là điều kiện để Coteccons hướng tới thị trường vốn xanh trong giai "}\n{"id": "B-A-0bdca6b162d336ab", "claim": "Nếu tư duy chỉ tập trung vào ngắn hạn và lợi nhuận trước mắt, sự phát triển sẽ không thể bền vững.", "evidence": "Một doanh nghiệp muốn bền vững phải phát triển bằng tư duy hướng về cái chung, được vận hành bởi một tập thể, phụng sự cho lợi ích hài hòa của các bên. Coteccons không tạo quyền lực cho thiểu số cá nhân và không đánh đổi mục tiêu xây dựng để trường tồn cho những lợi ích cục bộ. 7 6 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Một doanh nghiệp muốn bền vững phải phát triển bằng tư duy hướng về cái chung, được vận hành bởi một tập thể, phụng sự cho lợi ích hài hòa của các bên. Coteccons không tạo quyền lực cho thiểu số cá nhân và không đánh đổi mục tiêu xây dựng để trường tồn cho những lợi ích cục bộ. 189 188 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-2571b0fcf1edb668", "claim": "đã thực hiện trong năm trên tất cả công trường Coteccons Group Backlog Giá trị thắng thầu mới 22.000 30.000 Số giờ đào tạo về ESG Tỷ lệ công nhân được đào tạo về an toàn", "evidence": "21.045 Tỷ VNĐ 713 Tỷ VNĐ SỐ GIỜ LAO ĐỘNG AN TOÀN Doanh thu thuần Tỷ VNĐ trên tất cả công trường Coteccons Group GIÁ TRỊ THẮNG THẦU MỚI 30,8% 22.000 Tỷ VNĐ Coteccons nâng cao mức độ an toàn từ chính công đoạn phòng tránh. Điều này được thực hiện thông qua đào tạo về kỹ năng và nhận thức, đưa văn hóa tự quản lý, tự lãnh đạo về thực hiện an toàn lan tỏa vào toàn hệ thống. Trong năm qua, Ban HSE (Sức khỏe - An toàn - Môi trường) của Coteccons đã tiến hành hơn 7.100 giờ đào tạo cho công nhân, 5.900 giờ đào tạo cho kỹ sư và giám sát công trường."}\n{"id": "B-A-3d58e006c7f6b341", "claim": "“Xây Tết” Dự án của Công ty áp dụng phần mềm kiểm kê khí nhà kính Số lượng cây được trồng từ các chương trình trồng rừng thắng thầu mới 159.986 100% 27.473 Doanh thu thuần Tăng", "evidence": "Báo cáo và phân tích dữ liệu lượng phát thải của doanh nghiệp. Đặc biệt, Công ty đã xây dựng được một đội ngũ nhân sự có kỹ năng cơ bản trong kiểm kê khí nhà kính, có khả năng thu thập và phân tích dữ liệu, sử dụng thành thạo phần mềm quản lý khí nhà kính, có kiến thức về quy trình sản xuất và quản lý năng lượng... để tiếp tục hành trình giảm phát thải, hướng tới tương lai bền vững. Coteccons luôn bao hàm đối tượng là công nhân - lực lượng thi công, ngay cả khi đây là lực lượng lao động gián tiếp, được Coteccons hợp tác thông qua các nhà cung cấp, nhà thầu phụ. Qua 2 mùa “Xây Tết”, Coteccons đã trực tiếp tri ân đến 27.473 lượt công nhân từ 68 công trường, xây dựng niềm tin, tôn vinh vai trò "}\n{"id": "B-A-586c6cd9d71579e5", "claim": "VNSI là chỉ số đánh giá bao gồm 20 cổ phiếu thuộc VN100 có tính phát triển bền vững tốt nhất được sàng lọc thông qua các điều kiện nghiêm khắc.", "evidence": "Với vị thế trên, Coteccons gia tăng động lực để tiếp tục định hướng trở thành “Industry Leader” (đơn vị dẫn đầu ngành bằng đóng góp về tiêu chuẩn chất lượng, sáng kiến có giá trị và ảnh hưởng tích cực đến toàn ngành). Với nỗ lực xuyên suốt theo đuổi mục tiêu về phát triển bền vững, ngày 01/11/2023, Coteccons tiếp tục được vinh danh trong xếp hạng Chỉ số các công ty thực hành về phát triển bền vững (VNSI) do HOSE đánh giá. VNSI là chỉ số đánh giá bao gồm 20 cổ phiếu thuộc VN100 có tính phát triển bền vững tốt nhất được sàng lọc thông qua các điều kiện nghiêm khắc. Với nền tảng chắc chắn của doanh nghiệp, cổ phiếu CTD đã được HoSE xếp vào danh mục Top 20 cổ phiếu có chỉ số phát triển bền vững "}\n{"id": "B-A-b709420e964a8ee2", "claim": "Được xếp vào nhóm này là sự công nhận cho cam kết của Coteccons hướng về việc đặt con người vào trung tâm của hoạt động kinh doanh gắn liền với các mục tiêu ESG.", "evidence": "Được xếp vào nhóm này là sự công nhận cho cam kết của Coteccons hướng về việc đặt con người vào trung tâm của hoạt động kinh doanh gắn liền với các mục tiêu ESG. HIỆN DIỆN LẦN THỨ 6 LIÊN TIẾP TRONG TOP 50 DOANH Kết quả trên đặc biệt có ý nghĩa khi mức xếp hạng nhóm A gần như chỉ dành cho các doanh nghiệp năng lượng và ngân hàng và chưa có công ty xây dựng nào được xếp hạng ở nhóm này. Coteccons cũng là công ty xây dựng duy nhất đạt mức xếp hạng tín nhiệm BBB+, là mức cao nhất trong nhóm B. Việc được liệt kê trong danh sách Fortune 500 Đông Nam Á đã khẳng định thành công theo “Cách Coteccons” trong việc thúc đẩy những nỗ lực về sự minh bạch, tính chính trực để trở thành"}\n{"id": "B-A-b60a14768d6a58e7", "claim": "Coteccons nhận giải thưởng tại hai hạng mục quan trọng là E – Giảm thiểu dấu chân carbon và G – Tinh thần lãnh đạo ESG.", "evidence": "Coteccons nhận giải thưởng tại hai hạng mục quan trọng là E - Giảm thiểu dấu chân carbon và G - Tinh thần lãnh đạo ESG. Ngày 18/01/2024, Vietnam Report phối hợp cùng Báo điện tử VietnamNet tổ chức Lễ công bố Bảng xếp hạng VNR500 năm 2023. Trong đó, Coteccons và Unicons đã được vinh danh là doanh nghiệp thuộc Top 50 doanh nghiệp xuất sắc Việt Nam (Top 50 Vietnam The Best). Giải thưởng này ghi nhận những nỗ lực không ngừng của Coteccons trong việc tiên phong trên tinh thần đổi mới - sáng tạo trong thực hành ESG. Top 50 doanh nghiệp xuất sắc Việt Nam là bảng xếp hạng uy tín được Ban tổ chức thực hiện thường niên."}\n{"id": "B-A-d4ed4e7e25de92ae", "claim": "Top 10 Nhà thầu được Ban tổ chức xét duyệt dựa trên các tiêu chí tổng giá trị xây lắp của tất cả các dự án ở giai đoạn thi công được khởi công trong hai năm, từ 01/01/2022 đến 31/12/2023, nhân hệ số đối với dự án có chứng chỉ Xanh của Leed, Lotus, Green Mark, Green Star, EDGE và nhân hệ số đối với loại hình dự án văn phòng/trụ sở, khách sạn/", "evidence": "31/12/2023, nhân hệ số đối với dự án có chứng chỉ Xanh của Leed, Lotus, Green Mark, Green Star, EDGE và nhân hệ số đối với loại hình dự án văn phòng/trụ sở, khách sạn/nghỉ dưỡng, công nghiệp/nhà máy. Coteccons là một trong số “Doanh nghiệp đạt Chuẩn Công bố thông tin” trong danh sách kể trên. CTCP XÂY DỰNG COTECCONS HOSE: CTD Signed Verified Ký ngày: 29/06/2024 Signed Verified Ký ngày: 29/06/2024 Top 10 Nhà thầu được Ban tổ chức xét duyệt dựa trên các tiêu chí tổng giá trị xây lắp của tất cả các dự án ở giai đoạn thi công được khởi công trong hai năm, từ 01/01/2022 đến"}\n{"id": "B-A-d39b6e7f28f6a430", "claim": "Trong đó, một số chỉ tiêu điển hình là giá trị vốn hóa 1 tỷ USD, phát triển bền vững trên vai trò industry leader tại Việt Nam và hiện diện vững chắc trên thị trường nước ngoài.", "evidence": "TRỌNG TÂM CHIẾN LƯỢC COTECCONS 2025 HOẠCH ĐỊNH CHIẾN LƯỢC ĐỂ THỰC HIỆN TẦM NHÌN ĐẾN 2029 Với mục tiêu đạt tỷ lệ tăng trưởng kép hàng năm (CAGR) 20-30% trong 5 năm tới, hướng tới phát triển bền vững trên vai trò industry leader ngành xây dựng tại Việt Coteccons hướng tới mục tiêu trở thành và được công nhận là Industry Leader, là hình mẫu trong cam kết thực hành ESG trong lĩnh vực Xây dựng, mang đến các giá trị bền vững đóng góp vào sự phát triển kinh tế, xã hội và bảo vệ môi trường bằng cách kinh doanh có đạo đức và trách nhiệm thông qua Quản trị Tốt. Cơ cấu tổ chức Quản trị Phát triển bền vững được thiết lập nhằm đảm bảo việc thực thi Phát triển bền vững hiệu quả và có tác động."}\n{"id": "B-A-99c5c57f7f9d05ba", "claim": "Tham khảo các bộ tiêu chuẩn như GRI, 17 tiêu chí Phát triển bền vững của Liên Hợp Quốc, Hướng dẫn công bố thông tin của Ủy ban", "evidence": "Trong đó, phần cốt lõi là kết quả thực hiện phát triển bền vững của Coteccons trên các lĩnh vực trọng yếu. Báo cáo ESG của Coteccons được xây dựng trên cơ sở tham khảo phiên bản mới nhất của GRI (Bộ tiêu chuẩn quốc tế về Báo cáo Phát triển bền vững dành cho doanh nghiệp đang đượ c áp dụng tại nhiều quốc gia), 17 tiêu chí Phát triển bền vững của Liên Hợp Quốc, Hướ ng dẫn công bố thông tin của Ủy ban Chứng khoán Nhà nướ c, Thông tư 155/2015/TT- Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons KHUNG CHÍNH SÁCH/TIÊU CHUẨN THAM CHIẾU QUỐC TẾ 17 Mục tiêu phát triển bền vững của Liên Hợp Quốc"}\n{"id": "B-A-cbca97f0ce054d9f", "claim": "Căn cứ vào đặc thù ngành, 5 mục tiêu chiến lược và các giai đoạn phát triển của Coteccons cũng như thông qua hàng trăm tiếp xúc với các bên liên quan để thấu hiểu kỳ vọng, Coteccons đã xác định 15 vấn đề trọng yếu về ESG để tập trung đầu tư và ưu tiên giải quyết trong năm vừa qua.", "evidence": "Xác định phạm vi báo cáo và ảnh hưởng các bên liên quan. Xác định các vấn đề trọng yếu theo 5 mục tiêu của chiến lược ESG. Tham khảo các bộ tiêu chuẩn như GRI, 17 tiêu chí Phát triển bền vững của Liên Hợp Quốc, Hướng dẫn công bố thông tin của Ủy ban Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Do đặc thù của ngành xây dựng yêu cầu sự giám sát chặt chẽ và liên tục ở từng giai đoạn, nên trong giai đoạn tiếp theo của công tác kiểm kê khí nhà kính, Coteccons sẽ tập trung vào việc phân tích chi tiết hơn các kết quả kiểm kê, đồng thời nghiên cứu và triển khai các giải pháp giảm phát thải hiệu quả. Cụ thể, các hoạt động này bao gồm: vững - thép xanh (tương đương 3"}\n{"id": "B-A-94b1eabb61534657", "claim": "Trên hành trình đó, Coteccons sẽ tiếp tục kiên định với tinh thần minh bạch để dẫn đầu ngành xây dựng không chỉ bằng kết quả kinh doanh, mà còn bằng những kiến tạo và đóng góp mang tính tiên phong, góp phần nâng cao tiêu chuẩn của toàn ngành và giá trị bền vững cho nền kinh tế - xã hội.", "evidence": "Cụ thể hơn, Công ty phát huy tính tuân thủ với những tiêu chuẩn đã có, nâng cấp lên những tiêu chuẩn cao hơn và thực hiện các kế hoạch, sáng kiến, yêu cầu mới nhằm biến nhận thức về ESG vào thực hành cụ thể trên mọi phương diện của đời sống doanh nghiệp. Từ tiền đề đó, trong chiến lược phát triển tiếp theo, Coteccons đang hướng đến tiêu chuẩn và tầm nhìn mới về ESG. Công ty xác định tầm nhìn indus- try leader - thủ lĩnh ngành để nỗ lực thúc đẩy nâng cao tiêu chuẩn của toàn ngành và đóng góp giá trị tích cực hơn cho nền kinh tế - xã hội. Đây là sứ mệnh nâng cao tiêu chuẩn của toàn ngành xây dựng, khởi xướng và đóng góp giá trị đích thực cho sự phát triển kinh tế - xã hội, phục vụ cho lợi ích "}\n{"id": "B-A-7512a1835ed81399", "claim": "Giảm phát thải khí nhà kính bằng cách sử dụng nguyên nhiên liệu, thiết bị hiệu quả và thực hành tiết kiệm năng lượng.", "evidence": "TABLE 1 Giảm phát thải khí nhà kính bằng cách sử dụng nguyên nhiên liệu, thiết bị hiệu quả và thực hành tiết kiệm năng lượng. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons CHÍNH SÁCH VỀ MÔI TRƯỜNG CỦA COTECCONS 1 2 3 4 5 Giảm phát thải khí nhà kính bằng cách sử dụng nguyên nhiên liệu, thiết bị hiệu quả và thực hành tiết kiệm năng lượng."}\n{"id": "B-A-e4979a55aa42113d", "claim": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm thiểu dấu chân (footprint) carbon.", "evidence": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất và phát huy đổi mới sáng tạo trong hoạt động giảm phát thải; Lưu trữ và quản lý dữ liệu lượng phát thải của Coteccons, dành cho cả những công trình, dự án đã hoàn thiệ"}\n{"id": "B-A-6ff93f750c76a424", "claim": "Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và tối ưu hóa việc giảm thiểu chất thải.", "evidence": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững Thực hiện phương án giảm thiểu ồn, bụi trong quá trình thi công. Thực hiện việc thu gom và xử lý nước thải đạt chuẩn trước khi thải ra môi trường. Thực hiện thu gom và xử lý các chất thải/đất/bùn thải từ quá trình thi công của mình."}\n{"id": "B-A-38c6b594eb72898b", "claim": "Sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời và các nguồn khác.", "evidence": "Sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời 91 90 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Công ty đã chính thức ban hành cam kết sử dụng năng lượng bền vững và đưa vào hoạt động hàng ngày. Hiện Coteccons đã sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời để cung cấp năng lượng cho một số công trường xây dựng và tòa nhà văn phòng của Công ty. Nỗ lực này đang được nhân rộng đến tất cả các dự án của"}\n{"id": "B-A-aca0f01a2d2e9c6a", "claim": "ISO 14064-1: Greenhouse gases — Part 1: Specification with guidance at the organization level for quantification and reporting of greenhouse gas emissions and removals.", "evidence": "International Organization for Standardization. (2015). ISO 14001: Environmental Management Systems – Requirements with Guidance for Use. ISO ISO 14064-1: Tiêu chuẩn về đo lường và báo cáo phát thải khí nhà kính ở cấp độ tổ chức VVVF Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: VVVF drives, regenerative drive, efficient lighting."}\n{"id": "B-A-da315a5d2383b609", "claim": "VIỆT NAM Luật Bảo vệ môi trường 2020 Thông tư 155/2015/ TT-BTC: Trách nhiệm công bố thông tin của công ty, công bố thông tin kinh doanh, công bố báo cáo tài chính định kỳ Nghị định 06/2022/ NĐ-CP Quy định giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn", "evidence": "Coteccons trước hết tuân thủ các quy định của Luật Bảo vệ môi trường và dựa theo hệ thống các nghị định hướng dẫn về các yêu cầu cụ thể của luật để thực hiện. Nghị định hướng dẫn Luật Bảo vệ môi trường năm 2020 bao gồm: Nghị định 06/2022/NĐ-CP quy định về giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn. Theo Nghị định số 06/2022/NĐ-CP về giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn, các doanh nghiệp phải xây dựng kế hoạch giảm phát thải phù hợp với mục tiêu quốc gia. Dù Coteccons không thuộc nhóm 2.166 doanh nghiệp được Bộ Xây dựng yêu cầu kiểm kê bắt buộc, nhưng với tầm nhìn trở thành industry leader,"}\n{"id": "B-A-ec00fe0ad1703879", "claim": "Quản lý, điều phối triển khai các hoạt động ESG trên 3 lĩnh vực hoạt động: công trình, môi trường và cộng đồng bền vững.", "evidence": "Trong báo cáo này, nội dung cốt lõi là kết quả thực hiện phát triển bền vững của Coteccons trên các lĩnh vực trọng yếu. Việc xác định các lĩnh vực trọng yếu được dựa trên đánh giá kết hợp giữa mức độ quan tâm của các bên liên quan, tác động của chúng tới Coteccons, ngành xây dựng và môi trường kinh tế - xã hội. KINH TẾ QUẢN TRỊ MÔI TRƯỜNG XÃ HỘI 1. Hiệu quả hoạt động kinh tế CỘNG ĐỒNG BỀN VỮNG BỀN VỮNG TIỂU BAN ESG PHỐI HỢP VỚI CÁC BÊN LIÊN QUAN PHỐI HỢP VỚI CÁC BÊN LIÊN QUAN PHỐI HỢP VỚI CÁC BÊN LIÊN QUAN"}\n{"id": "B-A-aca0c3414d9d5fb3", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Ông.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 17 16 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-7fa9734fbe84b946", "claim": "Việc thành lập Tiểu ban ESG được xác nhận bằng Nghị quyết 13B/2022/NQ-HDQT với 7/7 thành viên HĐQT tán thành.", "evidence": "TIỂU BAN ESG 3 Võ Hoàng Lâm Thành viên phụ trách BU1 Được thành lập theo Nghị quyết HĐQT số 13B/2022/NQ-HDQT ngày 14/12/2022. Việc thành lập Tiểu ban ESG thể hiện rõ cam kết nâng cao hiệu quả của Công ty trong thực hành phát triển bền vững và quyết tâm lan tỏa tầm ảnh hưởng tích cực về lĩnh vực này đến cộng đồng xã hội cũng như ngành xây dựng tại Việt Nam. 4 Ông Nguyễn Chí Thiện Thành viên phụ trách BU2 Thành viên HĐQT; Phó TGĐ; TGĐ BU1 Thành viên TIỂU BAN CHIẾN LƯỢC Được thay đổi thành viên và tiếp tục hoạt động trở lại theo Nghị quyết 08/2022/NQ-HDQT ngày 03/05/2022. 3 Ông Phạm Quân Lực Phó TGĐ BU1 Thành viên"}\n{"id": "B-A-9d6532933e0127b5", "claim": "Tạo lập team ESG đa chức năng và tối ưu hóa nhóm đã có sẵn là một trong những bước quan trọng nhất để hiện thực hóa chiến lược ESG của Công ty và là điều cần thiết để thành công trong các nỗ lực ESG.", "evidence": "TABLE HỆ THỐNG VẬN HÀNH VÀ VĂN HÓA DOANH NGHIỆP TẠI BÌNH ĐẲNG VỀ CƠ HỘI, Bà Nguyễn Trình Thùy Trang PHÓ TỔNG GIÁM ĐỐC, KHỐI VẬN HÀNH KHÔNG BÌNH ĐẲNG VỀ Để khích lệ khả năng tự quyết và tinh thần sáng tạo, Công ty cũng chấp nhận những sai sót hoặc thiệt hại nhất định khi một bộ phận THÀNH TỰU nào đó chưa quen với hệ thống mới. Đây là cơ sở để mọi người thấy an tâm khi thực hiện sự đổi mới hoặc dám làm những điều chưa từng có. Khác với các mô hình kinh doanh Coteccons xác định “human capital” - con người là tài sản Một tiêu chí quan trọng khác trong cuộc tái khác, Coteccons chưa có nhà máy và nguồn vốn trọng tâm của Công ty - để từ đó, đầu tư về cấu trúc lần hai là xóa bỏ tập quán cào bằng. co"}\n{"id": "B-A-11f4bdc1e9020b79", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Tiến sĩ Đinh Thị Ngọc Bích Chuyên gia tư vấn Thành viên độc lập Tiểu ban ESG", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons NHỮNG CON SỐ TIÊU BIỂU TRONG NĂM Về thực thi ESG Về phát triển kinh doanh Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Ông. Herwig Guido H. Van Hove TIỂU BAN ESG Thành viên HĐQT Để thực thi phát triển bền vững có tính chuyên môn sâu và nhất quán từ chiến lược đến hành động cụ thể trên toàn chuỗi giá trị của doanh nghiệp, ngày 14/12/2022, HĐQT đã quyết định thành lập Tiểu ban ESG trực thuộc HĐQT, bên cạnh các tiểu ban chuyên trách như Tiểu ban"}\n{"id": "B-A-329049671903a1ea", "claim": "Chuyên gia tư vấn 2022 – Hiện tại: Nhà nghiên cứu và ủng hộ chính sách Quốc tịch: Việt Nam Kinh nghiệm nghề nghiệp Năm sinh: 1959 Tiến sĩ Bích là một chuyên gia hàng đầu trong tư vấn và triển khai các chiến lược phát triển bền vững cho các tập đoàn và tổ chức lớn.", "evidence": "Giám đốc Phát triển bền vững Tổ trưởng Tổ triển khai và thực thi ESG Quốc tịch: Việt Nam Quốc tịch: Việt Nam Năm sinh: 1968 Năm sinh: 1976 Trình độ chuyên môn: Cử nhân Quản trị Kinh doanh Với hơn 30 năm kinh nghiệm quản lý và điều phối các dự án phát triển trong và ngoài nước, bà chuyên sâu về xây dựng thể chế; tư vấn về tổ chức, quy trình và chính sách trong lĩnh vực phát triển bền vững và kinh doanh có trách nhiệm; cũng như đảm bảo sự phát triển bền vững của chuỗi cung ứng và kết nối chuỗi giá trị theo tiêu chuẩn quốc tế và trong nước."}\n{"id": "B-A-e934f0b44124d75b", "claim": "Những nỗ lực của bà đã giúp Coteccons đạt được nhiều thành tựu trong cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng những cải thiện trong chuỗi cung ứng, và sử dụng nguồn nhân lực.", "evidence": "Quản lý, điều phối triển khai các hoạt động ESG trên 3 lĩnh vực hoạt động: công trình, môi trường và cộng đồng bền vững. Chuyên gia tư vấn ESG - Thành viên độc lập tiểu ban: Bà ĐINH THỊ NGỌC BÍCH Tư vấn chiến lược cho các dự án phát triển bền vững nhằm: Cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng những cải thiện trong chuỗi cung ứng, và sử dụng nguồn nhân lực. Đồng thời, Công ty cam kết góp phần thúc đẩy nền kinh tế sở tại, tạo việc làm và hình thành một lực lượng lao động lành nghề và đa dạng tại địa phương. Cam kết này được thực hiện cụ thể thông qua việc sử dụng nguồn nguyên liệu và dịch vụ từ các nhà cung cấp bản địa, tuyển dụng lao động tại chỗ, xây dựng các c"}\n{"id": "B-A-14214cc3e026cef9", "claim": "Bà cũng đóng góp vào việc xây dựng đội ngũ nhân sự chuyên môn cao, có khả năng hiện thực hóa các mục tiêu ESG trong suốt quá trình phát triển của Coteccons.", "evidence": "Unicons. Bà Thùy Trang tích cực tham gia vào việc xây dựng và phát triển đội ngũ, văn hóa doanh nghiệp Coteccons. 51 50 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Đây là hoạt động có ý nghĩa kết nối thành viên, đánh thức năng lượng và trình diễn khả năng của nhân viên ngoài năng lực chuyên môn trong công việc. Đại hội thể thao là sự kiện để khích lệ người lao động tập luyện và phát triển khả năng xuyên suốt trong năm. Qua đó, xây dựng lối sống lành mạnh, cân bằng giữa cuộc sống và công việc."}\n{"id": "B-A-aa128c04911441af", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Bà Trần Thị Liễu Vinh Ông Đỗ Dương Trúc Giám đốc Sức Khỏe, An Toàn, Môi Trường (HSE)", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons NHỮNG CON SỐ TIÊU BIỂU TRONG NĂM Về thực thi ESG Về phát triển kinh doanh Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons QUẢN LÝ RỦI RO LIÊN QUAN ĐẾN ESG RỦI RO VỀ AN TOÀN LAO ĐỘNG"}\n{"id": "B-A-dce33a25305cceea", "claim": "Kinh nghiệm trên vai trò thành viên ESG/HSE tại Coteccons Bà trở thành Giám đốc Hành chính & Phát triển bền vững từ ngày 01/11/2022 và đảm nhiệm vai trò Tổ trưởng Tổ triển khai và thực thi Phát triển bền vững từ ngày 19/06/2023.", "evidence": "2020 - Hiện tại: Thành viên HĐQT Coteccons 2019 - Hiện tại: Thành viên sáng lập và Giám đốc The8th Pte. Ltd. TỔ TRIỂN KHAI VÀ THỰC THI ESG 2018 - Hiện tại: Giám đốc danh mục Quỹ đầu tư mạo hiểm Để triển khai các chiến lược, phương pháp và chính sách ESG của Công ty một cách cụ thể, biến kết quả phát triển bền vững thành tác động kinh doanh, Coteccons đã thành lập Tổ triển khai và thực thi Phát triển bền vững (Team ESG) theo quyết định chính thức được ban hành vào ngày Thành viên độc lập Ông TALGAT TURUMBAYEV Thành viên không điều hành Kinh nghiệm trên vai trò thành viên HĐQT tại Coteccons Kinh nghiệm trên vai trò thành viên HĐQT tại Coteccons"}\n{"id": "B-A-caeae4a6d433e254", "claim": "Các vị trí quản lý cấp cao đã đảm nhiệm 2022 - Hiện tại: Giám đốc Hành chính và Phát triển bền vững tại Coteccons Các vị trí quản lý cấp cao đã đảm nhiệm 2023 - hiện tại: Giám đốc HSE của Coteccons", "evidence": "Coteccons. Các vị trí quản lý cấp cao đã đảm nhiệm 2024 - Hiện tại: Phó Tổng giám đốc, Giám đốc vận hành 2023 - Hiện tại: Thành viên Tiểu ban ESG Trong đó, trách nhiệm điển hình là đưa ESG đến mọi lĩnh vực kinh doanh của công ty, làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả nhân viên và các bên liên quan. Các vị trí quản lý cấp cao đã đảm nhiệm 2024 - Hiện tại: Phó Tổng giám đốc, Giám đốc vận hành 2023 - Hiện tại: Thành viên Tiểu ban ESG"}\n{"id": "B-A-099b4a431918a51a", "claim": "Tận dụng thế mạnh về nhân sự về quy trình, sự thấu hiểu trong các mối quan hệ giữa người lao động và doanh nghiệp, bà Vinh chính thức đảm nhiệm vai trò Giám đốc Phát triển bền vững từ tháng 11/2022, dẫn dắt sự chuyển đổi và tích hợp kinh doanh có trách nhiệm với môi trường, cộng đồng và xã hội, đẩy mạnh triển khai thực hành tốt ESG trong vận hành tổ chức, hướng đến các mục tiêu bền vững theo chiến lược của Công ty.", "evidence": "Coteccons hướng tới mục tiêu trở thành và được công nhận là Industry Leader, là hình mẫu trong cam kết thực hành ESG trong lĩnh vực Xây dựng, mang đến các giá trị bền vững đóng góp vào sự phát triển kinh tế, xã hội và bảo vệ môi trường bằng cách kinh doanh có đạo đức và trách nhiệm thông qua Quản trị Tốt. Cơ cấu tổ chức Quản trị Phát triển bền vững được thiết lập nhằm đảm bảo việc thực thi Phát triển bền vững hiệu quả và có tác động. Coteccons nhận thức được trách nhiệm lớn của doanh nghiệp xây dựng trên hành trình thực thi ESG vì sự phát triển bền vững của nền kinh tế - xã hội và tương lai của cộng đồng. Theo chiến lược trên, Coteccons xác định 5 mục tiêu quan trọng trong hệ sinh thái ESG b"}\n{"id": "B-A-7a4de4483a2d8a23", "claim": "Với hơn 27 năm kinh nghiệm trong lĩnh vực HSE tại nhiều công ty và tập đoàn đa quốc gia, Ông Trúc đã chứng tỏ khả năng lãnh đạo xuất sắc và cam kết mạnh mẽ đối với việc tạo ra môi trường làm việc an toàn và bền vững.", "evidence": "Coteccons đã không còn phụ thuộc quá sâu vào lĩnh vực BĐS dân dụng trong nước nữa. Đây không chỉ là nền tảng cho sự tăng trưởng an toàn, ổn định mà còn là tiền đề để bước tiến ra thị trường nước ngoài thuận tiện hơn nhờ sự kết nối và kinh nghiệm làm việc với những tập đoàn toàn cầu. Việc thắng thầu những dự án FDI lớn nhất cũng minh chứng thuyết phục cho sự trưởng thành của đội ngũ Coteccons. Coteccons nâng cao mức độ an toàn từ chính công đoạn phòng tránh. Điều này được thực hiện thông qua đào tạo về kỹ năng và nhận thức, đưa văn hóa tự quản lý, tự lãnh đạo về thực hiện an toàn lan tỏa vào toàn hệ thống. Trong năm qua, Ban HSE (Sức khỏe - An toàn - Môi trường) của Coteccons đã tiến hành hơn"}\n{"id": "B-A-3a79b53ac05df21f", "claim": "Không chỉ đưa Coteccons quay lại đỉnh cao, kết quả đó còn mang đến động lực lớn và mở ra triển vọng mới cho hệ sinh thái và mạng lưới kinh doanh xung quanh Công ty.", "evidence": "Vị thế mới trên lĩnh vực FDI đưa Coteccons trở thành điểm kết nối quan trọng về kinh tế - công nghệ với các nhà đầu tư – tập đoàn quốc tế lớn, góp phần đáp ứng nhu cầu và tiêu chuẩn của họ để đón làn sóng đầu tư nước ngoài vào Việt Nam. Những bước đi mới tiến ra thị trường quốc tế đã góp phần thúc đẩy những triển vọng mới không chỉ cho Công ty mà cho nhiều bên liên quan, mở rộng chuỗi giá trị và mạng lưới hoạt động của nhiều lĩnh vực liên quan đến ngành xây dựng trong nước. 81 80 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Dưới góc nhìn của tôi, nếu kết quả kinh doanh không mang lại lợi ích thực sự cho con người thì đó"}\n{"id": "B-A-13fec83b8dcdf249", "claim": "Coteccons trở thành một thương hiệu hướng tới giá trị con người, nâng tầm tiêu chuẩn ngành xây dựng về môi trường, phục vụ cho lợi ích hài hòa của tất cả bên liên quan, phát triển bền vững vì cộng đồng và tương lai của xã hội.", "evidence": "Coteccons định hình lại toàn bộ hệ giá trị, tiêu chí, tầm nhìn, sứ mệnh, tuyên ngôn. Trong đó, Coteccons phải là một thương hiệu hướng tới giá trị con người, phát triển bền vững vì cộng đồng, xây dựng tương lai của xã hội. Đây là cách phát triển một doanh nghiệp chân chính vì lợi ích của tất cả bên liên quan. Đây là sứ mệnh nâng cao tiêu chuẩn của toàn ngành xây dựng, khởi xướng và đóng góp giá trị đích thực cho sự phát triển kinh tế - xã hội, phục vụ cho lợi ích dài hạn của cộng đồng. 6 CAM KẾT CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DỰA TRÊN KHUNG ESG VỮNG CHẮC XUYÊN SUỐT TRONG QUÁ TRÌNH VẬN HÀNH CỦA CÔNG TY để giảm thiểu tác động môi trường, nâng cao trách nhiệm xã hội và quản trị minh bạch, tạo g"}\n{"id": "B-A-0ec3cfe0d155081f", "claim": "Năng lực, lợi thế đã có và chiến lược thực hành ESG về chiều sâu là điều kiện để Coteccons hướng tới thị trường vốn xanh trong giai đoạn tiếp theo.", "evidence": "Coteccons đặt nhiệm vụ tiếp tục triển khai ESG cả về chiều rộng và chiều sâu để nâng cao lợi thế cạnh tranh trong lĩnh vực xây dựng công nghiệp và phân khúc khách hàng FDI. Đây cũng là điều kiện tiên quyết để Công ty củng cố tầm ảnh hưởng trên vị thế industry leader thay cho cuộc chạy đua với trị trí market leader. Tại Coteccons, Tiến sĩ Bích đã đóng góp quan trọng vào các dự án phát triển bền vững và tư vấn chiến lược ESG cho giai đoạn 2023 – 2027, dựa trên các tiêu chuẩn quốc tế. Những nỗ lực của bà đã giúp Coteccons đạt được nhiều thành tựu trong cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng những cải thiện trong chuỗi cung ứng, và sử dụng nguồn nhân lực. Bà cũng "}\n{"id": "B-A-d097d84cf51f72d2", "claim": "Có thể kể đến một số dự án điển hình cho sức cạnh tranh và vị thế của Coteccons từ năng lực đáp ứng các tiêu chuẩn quốc tế khắt khe về ESG như Nhà máy LEGO Vietnam (được xây dựng tại Bình Dương, có tổng vốn đầu tư 1,4 tỷ USD), Suntory PepsiCo (Long An, 300 triệu USD), Pandora", "evidence": "PepsiCo (được xây dựng tại Long An, tổng đầu tư 300 triệu USD), Pandora (Bình Dương, 150 triệu USD), LOGOS (Bắc Ninh), BWID (Bắc Ninh), Mapletree Logistics Park 2 (Bắc Giang)... Trong số đó, Pandora là nhà máy có diện tích lớn nhất ở Đông Nam Á đáp ứng tiêu chuẩn LEED, LEGO là nhà máy trung hoà carbon đầu tiên của Tập đoàn LEGO trên toàn thế giới, Suntory PepsiCo là nhà máy áp dụng công nghệ xanh hiện đại nhất khu vực Châu Á - Thái Bình Dương... Các dự án này đều đòi hỏi hồ sơ dự thầu phải chứng minh được khả năng thực thi ESG của doanh nghiệp - bao gồm cả kiểm tra và phỏng vấn trực tiếp. Những sắc thái cảm xúc công chúng liên tưởng khi nghe nói hoặc nhìn thấy thương hiệu"}\n{"id": "B-A-e7a2bf1e2a612e56", "claim": "Trong số đó, Pandora là nhà máy có diện tích lớn nhất ở Đông Nam Á đáp ứng tiêu chuẩn LEED, LEGO là nhà máy trung hoà carbon đầu tiên của Tập đoàn LEGO trên toàn thế giới, Suntory PepsiCo là nhà máy áp dụng công nghệ xanh hiện đại nhất khu vực Châu Á - Thái Bình Dương...", "evidence": "Mục tiêu cần đạt được đầu tiên là xây dựng nhà máy trung hòa carbon đầu tiên trên thế giới của Tập đoàn LEGO, với các tiêu chuẩn Leed Platinium, Energy Center. Đây sẽ là nhà máy bền vững nhất của LEGO trên thế giới về mặt thiết kế và xây dựng với trang thiết bị hiện đại, sử dụng hoàn toàn năng lượng mặt trời, mang đến 4.000 cơ hội việc làm trong vòng 15 năm tới... Có thể kể đến một số dự án điển hình cho sức cạnh tranh và vị thế của Coteccons từ năng lực đáp ứng các tiêu chuẩn quốc tế khắt khe về ESG như Nhà máy LEGO Vietnam (được xây dựng tại Bình Dương, có tổng vốn đầu tư 1,4 tỷ USD), Suntory PepsiCo (Long An, 300 triệu USD), Pandora"}\n{"id": "B-A-bea572fdc2ba1006", "claim": "Trong nỗ lực xây dựng chuỗi cung ứng lành mạnh (lồng ghép tiêu chuẩn ESG trong thang điểm đánh giá, lựa chọn nhà cung cấp), Coteccons đã đánh giá được 53 nhà cung cấp đạt tiêu chuẩn xanh, đã gởi bảng thẻ điểm cho nhà cung cấp để làm cơ sở đánh giá tiêu chuẩn ESG.", "evidence": "Đồng thời, Khối Thu mua đã thực hiện triển khai đánh giá ESG các nhà cung cấp để xác định chính sách và bước phát triển ESG của từng nhà cung cấp, đưa thành tiêu chí đánh giá lựa chọn nhà cung cấp. Đối với các nhà cung cấp chưa có khái niệm về ESG, việc đánh giá và trao đổi thông tin cũng giúp họ tiếp cận để hiểu hơn. Qua đó, các nhà cung cấp này bắt đầu triển khai các hoạt động liên quan để cùng nhau nâng cấp tiêu chuẩn sản phẩm - dịch vụ, mang lại giá trị phát triển bền vững và có trách nhiệm cho xã hội. Tại Coteccons, Tiến sĩ Bích đã đóng góp quan trọng vào các dự án phát triển bền vững và tư vấn chiến lược ESG cho giai đoạn 2023 – 2027, dựa trên các tiêu chuẩn quốc tế. Những nỗ lực của b"}\n{"id": "B-A-bc1cc99b0d9843f0", "claim": "Khối Thu mua luôn đặt mục tiêu cân bằng quyền lợi giữa công ty và các đối tác để đảm bảo sự đồng hành lâu dài và bền vững.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons CÁC TIÊU CHÍ MUA SẮM CHÍNH TRỰC VÀ MINH BẠCH Với tinh thần “minh bạch trong mọi hoạt động”, Khối Thu mua luôn đảm bảo giá mua luôn là giá cạnh tranh nhất với sự tham gia của nhiều nhà cung cấp. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhà máy Suntory PepsiCo do Coteccons thi công tại Long An Để đảm bảo rằng đang đạt được tiến bộ hướng tới các mục tiêu bảo tồn nước, Công ty đo lường lượng nước tiêu thụ trong các hoạt động xây dựng. Điều này giúp Công ty có thể xác định các lĩnh vực có thể cải thiện và theo dõi tiến trình hướng tới các mục tiêu bền vững."}\n{"id": "B-A-dcf93616554deeb8", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons SDG 13.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 02 MỞ RỘNG THỰC THI NGUYÊN TẮC 3R SDG 12. KINH TẾ TUẦN HOÀN TRONG XÂY DỰNG Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-9642a14ec6a5781f", "claim": "Các đặc thù về văn hóa, xã hội, địa lí luôn được quan tâm trong quá trình lập chính sách về môi trường, các qui định về an toàn và tuân thủ về bảo vệ môi trường trong quá trình thực hiện thi công các công trình nhằm mục tiêu đảm bảo cho sự phát triển bền vững, xây dựng thương hiệu và đặt niềm tin cho khách hàng, các tổ chức đầu tư trong và ngoài nước.", "evidence": "Coteccons hướng tới mục tiêu trở thành và được công nhận là Industry Leader, là hình mẫu trong cam kết thực hành ESG trong lĩnh vực Xây dựng, mang đến các giá trị bền vững đóng góp vào sự phát triển kinh tế, xã hội và bảo vệ môi trường bằng cách kinh doanh có đạo đức và trách nhiệm thông qua Quản trị Tốt. Cơ cấu tổ chức Quản trị Phát triển bền vững được thiết lập nhằm đảm bảo việc thực thi Phát triển bền vững hiệu quả và có tác động. 1. TUÂN THỦ LÀ YÊU CẦU HÀNG ĐẦU CHO MỌI HOẠT ĐỘNG TẠI CÔNG TRƯỜNG Hiện tại, tất cả công trường của Coteccons đều thấm nhuần nhiệm vụ bảo vệ môi trường và thực thi phát triển bền vững. Để đảm bảo tuân thủ pháp luật về môi trường và đạt các tiêu chuẩn quốc tế, qua"}\n{"id": "B-A-be02c03b9a7565d5", "claim": "Theo ước tính, ngành này sử dụng khoảng 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và sử dụng lượng lớn lực lượng lao động.", "evidence": "Quan trọng hơn, Coteccons là một trong những doanh nghiệp xây dựng hàng đầu. Xây dựng là hoạt động đặc thù có nhiều tác động với thế giới. Theo ước tính, ngành này tiêu thụ khoảng 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và sử dụng một lượng lao động khổng lồ. Theo ước lượng, ngành xây dựng sử dụng đến 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và liên quan đến lượng lao động rất lớn. Đặc biệt, sản phẩm mà ngành xây dựng tạo ra có vai trò sâu rộng và lâu dài với xã hội - cộng đồng. Như vậy, ngành xây dựng và doanh nghiệp xây dựng chắc chắn phải có trách nhiệm, phải nghiêm khắc với tiêu chuẩn"}\n{"id": "B-A-5847697bf6d3263f", "claim": "Mục tiêu hàng đầu của Coteccons: Giảm lượng phát thải trong hoạt động xây dựng thông qua sử dụng năng lượng hiệu quả, năng lượng thay thế.", "evidence": "Quan trọng hơn, Coteccons là một trong những doanh nghiệp xây dựng hàng đầu. Xây dựng là hoạt động đặc thù có nhiều tác động với thế giới. Theo ước tính, ngành này tiêu thụ khoảng 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và sử dụng một lượng lao động khổng lồ. Các đặc thù về văn hóa, xã hội, địa lí luôn được quan tâm trong quá trình lập chính sách về môi trường, các qui định về an toàn và tuân thủ về bảo vệ môi trường trong quá trình thực hiện thi công các công trình nhằm mục tiêu đảm bảo cho sự phát triển bền vững, xây dựng thương hiệu và đặt niềm tin cho khách hàng, các tổ chức đầu tư trong và ngoài nước. Xây dựng là ngành đặc thù có nhiề"}\n{"id": "B-A-6e2136e208f99a02", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Kết quả ghi nhận tất cả dự án của Coteccons đều được kiểm soát môi trường theo đúng tiêu chuẩn quy định.", "evidence": "Kết quả ghi nhận tất cả dự án của Coteccons đều được kiểm soát môi trường theo đúng tiêu chuẩn quy định. Tính tuân thủ pháp luật được xác nhận qua các lần quan trắc môi trường bởi đơn vị thứ 3 có năng lực và các lần thanh kiểm tra của cơ quan nhà nước. Tuân thủ chặt chẽ tiêu chuẩn và quy định pháp luật cũng chính là lý do giúp Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons THỰC HÀNH ESG THEO 5 MỤC TIÊU CỦA COTECCONS Coteccons triển khai thực hành ESG một cách quyết liệt trong nội bộ, đồng thời tìm kiếm, xây dựng quan hệ với các đối tác chung mục tiêu phát triển bền vững."}\n{"id": "B-A-95c44e9eec0c1177", "claim": "Nghị định hướng dẫn Luật Bảo vệ môi trường năm 2020 bao gồm: Nghị định 06/2022/NĐ-CP quy định về giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn.", "evidence": "Nghị định 08/2022/NĐ-CP hướng dẫn Luật Bảo vệ môi trường. Nghị định 45/2022/NĐ-CP quy định về xử phạt vi phạm hành chính trong lĩnh vực bảo vệ môi trường. Nghị định 27/2023/NĐ-CP quy định phí bảo vệ môi trường đối với khai thác khoáng sản. Washington, D.C.: U.S. Green Building Council. VIỆT NAM Luật Bảo vệ môi trường 2020 Thông tư 155/2015/ TT-BTC: Trách nhiệm công bố thông tin của công ty, công bố thông tin kinh doanh, công bố báo cáo tài chính định kỳ Nghị định 06/2022/ NĐ-CP Quy định giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn"}\n{"id": "B-A-fbb13ad601b0cd6a", "claim": "Thực hiện thu gom và xử lý các chất thải/đất/bùn thải từ quá trình thi công của mình.", "evidence": "1. Phương án giảm thiểu bụi, ồn 2. Thu gom và xử lý nước thải 3. Thu gom và xử lý rác thải/ bùn thải 4. Đăng ký xả thải vào hệ thống thu gom chính quy địa phương Coteccons Building. Khối công trường cũng áp dụng tối đa nguyên tắc 3R trong quá trình xây dựng, qua tất cả công đoạn, từ chọn giải pháp, kết cấu, hoàn thiện, cơ điện... Một số sáng kiến nổi bật thực hiện chỉ tiêu 3R của các công trường như tái chế vật liệu từ chất thải cơ điện, kim loại từ dây điện, sử dụng triệt để sơn nước thừa từ các công trình hoàn thiện cho dự án thiện nguyện “Nhà mình”, dọn dẹp vệ sinh, thu gom rác thải định kỳ với chương trình Green Day..."}\n{"id": "B-A-8c534982440872ec", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 2.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 17 16 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-5552e55461a41467", "claim": "Các công trường của Coteccons, đặc biệt là các công trường thuộc dự án FDI như công trường LEGO Vietnam, Suntory PepsiCo, luôn là những hình mẫu tuân thủ các yêu cầu về bảo vệ môi trường, thực hiện các chương trình tiết kiệm năng lượng, tài nguyên và sử dụng năng lượng tái tạo.", "evidence": "Coteccons còn hướng đến trở thành nhà đầu tư của các dự án năng lượng tái tạo. Điều này thể hiện quyết tâm của Công ty trong việc thúc đẩy nền kinh tế xanh tại Việt Nam. Sử dụng hệ thống điện năng lượng mặt trời: Suntory PepsiCo, BAN CHỈ HUY CÔNG TRƯỜNG LEGO Nam 1 Tất cả công trường của Coteccons đều tuân thủ nghiêm ngặt yêu cầu và tiêu chuẩn ESG do Công ty ban hành. Trong đó, Công ty tuyên dương Ban chỉ huy 7 công trường như danh sách dưới đây về sự tham gia thực thi hiệu quả và đóng góp tích cực nhất. Pandora Nam 2 Suntory PepsiCo Nam 3 Urban Green"}\n{"id": "B-A-010f8161dd6b94db", "claim": "- Tiết kiệm nước, tái sử dụng nước trong việc rửa bánh xe ra khỏi công trường.", "evidence": "Một cách khác để Công ty giảm lượng nước sử dụng là kiến tạo cảnh quan tiết kiệm nước. Bằng cách chọn những loại cây cần ít nước hơn và thiết kế cảnh quan hợp lý để giảm thiểu lượng nước chảy ra ngoài, Công ty có thể giảm đáng kể lượng nước sử dụng để duy trì các cơ sở của mình. Đồng thời, Coteccons cũng chú trọng việc tiết kiệm nước và tận tâm thực hiện yêu cầu này trong phạm vi của mình để giảm tác động từ hoạt động của Công ty đối với môi trường. Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước. Để đảm bảo kh"}\n{"id": "B-A-559c25b3e15e3796", "claim": "Đây là ISO Quản lí chiến lược được xác nhận của Quốc Tế về quản lí môi trường (EMP), cung cấp khung về tổ chức để thiết kế và thực hiện theo hệ thống quản lí môi trường (EMS), đánh giá vòng đời sản phẩm, sinh thái, xác định và kiểm kê khí nhà kính.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 2. XÂY DỰNG KẾ HOẠCH MÔI TRƯỜNG THEO CHUẨN QUỐC TẾ 3. TIẾT KIỆM NĂNG LƯỢNG (ISO 14001) Coteccons hiện đang tuân thủ tiêu chuẩn môi trường ISO 14001:2015 về hệ thống quản lý môi trường. Coteccons đã chủ động tự nguyện tham gia vào hoạt động kiểm kê và giảm phát thải khí nhà kính, hướng tới phát triển bền vững và nâng cao hình ảnh doanh nghiệp. Điều này cũng tăng cường sức thuyết phục của Coteccons đối với các nhà đầu tư, khẳng định khả năng tạo giá trị gia tăng thông qua việc kiểm soát và giảm thiểu khí nhà kính. Coteccons đã triển khai công tác kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-"}\n{"id": "B-A-aa9e731581621340", "claim": "Tất cả vì mục tiêu chung là giảm thiểu tối đa việc ảnh hưởng đến môi trường ở mọi khu vực có dự án do Coteccons triển khai, giảm thiểu phát thải khí nhà kính và hòa mình vào đề án phát triển bền vững của Việt Nam.", "evidence": "Coteccons đã chủ động tự nguyện tham gia vào hoạt động kiểm kê và giảm phát thải khí nhà kính, hướng tới phát triển bền vững và nâng cao hình ảnh doanh nghiệp. Điều này cũng tăng cường sức thuyết phục của Coteccons đối với các nhà đầu tư, khẳng định khả năng tạo giá trị gia tăng thông qua việc kiểm soát và giảm thiểu khí nhà kính. Coteccons đã triển khai công tác kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064- Trong năm qua, Công ty không ghi nhận bất kỳ vi phạm nào phải xử lý hành chính và không có kiến nghị vượt cấp liên quan đến vấn đề môi trường. Cụ thể, tất cả dự án của Coteccons đều ghi nhận giảm được lượng nước thải phát sinh, giảm được cường độ ồn/bụi ảnh hưởng đến khu vực xung quanh"}\n{"id": "B-A-901905338737d518", "claim": "Các công trình tại đây cũng đã triển khai hệ thống mái xanh với các loài cây bản địa trên khu vực bãi đỗ xe máy, với tổng diện tích 1.383,79m2, nhằm cải thiện chất lượng không khí và giảm hiệu ứng nhiệt đô thị.", "evidence": "TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon"}\n{"id": "B-A-d6538fb357040d93", "claim": "Những sáng kiến này thể hiện cam kết của khách hàng của chúng tôi đối với phát triển bền vững và bảo vệ môi trường trong suốt vòng đời dự án.", "evidence": "TÂM HUYẾT KHÁCH HÀNG LÀ TRỌNG TÂM Chúng tôi truyền cảm hứng và tạo động lực cho tất cả mọi người, bằng sự kiên định và cam kết với mục tiêu phát triển để được công nhận trên toàn thế giới. Chúng tôi đặt lợi ích của khách hàng là trọng tâm trong mọi quyết định và hoạt động kinh doanh. CHU ĐÁO BỀN VỮNG Chúng tôi thấu hiểu và nỗ lực đem đến những lợi ích cho tất cả các bên; Chúng tôi hoạt động với Chúng tôi cam kết hoạt động qua giới hạn bản thân và cùng trách nhiệm bảo vệ môi với sự trung thực và minh bạch. tạo nên sự vượt trội. trường và phát triển bền vững, vì thế hệ tương lai. 33 32 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-959798077d1709ca", "claim": "Theo Nghị định số 06/2022/NĐ-CP về giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn, các doanh nghiệp phải xây dựng kế hoạch giảm phát thải phù hợp với mục tiêu quốc gia.", "evidence": "Coteccons trước hết tuân thủ các quy định của Luật Bảo vệ môi trường và dựa theo hệ thống các nghị định hướng dẫn về các yêu cầu cụ thể của luật để thực hiện. Nghị định hướng dẫn Luật Bảo vệ môi trường năm 2020 bao gồm: Nghị định 06/2022/NĐ-CP quy định về giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn. Washington, D.C.: U.S. Green Building Council. VIỆT NAM Luật Bảo vệ môi trường 2020 Thông tư 155/2015/ TT-BTC: Trách nhiệm công bố thông tin của công ty, công bố thông tin kinh doanh, công bố báo cáo tài chính định kỳ Nghị định 06/2022/ NĐ-CP Quy định giảm nhẹ phát thải khí nhà kính và bảo vệ tầng ô-dôn"}\n{"id": "B-A-4955515269c1acf5", "claim": "Coteccons đã chủ động tự nguyện tham gia vào hoạt động kiểm kê và giảm phát thải khí nhà kính, hướng tới phát triển bền vững và nâng cao hình ảnh doanh nghiệp.", "evidence": "Báo cáo và phân tích dữ liệu lượng phát thải của doanh nghiệp. Đặc biệt, Công ty đã xây dựng được một đội ngũ nhân sự có kỹ năng cơ bản trong kiểm kê khí nhà kính, có khả năng thu thập và phân tích dữ liệu, sử dụng thành thạo phần mềm quản lý khí nhà kính, có kiến thức về quy trình sản xuất và quản lý năng lượng... để tiếp tục hành trình giảm phát thải, hướng tới tương lai bền vững. Bảo vệ và tăng cường tài sản cũng như hình ảnh doanh nghiệp. Phát triển và hỗ trợ nguồn nhân lực và nền tảng tri thức của doanh nghiệp. 57 56 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-1c9c2a8747b8a825", "claim": "Điều này cũng tăng cường sức thuyết phục của Coteccons đối với các nhà đầu tư, khẳng định khả năng tạo giá trị gia tăng thông qua việc kiểm soát và giảm thiểu khí nhà kính.", "evidence": "Việc Coteccons được công nhận trong Fortune Southeast Asia 500 một lần nữa khẳng định vị thế dẫn đầu, đại diện cho ngành xây dựng Việt Nam. Đồng thời cũng khẳng định kết quả kinh doanh ấn tượng và sức mạnh tài chính của Coteccons thông qua các công bố và thẩm định minh bạch. Đặc biệt, kết quả này được công bố vào thời điểm Đồng thời cũng khẳng định kết quả kinh doanh ấn tượng và sức mạnh tài chính của Coteccons thông qua các công bố và thẩm định minh bạch. Đặc biệt, kết quả này được công bố vào thời điểm Coteccons chuẩn bị kết thúc năm tài chính vào ngày"}\n{"id": "B-A-343ba79117539679", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Do đặc thù của ngành xây dựng yêu cầu sự giám sát chặt chẽ và liên tục ở từng giai đoạn, nên trong giai đoạn tiếp theo của công tác kiểm kê khí nhà kính, Coteccons sẽ tập trung vào việc phân tích chi tiết hơn các kết quả kiểm kê, đồng thời nghiên cứu và triển khai các giải pháp giảm phát thải hiệu quả.", "evidence": "tiết giảm CO2 ra môi trường Tiếp tục đánh giá và phân tích kết quả kiểm kê, các yếu tố tác động đến mức phát thải, hoàn thiện các chỉ số nền (baseline) đối với công trình xây dựng theo giai đoạn để có thể sử dụng làm cơ sở xây dựng các chỉ tiêu và so sánh với các năm tiếp theo; 34.429tấn Thu thập và lưu trữ số liệu kiểm kê khí nhà kính, được phân loại chi tiết theo từng loại hình dự án, tiến độ và giai đoạn cụ thể, giúp dễ dàng truy xuất và cung cấp thông tin tổng hợp cho các bên liên quan. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons CHÍNH SÁCH VỀ MÔI TRƯỜNG CỦA COTECCONS 1 2 3 4 5 Giảm phát thải khí nhà kính bằng cách sử dụng nguyên nhiên liệu, thiết bị h"}\n{"id": "B-A-5ebee5cfabbaf2b6", "claim": "Cụ thể, các hoạt động này bao gồm: vững - thép xanh (tương đương 34.429 tấn) tại 17 công trường từ các nhà cung cấp có chứng nhận LEED, EPD,", "evidence": "Một phần số liệu kiểm kê đã được sử dụng để đáp ứng yêu cầu phát thải của chứng chỉ LEED cho các dự án tiêu biểu của Coteccons. Tính toán tự động lượng phát thải khí nhà kính các hoạt động trong khối văn phòng, dự án và công trình. vật liệu bền vững thép xanh có chứng nhận LEED, EPD, LCA Thi Công Thi Công ĐÁNH GIÁ TÁC ĐỘNG MÔI TRƯỜNG CỦA CHỦ ĐẦU TƯ Phế Phẩm/rác Tiêu chuẩn Sắt thép CÁC YÊU CẦU KHÁC TIÊU CHUẨN XANH KIỂM (LEED, LOTUS...) Tập Kết"}\n{"id": "B-A-e547a6f75416343f", "claim": "Coteccons là doanh nghiệp xây dựng tiên phong triển khai hệ thống kiểm kê các nguồn phát thải trong chuỗi hoạt động và định lượng lượng phát thải.", "evidence": "Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất và phát huy đổi mới sáng tạo trong hoạt động giảm phát thải; Lưu trữ và quản lý dữ liệu lượng phát thải của Coteccons, dành cho cả những công trình, dự án đã hoàn thiện. Thiết lập biểu mẫu giám sát khí nhà kính tại mỗi công trường, ngay khi bắt đầu triển khai hoạt động xây dựng, đảm bảo quy trình theo dõi chặt chẽ và minh bạch. Mục tiêu hàng đầu của Coteccons: Giảm lượng phát thải trong hoạt động xây dựng thông qua sử dụng năng lượng"}\n{"id": "B-A-03eb166a03a138e4", "claim": "Công ty áp dụng công cụ kiểm kê khí nhà kính trên website https://co2.coteccons.", "evidence": "Coteccons đã chủ động tự nguyện tham gia vào hoạt động kiểm kê và giảm phát thải khí nhà kính, hướng tới phát triển bền vững và nâng cao hình ảnh doanh nghiệp. Điều này cũng tăng cường sức thuyết phục của Coteccons đối với các nhà đầu tư, khẳng định khả năng tạo giá trị gia tăng thông qua việc kiểm soát và giảm thiểu khí nhà kính. Coteccons đã triển khai công tác kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064- Phối hợp xác minh, kiểm tra chéo, đảm bảo tính chính xác của dữ liệu Tính toán, rà soát, kiểm tra kết quả Công bố đến các cơ quan hữu quan và đơn vị truyền thông Đăng tải trên website chính thức của Công ty tại www.coteccons.vn, tiểu mục ESG"}\n{"id": "B-A-8186013661a1400a", "claim": "Một phần số liệu kiểm kê đã được sử dụng để đáp ứng yêu cầu phát thải của chứng chỉ LEED cho các dự án tiêu biểu của Coteccons.", "evidence": "Hiện nay, giá trị thương hiệu của Coteccons vào năm 2024 theo bảng tự đánh giá theo phương pháp Isolated Brand Valuation© từ website Brand Equity Check®, thương hiệu Coteccons có giá trị khoảng 377 - 460 triệu USD, tăng gấp 10 lần so với năm 2015. Một minh chứng khác cho khả năng đáp ứng yêu cầu cao về tiêu chuẩn ESG của Coteccons là số lượng dự án đạt chứng chỉ LEED và LOTUS mà Công ty đã hoàn thành. Đến nay, Coteccons có 46 dự án trong số 305 công trình của cả nước đạt các chứng chỉ này. Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chu"}\n{"id": "B-A-efa8e89b8b67a21a", "claim": "Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất và phát huy đổi mới sáng tạo trong hoạt động giảm phát thải;", "evidence": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng tro"}\n{"id": "B-A-02e69ccff211d699", "claim": "Lưu trữ và quản lý dữ liệu lượng phát thải của Coteccons, dành cho cả những công trình, dự án đã hoàn thiện.", "evidence": "Báo cáo và phân tích dữ liệu lượng phát thải của doanh nghiệp. Đặc biệt, Công ty đã xây dựng được một đội ngũ nhân sự có kỹ năng cơ bản trong kiểm kê khí nhà kính, có khả năng thu thập và phân tích dữ liệu, sử dụng thành thạo phần mềm quản lý khí nhà kính, có kiến thức về quy trình sản xuất và quản lý năng lượng... để tiếp tục hành trình giảm phát thải, hướng tới tương lai bền vững. Triết lý này đã được thể hiện bằng kết quả thực tế thông qua trên 500 dự án Coteccons đã hoàn thành cho khách hàng. Tất cả công trình mang dấu ấn Coteccons đều đảm bảo chất lượng và giá trị bền vững. Phần lớn trong số đó đã trở thành những biểu tượng cho diện mạo của các đô thị, đóng vai trò quan trọng trong đời "}\n{"id": "B-A-ffa7f074937e3344", "claim": "Thiết lập biểu mẫu giám sát khí nhà kính tại mỗi công trường, ngay khi bắt đầu triển khai hoạt động xây dựng, đảm bảo quy trình theo dõi chặt chẽ và minh bạch.", "evidence": "NĐT, ảnh hưởng đến uy tín của Công ty. Xây dựng các chính sách, quy trình nội bộ và cơ chế giám sát phù hợp với quy định của luật pháp và thông lệ quốc tế để đảm bảo sự minh bạch trong giao dịch với các bên liên quan. Hiệu quả tài chính của dự án chắc chắn sẽ bị suy giảm bởi các vấn đề về dòng tiền do quá trình thanh toán, quyết toán chậm từ chủ đầu tư. Quy định chặt chẽ và đàm phán kỹ các điều khoản liên quan đến thanh toán trong hợp đồng ngay trong quá trình làm hồ sơ đấu thầu. Do đó, tiêu chuẩn và mục tiêu về sức khỏe và an toàn tại Coteccons được liên tục nâng cấp để vượt trên mức tuân thủ. Quy trình thực thi được đặt dưới hệ thống giám sát tập trung và hiệu suất chính xác cao. Trước khi"}\n{"id": "B-A-ca112c7c4d4b27a1", "claim": "Đặc biệt, Công ty đã xây dựng được một đội ngũ nhân sự có kỹ năng cơ bản trong kiểm kê khí nhà kính, có khả năng thu thập và phân tích dữ liệu, sử dụng thành thạo phần mềm quản lý khí nhà kính, có kiến thức về quy trình sản xuất và quản lý năng lượng...", "evidence": "tiết giảm CO2 ra môi trường Tiếp tục đánh giá và phân tích kết quả kiểm kê, các yếu tố tác động đến mức phát thải, hoàn thiện các chỉ số nền (baseline) đối với công trình xây dựng theo giai đoạn để có thể sử dụng làm cơ sở xây dựng các chỉ tiêu và so sánh với các năm tiếp theo; 34.429tấn Thu thập và lưu trữ số liệu kiểm kê khí nhà kính, được phân loại chi tiết theo từng loại hình dự án, tiến độ và giai đoạn cụ thể, giúp dễ dàng truy xuất và cung cấp thông tin tổng hợp cho các bên liên quan. “Xây Tết” Dự án của Công ty áp dụng phần mềm kiểm kê khí nhà kính Số lượng cây được trồng từ các chương trình trồng rừng thắng thầu mới 159.986 100% 27.473 Doanh thu thuần Tăng"}\n{"id": "B-A-337a7d23c3462167", "claim": "Trên thực tế, Coteccons đã tiến hành triển khai một số biện pháp như giảm thiểu dấu chân carbon và lựa chọn sử dụng các sản phẩm có chứng chỉ LCA và EPD đến từ hơn 57 đối tác thương hiệu uy tín có tác động môi trường thấp hơn trong toàn bộ vòng đời.", "evidence": "NGUYÊN TẮC 3R TRONG XÂY DỰNG Ở COTECCONS Coteccons thúc đẩy các sáng kiến xây dựng xanh, trong đó ứng dụng công nghệ tiên tiến để chỉ tác động tối thiểu đến môi trường trong tất cả giai đoạn của vòng đời của một công trình, từ thiết kế, xây dựng, vận hành, bảo trì và cải tạo cho đến phá dỡ. Cách này đồng thời cũng giảm thiểu chi phí trong công tác vận hành của doanh nghiệp, như giảm được áp lực nhân công và và tác động các cộng đồng lân cận của các dự án cả hiện tại và cả tương lai. Theo đó, Top 10 nơi làm việc tốt nhất được lựa chọn dựa trên các nhóm tiêu chí liên quan đến nhân sự và uy tín nhà tuyển dụng, hiệu quả sản xuất kinh doanh và triển vọng tăng trưởng, thu nhập và chế độ phúc lợi.."}\n{"id": "B-A-f370a74b1149de8f", "claim": "Theo ước tính ban đầu, việc sử dụng bê tông khoáng hóa CO2 (bê tông Fico) với khối lượng 132.603 tấn tiết giảm được 1.936 tấn", "evidence": "Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon trong giai đoạn xây dựng. CO2 ra môi trường trong năm tài chính vừa qua. Cùng với việc ưu tiên sử dụng vật liệu gạch không nung cũng góp phần tiết giảm được 926 tấn CO2, tiết kiệm được 171 tấn dầu tương đương. Song song, 26% dòng vật liệu bền Đặc biệt, hệ thống dữ liệu trên phần mềm được hiển thị trực quan hóa bằng các biểu đồ, từ đó, dễ dàng so sánh, đánh giá các xu hướng phát thải để đưa ra quyết định đúng đắn và chiến lược phù hợp với mức độ phát thải mà"}\n{"id": "B-A-263c06c844931485", "claim": "Cùng với việc ưu tiên sử dụng vật liệu gạch không nung cũng góp phần tiết giảm được 926 tấn CO2, tiết kiệm được 171 tấn dầu tương đương.", "evidence": "Trên thực tế, Coteccons đã tiến hành triển khai một số biện pháp như giảm thiểu dấu chân carbon và lựa chọn sử dụng các sản phẩm có chứng chỉ LCA và EPD đến từ hơn 57 đối tác thương hiệu uy tín có tác động môi trường thấp hơn trong toàn bộ vòng đời. Theo ước tính ban đầu, việc sử dụng bê tông khoáng hóa CO2 (bê tông Fico) với khối lượng 132.603 tấn tiết giảm được 1.936 tấn Việc ưu tiên sử dụng các vật liệu dễ phân hủy và tái sử dụng, cùng với việc lựa chọn nhà cung cấp bền vững không chỉ giảm thiểu tác động đến môi trường mà còn tạo ra một không gian làm việc thân thiện và thoải mái hơn. Thêm nữa, việc phủ xanh văn phòng và sử dụng dịch vụ Taxi Xanh đã góp phần thể hiện cam kết của chúng ta "}\n{"id": "B-A-2380a71f9b2b16aa", "claim": "Đơn cử, có thể tính toán được việc trồng bao nhiêu cây thân gỗ để hấp thụ lại lượng khí nhà kính đã thải ra, nhờ đó đạt trạng thái trung hòa carbon.", "evidence": "Theo ước tính, một dự án như vậy khi hoàn thành sẽ loại trừ và giảm tổng cộng 2.803 tấn carbon hàm chứa so với vật liệu thông thường, tương đương với lượng carbon được hấp thụ bởi 46.348 cây xanh trong 10 năm. Coteccons và Faslink bắt tay đưa trạm tái chế vải đầu tiên đến Việt Nam với dự án Second Life, nhằm tái sinh “cuộc đời thứ 2” cho những loại vải vi nhựa vốn cần đến 200 năm để phân hủy, hướng đến mục tiêu tận dụng lợi thế chuyên môn của mỗi bên để giảm 49% tiêu thụ nước và giảm 59% khí thải carbon. Quan trọng hơn, Coteccons là một trong những doanh nghiệp xây dựng hàng đầu. Xây dựng là hoạt động đặc thù có nhiều tác động với thế giới. Theo ước tính, ngành này tiêu thụ khoảng 40% tổng n"}\n{"id": "B-A-405126bdbd2dcc73", "claim": "Ngoài ra, để hoàn thành được báo cáo về lượng phát thải, cần trải qua nhiều bước kiểm tra kỹ càng và gắt gao từ nội bộ Coteccons đến bên thứ 3.", "evidence": "Hệ sinh thái đa dạng của Coteccons đảm bảo năng lực phụ trách những công trình quy mô từ giai đoạn thiết kế, thi công đến hoàn thiện với kết quả tối ưu về chất lượng, thời gian và chi phí. Với thực lực liên tục được vun đắp và hiệu quả hoạt động ngày càng cao, Coteccons nhận được tín nhiệm ngày càng sâu sắc từ khách hàng và kỳ vọng lớn từ thị trường. Chính vì vậy, đối với nhiều chủ đầu tư - nhà phát triển dự án, Coteccons luôn là đối tác đáng tin cậy. Vị thế mới trên lĩnh vực FDI đưa Coteccons trở thành điểm kết nối quan trọng về kinh tế - công nghệ với các nhà đầu tư – tập đoàn quốc tế lớn, góp phần đáp ứng nhu cầu và tiêu chuẩn của họ để đón làn sóng đầu tư nước ngoài vào Việt Nam. Những b"}\n{"id": "B-A-172d48168d4bb6e0", "claim": "Những kết quả này là minh chứng cho cam kết của Coteccons trong việc tiên phong phát triển bền vững và xây dựng những công trình xanh, thân thiện với môi trường.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TRIỂN KHAI HỆ THỐNG KIỂM KÊ KHÍ NHÀ KÍNH Coteccons đã triển khai các biện pháp bền vững và thân thiện với môi trường trong suốt quá trình xây dựng nhà máy LEGO tại Bình Dương. Tại Coteccons, Tiến sĩ Bích đã đóng góp quan trọng vào các dự án phát triển bền vững và tư vấn chiến lược ESG cho giai đoạn 2023 – 2027, dựa trên các tiêu chuẩn quốc tế. Những nỗ lực của bà đã giúp Coteccons đạt được nhiều thành tựu trong cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng những cải thiện trong chuỗi cung ứng, và sử dụng nguồn nhân lực. Bà cũng đóng góp vào việc xây dựng đội ngũ nhân sự chuyên môn cao, có"}\n{"id": "B-A-e7135b4ef71a5a43", "claim": "Reporting company Upstream activities Downstream activities Hệ thống kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 Dự án Thung lũng Thanh Xuân (tỉnh Vĩnh Phúc)", "evidence": "Coteccons đã chủ động tự nguyện tham gia vào hoạt động kiểm kê và giảm phát thải khí nhà kính, hướng tới phát triển bền vững và nâng cao hình ảnh doanh nghiệp. Điều này cũng tăng cường sức thuyết phục của Coteccons đối với các nhà đầu tư, khẳng định khả năng tạo giá trị gia tăng thông qua việc kiểm soát và giảm thiểu khí nhà kính. Coteccons đã triển khai công tác kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064- Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 2. XÂY DỰNG KẾ HOẠCH MÔI TRƯỜNG THEO CHUẨN QUỐC TẾ 3. TIẾT KIỆM NĂNG LƯỢNG (ISO 14001) Coteccons hiện đang tuân thủ tiêu chuẩn môi trường ISO 14001:2015 về hệ thống quản lý môi trường."}\n{"id": "B-A-481582deda8bbd8c", "claim": "Vì lẽ đó, ngày 30/03/2024, Coteccons đã triển khai chương trình “Sạch môi trường - Xanh thế giới” cho khối văn phòng và hơn 50 công trường cả nước.", "evidence": "Thông qua chương trình “Sạch môi trường – Xanh thế giới”, Coteccons thúc đẩy nhận thức và hành động thiết thực trong việc tạo ra môi trường lao động lành mạnh hơn. Đối với khối văn phòng, ngoài hoạt động khuyến khích các cán bộ - nhân viên dọn dẹp vệ sinh khu vực làm việc hằng tuần, Coteccons còn triển khai chiến dịch dọn vệ sinh trên đoạn đường Xô Viết Nghệ Tĩnh, quận Bình Thạnh, TP.HCM với sự tham dự của hơn 40 cán bộ - cán bộ nhân viên khối văn phòng của Coteccons và tình nguyện viên là các sinh viên đến từ trường Đại học Kiến Trúc TP.HCM. giờ thực hiện Trên 50 dự án của Coteccons khắp cả nước, hoạt động tổng vệ sinh được thực hiện từ bên trong đến bên ngoài công trường. Bằng nhiệm vụ dọn"}\n{"id": "B-A-4e21260016b77030", "claim": "Hoạt động này nhằm hưởng ứng Ngày quốc tế Không rác thải (The International Day of Zero Waste) được phát động lần đầu tiên bởi Đại hội đồng Liên hợp quốc.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons HƯỞNG ỨNG NGÀY QUỐC TẾ KHÔNG RÁC THẢI Rác thải đang trở thành gánh nặng khổng lồ trên toàn thế giới. Qua đó, Công ty không ngừng cải tiến quy chế và hoạt động để đáp ứng ngày càng cao hơn nhu cầu của các bên trên các khía cạnh quản trị - xã hội và môi trường. Công ty cũng chủ động công khai thông tin của các đầu mối liên hệ chính để các bên liên quan kịp thời liên hệ khi có nhu cầu về công việc cũng như các khiếu nại cần giải quyết. Đánh giá, phản Đánh giá mức độ ưu tiên dựa trên mức độ hồi, đưa ra các hành động phù hợp nhằm đáp ứng"}\n{"id": "B-A-d046af41526f6ee0", "claim": "Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự 651 Việc thu gom rác đã góp phần trả lại mỹ quan cho tuyến đường sầm uất nối 2 quận Bình Thạnh và quận 1.", "evidence": "giờ thực hiện Trên 50 dự án của Coteccons khắp cả nước, hoạt động tổng vệ sinh được thực hiện từ bên trong đến bên ngoài công trường. Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Coteccons. Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện. Thông qua chương trình “Sạch môi trường – Xanh thế giới”, Coteccons thúc đẩy nhận thức và hành động thiết thực trong việc tạo ra môi trường lao động lành mạnh hơn. Đối với khối văn phòng, ngoài hoạt động khuyến khích các cán bộ - nhân viên dọn dẹp vệ sinh khu vực làm việc hằng"}\n{"id": "B-A-019332cba9f81596", "claim": "Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Coteccons.", "evidence": "Electric tổ chức. Tại đây, Coteccons chia sẻ tầm nhìn trong việc định hình lại hệ sinh thái xây dựng bền vững và thúc đẩy ngành xây dựng xanh, tạo ra những không gian an toàn hơn, thoải mái hơn, ý nghĩa hơn trong toàn bộ chuỗi giá trị bằng những cam kết với các tiêu chuẩn ESG và HSE. Co- teccons cũng truyền đi thông điệp sẵn sàng đồng hành cùng các đối tác có cùng tầm nhìn và giá trị, nhằm lan tỏa giá trị ESG hiệu quả và thiết thực nhất đến mọi khía cạnh của ngành. 127 126 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Thông qua chương trình “Sạch môi trường – Xanh thế giới”, Coteccons thúc đẩy nhận thức và hành động thiết thực trong việc tạo ra môi trường lao động lành mạnh hơn. Đối với khối văn phò"}\n{"id": "B-A-91ee4b6c52a9f711", "claim": "Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện.", "evidence": "Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự 651 Việc thu gom rác đã góp phần trả lại mỹ quan cho tuyến đường sầm uất nối 2 quận Bình Thạnh và quận 1. Theo báo cáo từ đơn vị cung cấp dịch vụ bảo vệ môi trường, Coteccons đã hoàn thành dọn sạch gần 6 tấn rác thải trong nửa buổi sáng tại khu vực này. Thông qua chương trình “Sạch môi trường – Xanh thế giới”, Coteccons thúc đẩy nhận thức và hành động thiết thực trong việc tạo ra môi trường lao động lành mạnh hơn. Đối với khối văn phòng, ngoài hoạt động khuyến khích các cán bộ - nhân viên dọn dẹp vệ sinh khu vực làm việc hằng tuần, Coteccons còn triển khai chiến dịch dọn vệ sinh trên đoạn đường"}\n{"id": "B-A-a7e1019a73205802", "claim": "Theo đó, thứ Sáu hàng tuần được gọi là “Green day” trên các công trường của Coteccons, người lao động sẽ cùng nhau phân loại rác, dọn dẹp ở cả công trường và các tuyến đường xung quanh.", "evidence": "giờ thực hiện Trên 50 dự án của Coteccons khắp cả nước, hoạt động tổng vệ sinh được thực hiện từ bên trong đến bên ngoài công trường. Bằng nhiệm vụ dọn dẹp vệ sinh, Ban quản lý công trường đã thúc đẩy gắn kết nội bộ, chia sẻ cùng đội ngũ kỹ sư và công nhân về tầm nhìn ESG dài hạn của Coteccons. Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự với tổng 651 giờ thực hiện. Chiến dịch “Sạch môi trường - Xanh thế giới” tại các công trình đã thu hút hơn 1.000 người tham dự 651 Việc thu gom rác đã góp phần trả lại mỹ quan cho tuyến đường sầm uất nối 2 quận Bình Thạnh và quận 1. Theo báo cáo từ đơn vị cung cấp dịch vụ bảo vệ môi trường, Coteccons đã "}\n{"id": "B-A-30826e0b63432164", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhà máy Suntory PepsiCo do Coteccons thi công tại Long An Để đảm bảo rằng đang đạt được tiến bộ hướng tới các mục tiêu bảo tồn nước, Công ty đo lường lượng nước tiêu thụ trong các hoạt động xây dựng.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons THỰC HÀNH ESG THEO 5 MỤC TIÊU CỦA COTECCONS Coteccons triển khai thực hành ESG một cách quyết liệt trong nội bộ, đồng thời tìm kiếm, xây dựng quan hệ với các đối tác chung mục tiêu phát triển bền vững. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TRẢI NGHIỆM MỚI TỪ CÁC CHƯƠNG TRÌNH DU LỊCH NƯỚC NGOÀI PHÁT HUY HIỆU QUẢ HOẠT ĐỘNG CÔNG ĐOÀN"}\n{"id": "B-A-f314ef349d4ed841", "claim": "Điều này giúp Công ty có thể xác định các lĩnh vực có thể cải thiện và theo dõi tiến trình hướng tới các mục tiêu bền vững.", "evidence": "Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước. Để đảm bảo không ngừng cải thiện hiệu quả sử dụng các nguồn năng lượng tái tạo, Coteccons theo dõi tiến trình hướng tới việc tăng mức sử dụng nguồn năng lượng này theo thời gian. Điều này giúp Công ty theo dõi chặt chẽ hiệu quả hoạt động ESG và đưa ra các quyết định dựa trên dữ liệu để đạt được các mục tiêu bền vững. Quản lý, điều phối triển khai các hoạt động ESG trên 3 lĩnh vực hoạt động: công trình, môi trường và cộng đồng bền vững. Chuyên gia tư vấn ESG - Thà"}\n{"id": "B-A-5b2349e4c92f92d8", "claim": "Công ty đã chính thức ban hành cam kết sử dụng năng lượng bền vững và đưa vào hoạt động hàng ngày.", "evidence": "Coteccons trên khắp Việt Nam. CAM KẾT SỬ DỤNG NĂNG LƯỢNG VÀ QUẢN LÝ NƯỚC BỀN VỮNG Trên cơ sở nhận thức sâu sắc rằng các công ty xây dựng có thể giảm thiểu tác động đến môi trường bằng cách sử dụng các nguồn năng lượng thay thế như năng lượng gió, mặt trời, thủy điện và pin hydro... Coteccons hướng tới hợp tác với các đối tác năng lượng tái tạo để mua sắm năng lượng cần thiết cho các dự án. Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước. Để đảm bảo không ngừng cải thiện hiệu quả sử dụng các nguồn năng lượng tái"}\n{"id": "B-A-d51bc666da526e71", "claim": "Hiện Coteccons đã sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời để cung cấp năng lượng cho một số công trường xây dựng và tòa nhà văn phòng của Công ty.", "evidence": "Sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời 91 90 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Quản trị rủi ro trong lĩnh vực ESG. Sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời và các nguồn khác. Giải trình Công bằng 45 44 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-51f8ff9ef97f5ec6", "claim": "Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước.", "evidence": "Một trong những cách Coteccons thực hiện quản lý nước bền vững là sử dụng các thiết bị lưu lượng thấp. Những thiết bị này được thiết kế để giảm lượng nước sử dụng trong các cơ sở hoạt động mà không ảnh hưởng đến hiệu quả sử dụng của chúng. Công ty cũng triển khai các hệ thống thu gom nước mưa, thu thập và lưu trữ nước mưa để sử dụng sau này, giảm sự phụ thuộc vào nguồn cung cấp nước. Một cách khác để Công ty giảm lượng nước sử dụng là kiến tạo cảnh quan tiết kiệm nước. Bằng cách chọn những loại cây cần ít nước hơn và thiết kế cảnh quan hợp lý để giảm thiểu lượng nước chảy ra ngoài, Công ty có thể giảm đáng kể lượng nước sử dụng để duy trì các cơ sở của mình. Đồng thời, Coteccons cũng chú trọ"}\n{"id": "B-A-c8a24867bbf988cb", "claim": "Để đảm bảo không ngừng cải thiện hiệu quả sử dụng các nguồn năng lượng tái tạo, Coteccons theo dõi tiến trình hướng tới việc tăng mức sử dụng nguồn năng lượng này theo thời gian.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhà máy Suntory PepsiCo do Coteccons thi công tại Long An Để đảm bảo rằng đang đạt được tiến bộ hướng tới các mục tiêu bảo tồn nước, Công ty đo lường lượng nước tiêu thụ trong các hoạt động xây dựng. Điều này giúp Công ty có thể xác định các lĩnh vực có thể cải thiện và theo dõi tiến trình hướng tới các mục tiêu bền vững. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons GIÁ TRỊ CỐT LÕI 1 4 KHÔNG NGỪNG CẢI TIẾN TÂM HUYẾT CHẤT LƯỢNG VÀ AN TOÀN"}\n{"id": "B-A-db2f95e2618c71aa", "claim": "Điều này giúp Công ty theo dõi chặt chẽ hiệu quả hoạt động ESG và đưa ra các quyết định dựa trên dữ liệu để đạt được các mục tiêu bền vững.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhà máy Suntory PepsiCo do Coteccons thi công tại Long An Để đảm bảo rằng đang đạt được tiến bộ hướng tới các mục tiêu bảo tồn nước, Công ty đo lường lượng nước tiêu thụ trong các hoạt động xây dựng. Điều này giúp Công ty có thể xác định các lĩnh vực có thể cải thiện và theo dõi tiến trình hướng tới các mục tiêu bền vững. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons THỰC HÀNH ESG THEO 5 MỤC TIÊU CỦA COTECCONS Coteccons triển khai thực hành ESG một cách quyết liệt trong nội bộ, đồng thời tìm kiếm, xây dựng quan hệ với các đối tác chung mục tiêu phát triển bền vững."}\n{"id": "B-A-806ec7014b304c00", "claim": "Coteccons còn hướng đến trở thành nhà đầu tư của các dự án năng lượng tái tạo.", "evidence": "Công ty đã chính thức ban hành cam kết sử dụng năng lượng bền vững và đưa vào hoạt động hàng ngày. Hiện Coteccons đã sử dụng các nguồn năng lượng tái tạo như năng lượng mặt trời để cung cấp năng lượng cho một số công trường xây dựng và tòa nhà văn phòng của Công ty. Nỗ lực này đang được nhân rộng đến tất cả các dự án của Các công trường của Coteccons, đặc biệt là các công trường thuộc dự án FDI như công trường LEGO Vietnam, Suntory PepsiCo, luôn là những hình mẫu tuân thủ các yêu cầu về bảo vệ môi trường, thực hiện các chương trình tiết kiệm năng lượng, tài nguyên và sử dụng năng lượng tái tạo. Có thể kể đến nhiều sáng kiến đã được áp dụng như:"}\n{"id": "B-A-2afe0cebfea44191", "claim": "Ước tính công suất/hiệu quả tiết kiệm năng lượng: Đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết - giúp tiết kiệm trung bình 300-1.000 kw điện trong 1 tháng.", "evidence": "Thống kê số công trường của Coteccons sử dụng năng lượng mặt trời: Chỉ sử dụng đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết. Bên cạnh đó, trong mô hình phát triển kinh doanh của mình, Cho thuê máy móc, thiết bị đun nước nóng dùng năng lượng mặt trời, máy móc thiết bị tiết kiệm năng lượng 0316438470 47 Lê Văn Thịnh, KP5, P. Bình Trưng Đông, TP. Thủ Đức, TP.HCM 110 99,54 Công ty con gián tiếp"}\n{"id": "B-A-639a6a93a8fd2e01", "claim": "Một cách khác để Công ty giảm lượng nước sử dụng là kiến tạo cảnh quan tiết kiệm nước.", "evidence": "Một trong những cách Coteccons thực hiện quản lý nước bền vững là sử dụng các thiết bị lưu lượng thấp. Những thiết bị này được thiết kế để giảm lượng nước sử dụng trong các cơ sở hoạt động mà không ảnh hưởng đến hiệu quả sử dụng của chúng. Công ty cũng triển khai các hệ thống thu gom nước mưa, thu thập và lưu trữ nước mưa để sử dụng sau này, giảm sự phụ thuộc vào nguồn cung cấp nước. Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước. Để đảm bảo không ngừng cải thiện hiệu quả sử dụng các nguồn năng lượng tái tạo, "}\n{"id": "B-A-4a5340d9544a876f", "claim": "Đồng thời, Coteccons cũng chú trọng việc tiết kiệm nước và tận tâm thực hiện yêu cầu này trong phạm vi của mình để giảm tác động từ hoạt động của Công ty đối với môi trường.", "evidence": "TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Coteccons cũng ưu tiên nguyên vật liệu địa phương, vật liệu có độ bền cao, vật liệu xanh từ công nghệ mới... Chúng tôi đã có cơ hội gặp gỡ những đối tác chuyên nghiệp với những sản phẩm, giải pháp t"}\n{"id": "B-A-8b6f2cd2da6ef563", "claim": "Mức giảm phát thải CO2 tương đương 10,78 tấn.", "evidence": "CO2 ra môi trường trong năm tài chính vừa qua. Cùng với việc ưu tiên sử dụng vật liệu gạch không nung cũng góp phần tiết giảm được 926 tấn CO2, tiết kiệm được 171 tấn dầu tương đương. Song song, 26% dòng vật liệu bền Đặc biệt, hệ thống dữ liệu trên phần mềm được hiển thị trực quan hóa bằng các biểu đồ, từ đó, dễ dàng so sánh, đánh giá các xu hướng phát thải để đưa ra quyết định đúng đắn và chiến lược phù hợp với mức độ phát thải mà Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất v"}\n{"id": "B-A-721e05a65274ea79", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Trong phạm vi năm tài chính 2024, Coteccons đã tiến hành trồng mới 10.175 cây rừng, gồm 2.175 cây tại Vườn Quốc gia Phong Nha – Kẻ Bảng và 8.000 cây tại khu vực rừng đặc dụng Bảo vệ voọc gáy trắng tại xã Thạch Hóa, tỉnh Quảng Bình.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TRỒNG RỪNG VÀ BẢO VỆ QUẦN THỂ VOỌC GÁY TRẮNG 21.285 cây bản địa Thái Lan vào năm 2025. Sau khi giải được tổ chức thành công, với quãng đường mà các vận động viên tích lũy được, Coteccons đã thực hiện trồng 11.111 cây xanh tại khu vực rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa, tỉnh Quảng Bình. 135 134 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-4bbe982bc784c9c7", "claim": "Coteccons tổ chức, với nguồn quỹ huy động được, vào 09/2024, Công ty tiếp tục khởi công trồng 11.111 cây xanh tại rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa.", "evidence": "Thái Lan vào năm 2025. Sau khi giải được tổ chức thành công, với quãng đường mà các vận động viên tích lũy được, Coteccons đã thực hiện trồng 11.111 cây xanh tại khu vực rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa, tỉnh Quảng Bình. 135 134 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Bên cạnh trồng rừng đặc dụng, Coteccons còn tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương nhằm góp phần gia tăng hành lang bảo vệ voọc, hạn chế tình trạng lấn chiếm đất rừng, khai thác gỗ rừng và tài nguyên rừng trái phép hoặc chăn thả gia súc tại những nơi đàn voọc sinh sống. Bác Nguyễn Thanh Tú - công tác tại Hợp tác xã Sản xuất Kinh doanh Nông Lâm nghiệp và t"}\n{"id": "B-A-a3a120d070a6eaf4", "claim": "Coteccons là một trong những đơn vị hỗ trợ cho Hợp tác xã Voọc gáy trắng – nhóm tình nguyện viên bảo vệ loài voọc gáy trắng để xây dựng một hệ sinh thái an toàn hơn.", "evidence": "Bên cạnh trồng rừng đặc dụng, Coteccons còn tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương nhằm góp phần gia tăng hành lang bảo vệ voọc, hạn chế tình trạng lấn chiếm đất rừng, khai thác gỗ rừng và tài nguyên rừng trái phép hoặc chăn thả gia súc tại những nơi đàn voọc sinh sống. Bác Nguyễn Thanh Tú - công tác tại Hợp tác xã Sản xuất Kinh doanh Nông Lâm nghiệp và tự nguyện bảo tồn voọc gáy trắng tại xã Thạch Hóa, huyện Tuyên Hóa, Quảng Thái Lan vào năm 2025. Sau khi giải được tổ chức thành công, với quãng đường mà các vận động viên tích lũy được, Coteccons đã thực hiện trồng 11.111 cây xanh tại khu vực rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện T"}\n{"id": "B-A-dadc7c27db97f90c", "claim": "Giảm thiểu phát thải thông qua tái sản xuất, tái sử dụng để tạo nên một vòng tuần hoàn ít tác động nhất đến tài nguyên và môi trường là nhiệm vụ mà cả thế giới đang hướng đến.", "evidence": "Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ việc sử dụng nguyên liệu sản xuất và rác thải khó phân hủy từ đồng phục cũ. Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và "}\n{"id": "B-A-dfb7ecf209b07ca7", "claim": "Với nhu cầu sử dụng nguyên vật liệu, nhân lực và tạo ra nguồn phát thải lớn, ngành xây dựng càng khẩn thiết áp dụng các giải pháp kinh tế tuần hoàn.", "evidence": "Trong nỗ lực thúc đẩy các giải pháp về kinh tế tuần hoàn, Coteccons nghiêm khắc yêu cầu áp dụng nguyên tắc 3R trong mọi phương diện xuyên suốt phát triển dự án, tối ưu hiệu quả sử dụng nguyên vật liệu trong một vòng đời khép kín để phát huy giá trị tối đa của chúng. Coteccons đang nỗ lực triển khai Mô hình hóa Dữ liệu Công trình (BIM) cho tất cả các dự án của mình. Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đ"}\n{"id": "B-A-2a85dc116145b3b3", "claim": "Trái Đất xanh” với kết quả trong vòng 18 ngày, đã thu hồi được hơn 7.000 viên pin cũ để tái chế và tái sử dụng.", "evidence": "Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Từ đó, xây dựng nên các giá trị lâu dài hơn cho cổ đông và Công ty. Công ty bắt đầu sử dụng phương pháp phát hành trái phiếu để huy động vốn. Việc phát hành trái phiếu có thể gây một số quan ngại với nhà "}\n{"id": "B-A-ee25fc43dd26b003", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Người Coteccons đã và đang thực hiện giảm thiểu sử dụng chai, hộp nhựa, phân loại rác tại nguồn và xử lý rác thải phù hợp.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons HƯỞNG ỨNG NGÀY QUỐC TẾ KHÔNG RÁC THẢI Rác thải đang trở thành gánh nặng khổng lồ trên toàn thế giới. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons CHÍNH SÁCH VỀ MÔI TRƯỜNG CỦA COTECCONS 1 2 3 4 5 Giảm phát thải khí nhà kính bằng cách sử dụng nguyên nhiên liệu, thiết bị hiệu quả và thực hành tiết kiệm năng lượng."}\n{"id": "B-A-065d510ec2f64edc", "claim": "Khối Kỹ thuật và Tuân thủ đã dùng phương châm “REUSE đồ cũ để REDUCE phát thải rác” trong quá trình trang trí cho khu vực chung của tầng 11 – tòa nhà Coteccons với tác phẩm “Đẹp không đụng hàng” trong dịp chào mừng Tết Nguyên đán.", "evidence": "Mục tiêu hàng đầu của Coteccons: Thực hành tốt nguyên tắc 3R (Reduce – Reuse – Recycle) do Công ty thiết lập, giảm thiểu những tác động không cần thiết đến môi trường, sử dụng hiệu quả nguồn nguyên vật liệu và có các giải pháp khoa học kỹ thuật để tối ưu hóa các nguồn lực hiện có. Nguyên tắc 3R (Reduce – Reuse – Recycle) được Coteccons thiết lập và triển khai từ năm Mỗi nhà cung cấp đã, đang và sẽ hợp tác đều là các đối tác quan trọng góp phần tạo nên thành công của Coteccons nói chung và dự án nói riêng. Khối Thu mua luôn đặt mục tiêu cân bằng quyền lợi giữa công ty và các đối tác để đảm bảo sự đồng hành lâu dài và bền vững. Hàng năm, Khối Thu mua sẽ chọn các đối tác tiềm năng và phát triển"}\n{"id": "B-A-1687c9c79803fd91", "claim": "Một số sáng kiến nổi bật thực hiện chỉ tiêu 3R của các công trường như tái chế vật liệu từ chất thải cơ điện, kim loại từ dây điện, sử dụng triệt để sơn nước thừa từ các công trình hoàn thiện cho dự án thiện nguyện “Nhà mình”, dọn dẹp vệ sinh, thu gom rác thải định kỳ với chương trình Green Day...", "evidence": "Coteccons, từ văn phòng đến công trường không còn sử dụng nước đóng chai hay cốc nhựa, mà thay thế bằng những bình nước cỡ lớn và cốc thuỷ tinh dùng nhiều lần. Dự án “Nhà mình” tận dụng những vật liệu xây dựng như sơn, xi măng dư thừa từ các công trình đã hoàn thiện để sử dụng cho nhu cầu chỉnh trang, làm đẹp các công trình công cộng hoặc nhà dân ở những vùng khó khăn do dự án hỗ trợ. Coteccons và Unicons phối hợp triển khai chương trình “Thu hồi pin bảo vệ Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các "}\n{"id": "B-A-7d6c32db2f971a06", "claim": "Trong nỗ lực thúc đẩy các giải pháp về kinh tế tuần hoàn, Coteccons nghiêm khắc yêu cầu áp dụng nguyên tắc 3R trong mọi phương diện xuyên suốt phát triển dự án, tối ưu hiệu quả sử dụng nguyên vật liệu trong một vòng đời khép kín để phát huy giá trị tối đa của chúng.", "evidence": "Giảm thiểu phát thải thông qua tái sản xuất, tái sử dụng để tạo nên một vòng tuần hoàn ít tác động nhất đến tài nguyên và môi trường là nhiệm vụ mà cả thế giới đang hướng đến. Với nhu cầu sử dụng nguyên vật liệu, nhân lực và tạo ra nguồn phát thải lớn, ngành xây dựng càng khẩn thiết áp dụng các giải pháp kinh tế tuần hoàn. 2022 nhưng một trong những định hướng cụ thể để giảm thiểu rác thải, chất thải và các tác động đến môi trường xuyên suốt trên phạm vi rộng. Trong năm qua, nguyên tắc 3R tiếp tục được yêu cầu triển khai sâu rộng hơn. Coteccons đã áp dụng các mức chỉ tiêu cụ thể cho từng bộ phận, nhóm và dự án trong doanh nghiệp, từ văn phòng đến công trường, để thúc đẩy kết quả rõ ràng."}\n{"id": "B-A-687b81e5096684e5", "claim": "Phần mềm này bao gồm cả nội dung giảm thiểu chất thải trong các giai đoạn thiết kế, tăng khả năng định lượng và cho phép thử nghiệm nhiều phương án thiết kế để tìm hiệu suất giảm chất thải khác nhau, cung cấp cơ sở chuyên môn cao nhất cho việc ra quyết định giảm thiểu chất thải xây dựng.", "evidence": "Thực hiện phương án giảm thiểu ồn, bụi trong quá trình thi công. Thực hiện việc thu gom và xử lý nước thải đạt chuẩn trước khi thải ra môi trường. Thực hiện thu gom và xử lý các chất thải/đất/bùn thải từ quá trình thi công của mình. Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất và phát huy đổi mới sáng tạo trong hoạt động giảm phát thải; Lưu trữ và quản lý dữ liệu lượng phát thải của Coteccons, dành cho cả những công trình, dự án đã hoàn thiện. Thiết lập biểu mẫu giám sát khí nh"}\n{"id": "B-A-96185ac8f007f8b6", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Hai năm trước, Coteccons đã thực hiện thành công dự án Ecopark Swan Lake Residences với trị giá lên đến gần 4.000 tỷ đồng.", "evidence": "Mối hợp tác với Ecopark là một điển hình cho sự thành công của chiến lược repeat sales. Đồng thời, các dự án có tầm vóc và tiêu chuẩn vượt trội của Ecopark đã thách thức tư duy sáng tạo, thôi thúc ý chí vượt khó của người Coteccons. Hai năm trước, Coteccons đã thực hiện thành công dự án Ecopark Swan Lake Residences với trị giá lên đến gần 4.000 tỷ đồng. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons HIỆU QUẢ HOẠT ĐỘNG KINH TẾ 66 60 ĐVT: Tỷ đồng dự án dự án 2024 So với năm 2023"}\n{"id": "B-A-2efb4c6ccbda0cfe", "claim": "Dự án này có điểm đặc biệt là đem 1.000 khu vườn với hơn 10.000m2 cây xanh lên không trung để “vẽ thiên nhiên trên bầu trời”.", "evidence": "2,3 hecta Sky Forest Residences được gọi là dự án biểu tượng cho kiến trúc xanh với thiết kế “nhà trong vườn và vườn trong nhà”, “căn hộ nằm trong công viên và công viên nằm trong căn hộ”. Do đó, dự án này có điểm đặc biệt là đem 1.000 khu vườn với hơn 10.000m2 cây xanh lên không trung để “vẽ thiên nhiên trên bầu trời”. Loại hình dự án : Khu căn hộ phức hợp Chủ đầu tư : Tập đoàn Ecopark Địa bàn : Tỉnh Hưng Yên Gói thầu : Design & Build Thiết kế trên không những khiến chi phí xây dựng dự án tăng mạnh do phải mang số lượng cây xanh và hệ thống tiện ích khổng lồ trên cao, mà còn đặt ra các thách thức về thiết kế hệ thống móng, hệ thống chịu lực đặc biệt của mỗi tòa nhà, kỹ thuật xử lý chống thấ"}\n{"id": "B-A-fbf96685253b2680", "claim": "Các yêu cầu cụ thể như ưu tiên sử dụng các sản phẩm có thể phân hủy sinh học, gạch không nung, kính Low-E, thép xanh...", "evidence": "Giữa bối cảnh thị trường còn nhiều thách thức như hiện nay, Coteccons xác định ưu tiên đẩy mạnh đa dạng hóa sản phẩm để đa dạng hóa nguồn doanh thu trước. Theo đó, Công ty đẩy mạnh các lĩnh vực sản phẩm - dịch vụ mới dựa trên năng lực cốt lõi là xây dựng, đồng thời mở rộng hệ sinh thái bằng cách tối ưu lợi thế sẵn có như nguồn tiền mặt và tài sản dồi dào. Đa dạng hóa theo định hướng này đảm bảo sự an toàn trước khi thị trường khởi sắc hơn và nguồn lực cho các ngành nghề mới được chuẩn bị một cách chắc chắn. Việc ưu tiên sử dụng các vật liệu dễ phân hủy và tái sử dụng, cùng với việc lựa chọn nhà cung cấp bền vững không chỉ giảm thiểu tác động đến môi trường mà còn tạo ra một không gian làm vi"}\n{"id": "B-A-aa19073d427bf84c", "claim": "8 Non-baked Materials (Vật liệu tái chế) Concrete bricks, gypsum panels, pre-cast concrete panels, AAC blocks", "evidence": "TABLE 8 | Non-baked Materials (Vật liệu tái chế) | Concrete bricks, gypsum panels, pre-cast concrete panels, AAC blocks (Gạch bê tông, tấm thạch cao, bê tông tiền chế, gạch AAC) 9 | Low-Emitting Materials (Vật liệu ít phát tán các chất hữu cơ dễ bay hơi (VOC)) | Low or no VOC paints, coatings, adhesives, sealants (Sơn, lớp phủ, chất kết dính, keo trám khe chứa hàm lượng VOC thấp hoặc không chứa VOC) Low or no formaldehyde composite wood products (Gỗ tổng hợp chứa hàm lượng fomanđehit thấp hoặc không chứa fomandehit) 10 | High Efficiency Water Fixtures (Thiết bị sử dụng nước hiệu quả cao) | High efficiency or dual flush toilets, waterless urinals, low flow faucets (Vật liệu tái chế) 11 | Effi"}\n{"id": "B-A-54ef02f67d1941d5", "claim": "Sản phẩm và vật liệu có đặc tính xanh Ví dụ 10 High Efficiency Water Fixtures (Thiết bị sử dụng nước hiệu quả cao) High efficiency or dual flush toilets, waterless urinals, low flow faucets", "evidence": "TABLE 8 | Non-baked Materials (Vật liệu tái chế) | Concrete bricks, gypsum panels, pre-cast concrete panels, AAC blocks (Gạch bê tông, tấm thạch cao, bê tông tiền chế, gạch AAC) 9 | Low-Emitting Materials (Vật liệu ít phát tán các chất hữu cơ dễ bay hơi (VOC)) | Low or no VOC paints, coatings, adhesives, sealants (Sơn, lớp phủ, chất kết dính, keo trám khe chứa hàm lượng VOC thấp hoặc không chứa VOC) Low or no formaldehyde composite wood products (Gỗ tổng hợp chứa hàm lượng fomanđehit thấp hoặc không chứa fomandehit) 10 | High Efficiency Water Fixtures (Thiết bị sử dụng nước hiệu quả cao) | High efficiency or dual flush toilets, waterless urinals, low flow faucets (Vật liệu tái chế) 11 | Effi"}\n{"id": "B-A-92233c14c56238c6", "claim": "(Vật liệu tái chế) 1 Reused Materials (Vật liệu tái chế) Salvaged brick, pavers, lumber (Gạch, tấm lát, gỗ phế liệu) 2 Materials with Recycled Content (Vật liệu có thành phần tái chế)", "evidence": "TABLE | Sản phẩm và vật liệu có đặc tính xanh | Ví dụ 1 | Reused Materials (Vật liệu tái chế) | Salvaged brick, pavers, lumber (Gạch, tấm lát, gỗ phế liệu) 2 | Materials with Recycled Content (Vật liệu có thành phần tái chế) | Steel, gypsum ceilings/boards, carpet, insulation (Thép, trần, tường thạch cao, thảm, lớp cách âm cách nhiệt) 3 | Regionally sourced Materials (Vật liệu địa phương) | Harvested, extracted and/or manufactured within 500 km of project site (Vật liệu được khai thác, chiết xuất hoặc sản xuất trong phạm vi 500 km từ khu vực dự án) 4 | Rapidly Renewable Materials (Vật liệu tái tạo nhanh) | Cotton insulation, bamboo flooring, linoleum flooring, straw board (Bông cách nhiệt, s"}\n{"id": "B-A-4582993abc5df24b", "claim": "Drip irrigation, irrigation system with soil moisture sensor (Hệ thống vệ sinh tiết kiệm nước hoặc có 2 chế độ xả, bồn tiểu không dùng nước, vòi nước có lưu lượng chảy thấp) 3 Regionally sourced Materials (Vật liệu địa phương)", "evidence": "TABLE 8 | Non-baked Materials (Vật liệu tái chế) | Concrete bricks, gypsum panels, pre-cast concrete panels, AAC blocks (Gạch bê tông, tấm thạch cao, bê tông tiền chế, gạch AAC) 9 | Low-Emitting Materials (Vật liệu ít phát tán các chất hữu cơ dễ bay hơi (VOC)) | Low or no VOC paints, coatings, adhesives, sealants (Sơn, lớp phủ, chất kết dính, keo trám khe chứa hàm lượng VOC thấp hoặc không chứa VOC) Low or no formaldehyde composite wood products (Gỗ tổng hợp chứa hàm lượng fomanđehit thấp hoặc không chứa fomandehit) 10 | High Efficiency Water Fixtures (Thiết bị sử dụng nước hiệu quả cao) | High efficiency or dual flush toilets, waterless urinals, low flow faucets (Vật liệu tái chế) 11 | Effi"}\n{"id": "B-A-b58e361453afb380", "claim": "Low-impact alternatives to refrigerants commonly used in building refrigeration, air-conditioning and heat pump systems (Sử dụng chất làm lạnh thân thiện với môi trường cho các hệ thống làm mát, ĐHKK và bơm nhiệt) 5 Timber from sustainable sources (Gỗ từ các nguồn bền vững)", "evidence": "TABLE 8 | Non-baked Materials (Vật liệu tái chế) | Concrete bricks, gypsum panels, pre-cast concrete panels, AAC blocks (Gạch bê tông, tấm thạch cao, bê tông tiền chế, gạch AAC) 9 | Low-Emitting Materials (Vật liệu ít phát tán các chất hữu cơ dễ bay hơi (VOC)) | Low or no VOC paints, coatings, adhesives, sealants (Sơn, lớp phủ, chất kết dính, keo trám khe chứa hàm lượng VOC thấp hoặc không chứa VOC) Low or no formaldehyde composite wood products (Gỗ tổng hợp chứa hàm lượng fomanđehit thấp hoặc không chứa fomandehit) 10 | High Efficiency Water Fixtures (Thiết bị sử dụng nước hiệu quả cao) | High efficiency or dual flush toilets, waterless urinals, low flow faucets (Vật liệu tái chế) 11 | Effi"}\n{"id": "B-A-79752d74f26ce67c", "claim": "(Hệ thống vệ sinh tiết kiệm nước hoặc có 2 chế độ xả, bồn tiểu không dùng nước, vòi nước có lưu lượng chảy thấp)", "evidence": "Drip irrigation, irrigation system with soil moisture sensor (Hệ thống vệ sinh tiết kiệm nước hoặc có 2 chế độ xả, bồn tiểu không dùng nước, vòi nước có lưu lượng chảy thấp) 3 Regionally sourced Materials (Vật liệu địa phương) Các biện pháp cụ thể trong việc bảo vệ và bảo tồn tài nguyên nước bao gồm việc thực hiện các biện pháp quản lý nước bền vững như sử dụng thiết bị cố định dòng chảy thấp, hệ thống thu gom nước mưa và sử dụng cảnh quan tiết kiệm nước. Để đảm bảo không ngừng cải thiện hiệu quả sử dụng các nguồn năng lượng tái tạo, Coteccons theo dõi tiến trình hướng tới việc tăng mức sử dụng nguồn năng lượng này theo thời gian. Điều này giúp Công ty theo dõi chặt chẽ hiệu quả hoạt động ES"}\n{"id": "B-A-cbe3f0fee76d35e0", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 16 Efficient ventilation and air-conditioning equip- ment (Thiết bị thông gió và ĐHKK có hiệu quả cao)", "evidence": "TABLE 16 | Efficient ventilation and air-conditioning equip- ment (Thiết bị thông gió và ĐHKK có hiệu quả cao) | Air-conditioning systems with high CSPF values (for non-ducted units) and high COP values, systems with variable controls (inverter, VSD/VFD) (Hệ thống ĐHKK có hệ số CSPF cao [đối với ĐHKK không ống gió] và chỉ số COP cao, ĐHKK có điều khiển biến tần [inverter, VSD/VFD]) 17 | Efficient lighting equipment (Thiết bị chiếu sáng có hiệu quả cao) | Energy-efficient lighting fixtures (with high lumen effi- cacy lm/W), lighting controls and sensors (Thiết bị chiếu sáng sử dụng năng lượng hiệu quả [có hiệu suất chiếu sáng cao], thiết bị điều khiển chiếu sáng, cảm biến ánh sáng) 18 | Renew"}\n{"id": "B-A-52a173d12f841a50", "claim": "(Thiết bị chiếu sáng sử dụng năng lượng hiệu quả [có hiệu suất chiếu sáng cao], thiết bị điều khiển chiếu sáng, cảm biến ánh sáng) 18 Renewable Energy Equipment (Hệ thống năng lượng tái tạo) Solar panels, wind energy, geothermal (Hệ thống năng lượng mặt trời, phong năng, địa nhiệt)", "evidence": "TABLE 16 | Efficient ventilation and air-conditioning equip- ment (Thiết bị thông gió và ĐHKK có hiệu quả cao) | Air-conditioning systems with high CSPF values (for non-ducted units) and high COP values, systems with variable controls (inverter, VSD/VFD) (Hệ thống ĐHKK có hệ số CSPF cao [đối với ĐHKK không ống gió] và chỉ số COP cao, ĐHKK có điều khiển biến tần [inverter, VSD/VFD]) 17 | Efficient lighting equipment (Thiết bị chiếu sáng có hiệu quả cao) | Energy-efficient lighting fixtures (with high lumen effi- cacy lm/W), lighting controls and sensors (Thiết bị chiếu sáng sử dụng năng lượng hiệu quả [có hiệu suất chiếu sáng cao], thiết bị điều khiển chiếu sáng, cảm biến ánh sáng) 18 | Renew"}\n{"id": "B-A-d501b2ec6ea4197b", "claim": "22 Energy-efficient lifts (Thang máy sử dụng năng lượng hiệu quả) Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with:", "evidence": "VVVF Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: VVVF drives, regenerative drive, efficient lighting. TABLE 16 | Efficient ventilation and air-conditioning equip- ment (Thiết bị thông gió và ĐHKK có hiệu quả cao) | Air-conditioning systems with high CSPF values (for non-ducted units) and high COP values, systems with variable controls (inverter, VSD/VFD) (Hệ thống ĐHKK có hệ số CSPF cao [đối với ĐHKK không ống gió] và chỉ số COP cao, ĐHKK có điều khiển biến tần [inverter, VSD/VFD]) 17 | Efficient lighting equipment (Thiết bị chiếu sáng có hiệu quả cao) | Energy-efficient lighting fixtures (with high lumen effi- cacy lm/W)"}\n{"id": "B-A-5b86049b0943a685", "claim": "VVVF Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: VVVF drives, regenerative drive, efficient lighting.", "evidence": "22 Energy-efficient lifts (Thang máy sử dụng năng lượng hiệu quả) Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: TABLE 16 | Efficient ventilation and air-conditioning equip- ment (Thiết bị thông gió và ĐHKK có hiệu quả cao) | Air-conditioning systems with high CSPF values (for non-ducted units) and high COP values, systems with variable controls (inverter, VSD/VFD) (Hệ thống ĐHKK có hệ số CSPF cao [đối với ĐHKK không ống gió] và chỉ số COP cao, ĐHKK có điều khiển biến tần [inverter, VSD/VFD]) 17 | Efficient lighting equipment (Thiết bị chiếu sáng có hiệu quả cao) | Energy-efficient lighting fixtures (with high lumen effi- ca"}\n{"id": "B-A-8db1238d5861d06d", "claim": "Lifts with energy efficiency class A following the certi-", "evidence": "22 Energy-efficient lifts (Thang máy sử dụng năng lượng hiệu quả) Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: VVVF Lifts with energy efficiency class A following the certi- fication guideline VDI 4707 Part 1, lifts equipped with: VVVF drives, regenerative drive, efficient lighting."}\n{"id": "B-A-64aac49d44eb9477", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Đồng phục xanh của Coteccons sử dụng những màu sắc chủ đạo của thương hiệu là Blue navy, blue teal và xám với thiết kế tôn vinh tuổi trẻ, tinh thần tự do, tính sáng tạo và “chất” của người xây dựng.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons SỨC MẠNH CỦA THƯƠNG HIỆU XANH Coteccons đang vươn đến giá trị thương hiệu cao nhất trong lịch sử ngành xây dựng Việt Nam. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TÁC ĐỘNG KINH TẾ GIÁN TIẾP Là doanh nghiệp dẫn đầu ngành xây dựng"}\n{"id": "B-A-8f0bdbf9a5058d9c", "claim": "Đây là loại sản phẩm đồng phục thân thiện với môi trường được làm từ vải nano (dệt từ 100% chai nhựa tái chế) và vải cafe (dệt từ chai nhựa tái chế và bã cafe).", "evidence": "Phát triển Bền vững IMPACTD của Coteccons nhằm thực hiện cam kết thực hành phát triển bền vững bắt đầu từ những hành động nhỏ nhất. Hai chất liệu được sử dụng là vải nano và vải cà phê được dệt hoàn toàn từ nguyên liệu tái chế. Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê. Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ "}\n{"id": "B-A-b4c5c62b588d4e89", "claim": "Phát triển Bền vững IMPACTD của Coteccons nhằm thực hiện cam kết thực hành phát triển bền vững bắt đầu từ những hành động nhỏ nhất.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Coteccons nhất quán với định hướng phát triển bền vững, phát triển có trách nhiệm bằng cách thực thi cụ thể các nhiệm vụ với môi trường, xã hội và con người xuyên suốt tiến trình thực hiện hoạt động kinh doanh. Chiến lược ESG tại Coteccons được thực hiện từ cấp lãnh đạo cao nhất. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons PHƯƠNG PHÁP THỰC HIỆN BÁO CÁO Nhằm mang đến cho các bên liên quan thông tin minh bạch, ghi nhận và xác thực kết quả đã thực hiện được, xây dựng cơ sở tham khảo cho lộ trình hành động tiếp theo, Coteccons thực hiện Báo cáo Phát triển bền vững theo quy trình nh"}\n{"id": "B-A-de035c7bb7d044c9", "claim": "Hai chất liệu được sử dụng là vải nano và vải cà phê được dệt hoàn toàn từ nguyên liệu tái chế.", "evidence": "ĐỒNG PHỤC TỪ NGUYÊN LIỆU TÁI CHẾ Từ năm 2022, Coteccons đưa vào sử dụng đồng phục xanh (Coteccons Green Uniform). Đây là loại sản phẩm đồng phục thân thiện với môi trường được làm từ vải nano (dệt từ 100% chai nhựa tái chế) và vải cafe (dệt từ chai nhựa tái chế và bã cafe). Đồng phục xanh là một trong những sáng kiến nằm trong Chương trình Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ việc sử d"}\n{"id": "B-A-50554ae50c2e649c", "claim": "Với loại vải cà phê, cứ mỗi 3 tách cà phê và 5 chai nhựa tái chế sẽ đủ lượng nguyên liệu để tạo ra một sản phẩm áo cà phê.", "evidence": "ĐỒNG PHỤC TỪ NGUYÊN LIỆU TÁI CHẾ Từ năm 2022, Coteccons đưa vào sử dụng đồng phục xanh (Coteccons Green Uniform). Đây là loại sản phẩm đồng phục thân thiện với môi trường được làm từ vải nano (dệt từ 100% chai nhựa tái chế) và vải cafe (dệt từ chai nhựa tái chế và bã cafe). Đồng phục xanh là một trong những sáng kiến nằm trong Chương trình Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ việc sử d"}\n{"id": "B-A-2b56f7d7f44903cb", "claim": "Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường.", "evidence": "Giảm thiểu phát thải thông qua tái sản xuất, tái sử dụng để tạo nên một vòng tuần hoàn ít tác động nhất đến tài nguyên và môi trường là nhiệm vụ mà cả thế giới đang hướng đến. Với nhu cầu sử dụng nguyên vật liệu, nhân lực và tạo ra nguồn phát thải lớn, ngành xây dựng càng khẩn thiết áp dụng các giải pháp kinh tế tuần hoàn. Một cách khác để Công ty giảm lượng nước sử dụng là kiến tạo cảnh quan tiết kiệm nước. Bằng cách chọn những loại cây cần ít nước hơn và thiết kế cảnh quan hợp lý để giảm thiểu lượng nước chảy ra ngoài, Công ty có thể giảm đáng kể lượng nước sử dụng để duy trì các cơ sở của mình. Đồng thời, Coteccons cũng chú trọng việc tiết kiệm nước và tận tâm thực hiện yêu cầu này trong "}\n{"id": "B-A-7c6fa02cb0090567", "claim": "Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ việc sử dụng nguyên liệu sản xuất và rác thải khó phân hủy từ đồng phục cũ.", "evidence": "Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. ĐỒNG PHỤC TỪ NGUYÊN LIỆU TÁI CHẾ Từ năm 2022, Coteccons đưa vào sử dụng đồng phục xanh (Coteccons Green Uniform). Đây là loại sản phẩm đồng phục thân thiện với môi trường được làm từ vải nano (dệt từ 100%"}\n{"id": "B-A-f217899b34ee5088", "claim": "Các loại vải “xanh” như trên không chỉ thân thiện với môi trường mà còn có bề mặt vải thoáng khí, thấm hút nhanh, độ cơ giãn cơ học cao, mang đến cảm giác thoải mái cho người mặc.", "evidence": "Việc ưu tiên sử dụng các vật liệu dễ phân hủy và tái sử dụng, cùng với việc lựa chọn nhà cung cấp bền vững không chỉ giảm thiểu tác động đến môi trường mà còn tạo ra một không gian làm việc thân thiện và thoải mái hơn. Thêm nữa, việc phủ xanh văn phòng và sử dụng dịch vụ Taxi Xanh đã góp phần thể hiện cam kết của chúng ta với một tương lai xanh hơn. Trong các sự kiện, chúng tôi áp dụng nguyên tắc làm đúng ngay từ đầu. Số lượng công trình do Nhà Mình hỗ trợ sơn mới được tiếp tục mở rộng trong năm qua, đưa tổng số công trình được hoàn thành qua dự án lên đến 121 tại nhiều địa phương trên cả nước, gồm nhiều loại hình như nhà văn hóa – sinh hoạt cộng đồng, mái ấm, trường học, nghĩa trang liệt sĩ"}\n{"id": "B-A-e59548d7ce732405", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 03 SDG17.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 17 16 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-6daeb5de5e9f57e8", "claim": "Với chủ đề “Chung tay cùng thịnh vượng”, đây là dịp để Ban lãnh đạo Coteccons chia sẻ với hơn 200 đối tác về chiến lược, tinh thần hợp tác và định hướng phát triển bền vững của Công ty.", "evidence": "Cũng tại hội nghị, Coteccons đã tiến hành ký kết thỏa thuận hợp tác chiến lược với một số đối tác để phát triển sản phẩm - dịch vụ theo hướng bền vững. Trong năm qua, Coteccons tích cực tổ chức và tham gia nhiều chương trình, sự kiện, gặp gỡ để thúc đẩy tinh thần hợp tác và chia sẻ về phát triển bền vững với đối tác, khách hàng và cộng đồng. Khi chọn định hướng phát triển trên tầm nhìn industry leader, Coteccons xác định tâm thế tiên phong, khởi xướng xu hướng, tiêu chuẩn và giá trị mới cho các bên liên quan nhằm nâng cao chất lượng của ngành xây dựng. TABLE 03 SDG17. HỢP TÁC CÙNG PHÁT TRIỂN Khi chọn định hướng phát triển trên tầm nhìn industry leader, Coteccons xác định tâm thế tiên phong, "}\n{"id": "B-A-dba9e8dba50dc431", "claim": "Tham gia hội nghị “Xúc tiến đầu tư phát triển khu công nghiệp thông minh, bền vững, hướng đến Net Zero” vào ngày 15/08/2024.", "evidence": "Tổng quan Chiến lược phát triển 2025 – 2029 28 - 33 34 - 35 Giảm phát thải - hướng tới Net Zero Kinh tế tuần hoàn Hợp tác cùng phát triển Tham gia hội thảo “Hỗ trợ doanh nghiệp kinh doanh bền vững tạo lợi thế cạnh tranh mới”. Sự kiện diễn ra vào 14/08/2024, tại Cần Thơ, do Cục Phát triển Doanh nghiệp và USAID tổ chức. Hội thảo tập trung vào xu hướng kinh doanh mới và hỗ trợ doanh nghiệp bền vững tại Việt Nam, bao gồm mô hình Kinh doanh bao trùm (IB), Kinh tế tuần hoàn (CE), và khung đánh giá ESG."}\n{"id": "B-A-16bfbda7850fd7d7", "claim": "Chương trình có sự tham dự của đại diện các khu công nghiệp, các tập đoàn lớn tại “Hội nghị xúc tiến đầu tư phát triển khu công nghiệp thông minh, bền vững, hướng đến Net Zero” do Cổng thông tin Khu công nghiệp Việt Nam tổ chức, trong nỗ lực triển khai đề án “Đầu tư phát triển Khu công nghiệp thông minh bền vững” (SSIP).", "evidence": "Đạt giải thưởng Top Doanh nghiệp Dẫn dắt Đa dạng, Công bằng và Hòa nhập (DEI) tại Chương trình Bình chọn TOP 50 Doanh nghiệp phát triển bền vững tiêu biểu Việt Nam 2023, do Tạp chí Nhịp cầu Đầu tư tổ chức. 2021 2015-2017 Tiến hành tái cấu trúc doanh nghiệp, xây dựng nền tảng mới để thích ứng với sự thay đổi từ thị trường và định hướng lộ trình phát triển đến năm 2025. Triển khai nhiều hợp đồng D&B lớn. Đánh dấu mức tăng trưởng về doanh thu và lợi nhuận cao kỷ lục. Chuyên gia tư vấn 2022 – Hiện tại: Nhà nghiên cứu và ủng hộ chính sách Quốc tịch: Việt Nam Kinh nghiệm nghề nghiệp Năm sinh: 1959 Tiến sĩ Bích là một chuyên gia hàng đầu trong tư vấn và triển khai các chiến lược phát triển bền vững"}\n{"id": "B-A-915ef506129f7a42", "claim": "Tại đây, Coteccons chia sẻ tầm nhìn trong việc định hình lại hệ sinh thái xây dựng bền vững và thúc đẩy ngành xây dựng xanh, tạo ra những không gian an toàn hơn, thoải mái hơn, ý nghĩa hơn trong toàn bộ chuỗi giá trị bằng những cam kết với các tiêu chuẩn ESG và HSE.", "evidence": "“Industry Leader” bằng cách tạo ra những không gian an toàn hơn, thoải mái hơn và ý nghĩa hơn cho tất cả mọi người. Đây là trái ngọt cho những nỗ lực tuyệt vời của toàn thể nhân viên Coteccons. CÂU CHUYỆN SAU NHỮNG CON SỐ 86 87 BÁO CÁO THƯỜNG NIÊN NĂM TÀI CHÍNH 2024 Bằng nhận thức đó, Coteccons lựa chọn triết lý phát triển dựa trên chiến lược ESG có tính cam kết sâu, rộng và tầm nhìn dài hạn. Trong đó, Coteccons đón nhận thách thức, khảo nghiệm và nguy cơ trên vai trò là người tiên phong nỗ lực thúc đẩy các tiêu chuẩn ESG của toàn ngành, góp phần giải quyết các vấn đề của ngành, đóng góp những giá trị có chiều sâu cho sự phát triển chung. Qua đó, Công ty mong muốn cùng các bên thiết lập con "}\n{"id": "B-A-74ab3cb3f01a21e4", "claim": "Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon trong giai đoạn xây dựng.", "evidence": "TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon"}\n{"id": "B-A-ca60909990576945", "claim": "Theo ước tính, một dự án như vậy khi hoàn thành sẽ loại trừ và giảm tổng cộng 2.803 tấn carbon hàm chứa so với vật liệu thông thường, tương đương với lượng carbon được hấp thụ bởi 46.348 cây xanh trong 10 năm.", "evidence": "TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon"}\n{"id": "B-A-0f11d512ea2c5812", "claim": "Coteccons và Faslink bắt tay đưa trạm tái chế vải đầu tiên đến Việt Nam với dự án Second Life, nhằm tái sinh “cuộc đời thứ 2” cho những loại vải vi nhựa vốn cần đến 200 năm để phân hủy, hướng đến mục tiêu tận dụng lợi thế chuyên môn của mỗi bên để giảm 49% tiêu thụ nước và giảm 59% khí thải carbon.", "evidence": "TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon"}\n{"id": "B-A-7db4df76419291a8", "claim": "Thông qua hợp tác với Taxi Xanh, người của Coteccons sẽ chuyển đổi nhu cầu đi lại qua phương tiện này.", "evidence": "TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH • Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon"}\n{"id": "B-A-4498395e3f49d39b", "claim": "Coteccons và Công ty TNHH LIXIL Việt Nam (LIXIL Việt Nam) – Thành viên thuộc LIXIL Nhật Bản – chính thức ký kết biên bản ghi nhớ hợp tác chiến lược với mục tiêu theo đuổi chiến lược phát triển bền vững và hoàn thiện chuỗi cung ứng, mang đến giải pháp toàn diện cùng nhiều giá trị hơn cho khách hàng trong chuỗi giá trị ngành xây dựng.", "evidence": "Coteccons - Daewoo Engineering & Construction: Ký kết biên bản ghi nhớ hợp tác quốc tế, với mục tiêu cùng nhau phát triển các cơ hội dự án tiềm năng trong lĩnh vực xây dựng trên thị trường toàn cầu (trong đó có Việt Nam). Coteccons - LIXIL: Ký kết biên bản ghi nhớ hợp tác chiến lược với mục tiêu theo đuổi chiến lược phát triển bền vững và hoàn thiện chuỗi cung ứng, mang đến giải pháp toàn diện cùng nhiều giá trị hơn cho khách hàng trong chuỗi giá trị ngành xây dựng. TABLE Phần II a Tổng quan về Coteccons | Phần III Phần IV Phần V Phần VI Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LI"}\n{"id": "B-A-3a63578f5beeb09a", "claim": "Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons CÙNG ĐỐI TÁC HƯỚNG TỚI VẬT LIỆU XANH Coteccons đã ký hợp tác với Công ty Cổ phần Bê Tông FiCO Pan-United và Keppel Việt Nam, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Coteccons - Fico PanU - Keppel Việt Nam: Ký kết Biên bản ghi nhớ để phát triển các cơ hội hợp tác trong lĩnh vực BĐS, bao gồm việc nghiên cứu và ứng dụng bê tông bền vững, vữa thương phẩm và các vật liệu cải tiến dùng trong kiến trúc cũng như các phương pháp xây dựng sử dụng công nghệ in 3D. Các hoạt động cải tiến và t"}\n{"id": "B-A-ebde3bd895dcfc45", "claim": "Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2", "evidence": "Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon trong giai đoạn xây dựng. Trên thực tế, Coteccons đã tiến hành triển khai một số biện pháp như giảm thiểu dấu chân carbon và lựa chọn sử dụng các sản phẩm có chứng chỉ LCA và EPD đến từ hơn 57 đối tác thương hiệu uy tín có tác động môi trường thấp hơn trong toàn bộ vòng đời. Theo ước tính ban đầu, việc sử dụng bê tông khoáng hóa CO2 (bê tông Fico) với khối lượng 132.603 tấn tiết giảm được 1.936 tấn"}\n{"id": "B-A-f44842de456cd8e5", "claim": "(CMC) giúp giảm lượng phát thải carbon trong giai đoạn xây dựng.", "evidence": "Việc thử nghiệm sẽ được triển khai ở The Infiniti, phân khu thứ ba của dự án Riviera Point. Đây sẽ là dự án đầu tiên ở Việt Nam ứng dụng Bê tông khoáng hóa CO2 (CMC) giúp giảm lượng phát thải carbon trong giai đoạn xây dựng. Phần mềm này bao gồm cả nội dung giảm thiểu chất thải trong các giai đoạn thiết kế, tăng khả năng định lượng và cho phép thử nghiệm nhiều phương án thiết kế để tìm hiệu suất giảm chất thải khác nhau, cung cấp cơ sở chuyên môn cao nhất cho việc ra quyết định giảm thiểu chất thải xây dựng. 115 114 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-b2d5e887072c97b4", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 11.000 cây xanh cho dự án Việt Nam Xanh Sau 3 năm với 3 mùa giải, GreenUp Marathon do Coteccons tổ chức đã trở thành một trong những sự kiện được trông đợi với những người yêu thích bộ môn chạy bộ trên khắp Việt Nam.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons GREENUP MARATHON LONG AN HALF MARATHON 2024 Dự án thể thao này còn mang sứ mệnh lớn là lan tỏa rộng rãi ra cộng đồng tinh thần sống hướng về môi trường, thông điệp về phát triển bền vững và gây quỹ cho các chương trình trồng rừng hưởng ứng dự án Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Trong phạm vi năm tài chính 2024, Coteccons đã tiến hành trồng mới 10.175 cây rừng, gồm 2.175 cây tại Vườn Quốc gia Phong Nha – Kẻ Bảng và 8.000 cây tại khu vực rừng đặc dụng Bảo vệ voọc gáy trắng tại xã Thạch Hóa, tỉnh Quảng Bình. Sau giải chạy GreenUP Marathon 2024 do"}\n{"id": "B-A-2e27e97b1ac70a2f", "claim": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 11.000 cây xanh cho dự án Việt Nam Xanh Sau 3 năm với 3 mùa giải, GreenUp Marathon do Coteccons tổ chức đã trở thành một trong những sự kiện được trông đợi với những người yêu thích bộ môn chạy bộ trên khắp Việt Nam. Đây là động lực để chúng tôi vượt qua bất lợi về thời tiết và ngoại cảnh. Chúng tôi còn tham gia chạy bộ với chương trình GreenUP marathon. Mỗi bước chạy không chỉ vì sức khỏe và niềm vui của chính mình mà còn có thể đóng góp cây xanh cho các dự án trồng rừng."}\n{"id": "B-A-f0e9206a937a2646", "claim": "Mục tiêu là góp 11.000 cây xanh vào dự án trồng rừng nguyên sinh tỉnh Quảng Bình, Quảng Trị.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Trong phạm vi năm tài chính 2024, Coteccons đã tiến hành trồng mới 10.175 cây rừng, gồm 2.175 cây tại Vườn Quốc gia Phong Nha – Kẻ Bảng và 8.000 cây tại khu vực rừng đặc dụng Bảo vệ voọc gáy trắng tại xã Thạch Hóa, tỉnh Quảng Bình. Sau giải chạy GreenUP Marathon 2024 do Trồng rừng - bảo tồn đa dạng sinh học là hoạt động ESG nổi bật được Coteccons triển khai xuyên suốt từ năm 2022 với địa phương tiểu biểu là tỉnh Quảng Bình. Đến nay, hoạt động này đã phát huy kết quả rõ nét, góp phần gia tăng nhận thức của công chúng về môi trường và các vấn đề sinh kế tại địa phương."}\n{"id": "B-A-85520be1d1cf0d23", "claim": "Sau khi giải được tổ chức thành công, với quãng đường mà các vận động viên tích lũy được, Coteccons đã thực hiện trồng 11.111 cây xanh tại khu vực rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa, tỉnh Quảng Bình.", "evidence": "Coteccons tổ chức, với nguồn quỹ huy động được, vào 09/2024, Công ty tiếp tục khởi công trồng 11.111 cây xanh tại rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa. Cây được trồng mới bao gồm các loài cây bản địa như lát hoa, lim xanh, huỷnh, dâu da, gáo vàng... Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Trong phạm vi năm tài chính 2024, Coteccons đã tiến hành trồng mới 10.175 cây rừng, gồm 2.175 cây tại Vườn Quốc gia Phong Nha – Kẻ Bảng và 8.000 cây tại khu vực rừng đặc dụng Bảo vệ voọc gáy trắng tại xã Thạch Hóa, tỉnh Quảng Bình. Sau giải chạy GreenUP Marathon 2024 do"}\n{"id": "B-A-211d09c2a4894efa", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Phó Tổng giám đốc, Khối vận hành Nguyễn Trình Thùy Trang chia sẻ: “Chúng tôi luôn tin tưởng rằng sự đa dạng và hòa nhập sẽ mang lại nguồn năng lượng sáng tạo mạnh mẽ, giúp Coteccons không ngừng vươn lên và khẳng định vị thế của mình trong ngành xây dựng.", "evidence": "DEI tại Coteccons được ưu tiên và triển khai không chỉ với nhân viện của mình mà còn hướng đến các bên liên quan, như công nhân lạo động ở công trình, đặc biệt công nhân nữ, hay công nhân nữ thuộc nhóm đồng bào thiểu số, các nhà cung cấp và nhà thầu phụ...Coteccons tin tưởng rằng sự đa dạng và hòa nhập sẽ mang lại nguồn năng lượng sáng tạo mạnh mẽ, giúp Coteccons không ngừng vươn lên và khẳng định vị thế của mình trong ngành xây dựng. Những nỗ lực DEI không chỉ là trách nhiệm xã hội mà còn là chiến lược phát triển bền vững của chúng tôi. TABLE TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS Coteccons xác định đa dạng là yếu tố quyết định, không chỉ dừng lại ở sự khác biệt về vă"}\n{"id": "B-A-fd353fbb3d14a948", "claim": "Những nỗ lực DEI không chỉ là trách nhiệm xã hội mà còn là chiến lược phát triển bền vững của chúng tôi.” Chính sách về môi trường làm việc Coteccons không ngừng cải thiện môi trường lao động theo tiêu chí đa dạng, công bằng và hòa nhập.", "evidence": "DEI tại Coteccons được ưu tiên và triển khai không chỉ với nhân viện của mình mà còn hướng đến các bên liên quan, như công nhân lạo động ở công trình, đặc biệt công nhân nữ, hay công nhân nữ thuộc nhóm đồng bào thiểu số, các nhà cung cấp và nhà thầu phụ...Coteccons tin tưởng rằng sự đa dạng và hòa nhập sẽ mang lại nguồn năng lượng sáng tạo mạnh mẽ, giúp Coteccons không ngừng vươn lên và khẳng định vị thế của mình trong ngành xây dựng. Những nỗ lực DEI không chỉ là trách nhiệm xã hội mà còn là chiến lược phát triển bền vững của chúng tôi. TABLE TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS Coteccons xác định đa dạng là yếu tố quyết định, không chỉ dừng lại ở sự khác biệt về vă"}\n{"id": "B-A-bf060907725c8491", "claim": "Pantone Xanh Navy đậm 2768C: đó là màu sắc đặc trưng của ngành xây dựng thể hiện sự mạnh mẽ nam tính, màu sắc của kỹ thuật, công nghệ xây dựng.", "evidence": "Pantone Xanh Teal 3262C: Là màu sắc đại diện cho các bạn trẻ hoặc những người có tính cách cởi mở và sáng suốt, sáng tạo, đây cũng là màu sắc có ý nghĩa liên quan đến những dòng nước chảy tự nhiên thuộc về môi trường, đặc trưng của tính cách thông thái, khiêm tốn và lòng tốt và thuận theo dòng chảy hơn là bám chắc vào những kinh nghiệm cũ. Pantone Vàng nâu 721C: Màu của những hạt cát, một vật liệu không thế thiếu trong ngành xây dụng. Vì lẽ đó, vàng nâu đại diện cho tính cách mạnh mẽ, lòng dũng cảm, sự dứt khoát và sức dẻo dai, bền bỉ. TABLE TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS Coteccons xác định đa dạng là yếu tố quyết định, không chỉ dừng lại ở sự khác biệt về văn "}\n{"id": "B-A-0be3ce69699189ec", "claim": "Pantone Xanh lá 7482C: màu xanh lá cây mang ý nghĩa về sự phát triển và tái sinh, màu của cây cỏ lá, thiên nhiên.", "evidence": "Coteccons với 6 pantone màu chủ đạo. Đây chính là sự khẳng định ý chí đa dạng hóa “bảng màu” nhân sự, tạo ra văn hóa và môi trường làm việc mà ở đó tất cả mọi người từ các nền văn hóa, tôn giáo, chủng tộc, giới tính khác nhau đều góp phần tạo nên sự phát triển bền vững cho công ty. MÀU XANH LỤC CỦA CÂY LÁ MÀU XÁM CỦA SẮT THÉP, XI MĂNG Màu sắc này cũng đại diện cho tính cách của những nhóm nhân sự mang lại cho người khác cảm giác tin cậy, an toàn và trách nhiệm. Pantone Xám 11C: ngoài màu sắc gợi tới hình ảnh của sắt, thép, bê tông, màu xám cũng là màu của sự trung lập, thông thái và kiên định. Chúng tôi đề cao những nhóm nhân sự mang yếu tố chắc chắn và vững chãi này."}\n{"id": "B-A-901febd22668e40d", "claim": "Pantone Xanh Teal 3262C: Là màu sắc đại diện cho các bạn trẻ hoặc những người có tính cách cởi mở và sáng suốt, sáng tạo, đây cũng là màu sắc có ý nghĩa liên quan đến những dòng nước chảy tự nhiên thuộc về môi trường, đặc trưng của tính cách thông thái, khiêm tốn và lòng tốt và thuận theo dòng chảy hơn là bám chắc vào những kinh nghiệm cũ.", "evidence": "TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS MÀU XANH DƯƠNG CỦA CÔNG NGHỆ MÀU XANH TEAL ĐẠI DIỆN CHO DÒNG CHẢY Đại diện cho các nhân sự trẻ hoặc những người có tính cách cởi mở năng động và sáng tạo. TABLE TIẾN BỘ VỀ ĐA DẠNG, CÔNG BẰNG VÀ HÒA NHẬP (DEI) TẠI COTECCONS Coteccons xác định đa dạng là yếu tố quyết định, không chỉ dừng lại ở sự khác biệt về văn hóa, sắc tộc, hay tuổi tác, mà còn bao gồm cả tư duy, sở thích, sở trường, trình độ và phong cách làm việc. Công ty tin rằng một đội ngũ nhân sự đa dạng sẽ mang đến nhiều lợi ích, giúp tối ưu hóa quy trình làm việc và nâng cao hiệu quả công việc. Việc xây dựng một đội ngũ đa dạng không chỉ là mở rộng các tiêu chí tuyển dụng"}\n{"id": "B-A-16ae9bc54a9b4a32", "claim": "Vì vậy, trong tiêu chuẩn về phát triển bền vững, an toàn lao động được Coteccons xác định là ưu tiên hàng đầu, luôn được đặt ở mức yêu cầu cao nhất để tuân thủ và quản lý, dành cho tất cả cấp độ và bộ phận, từ nhân sự trực tiếp đến lao động gián tiếp.", "evidence": "Trong đó, bảo vệ người lao động và không ngừng cải thiện quyền lợi cho họ là một trong những mục tiêu trọng tâm. Ngành xây dựng có rủi ro về an toàn lao động rất cao. Vì vậy, trong tiêu chuẩn về phát triển bền vững, an toàn lao động được Coteccons xác định là ưu tiên hàng đầu, luôn được đặt ở mức yêu cầu cao nhất để tuân thủ và quản lý, dành cho tất cả cấp độ và bộ phận, từ nhân sự trực tiếp đến lao động gián tiếp. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons ƯU TIÊN HÀNG ĐẦU VỀ SỨC KHỎE VÀ AN TOÀN CHO NGƯỜI LAO ĐỘNG ... an toàn lao động được Coteccons xác định là ưu tiên hàng đầu, luôn được đặt ở mức yêu cầu cao nhất để tuân thủ và quản lý, dành cho tất cả"}\n{"id": "B-A-13926c63b0d4d71d", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Bên cạnh an toàn lao động, Coteccons cũng thực hiện các nguyên tắc để bảo vệ an toàn về tâm lý và tinh thần.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons QUẢN LÝ RỦI RO LIÊN QUAN ĐẾN ESG RỦI RO VỀ AN TOÀN LAO ĐỘNG Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons GIỚI THIỆU VỀ BÁO CÁO Coteccons thực hiện báo cáo ESG riêng để phục vụ cho định hướng thực hành ESG chuyên sâu của Công ty."}\n{"id": "B-A-b3610382994bb622", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Coteccons đặc biệt đề cao và thực hiện quyết liệt quyền con người trong ngành xây dựng.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons THỰC HÀNH ESG THEO 5 MỤC TIÊU CỦA COTECCONS Coteccons triển khai thực hành ESG một cách quyết liệt trong nội bộ, đồng thời tìm kiếm, xây dựng quan hệ với các đối tác chung mục tiêu phát triển bền vững. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TÁC ĐỘNG KINH TẾ GIÁN TIẾP Là doanh nghiệp dẫn đầu ngành xây dựng"}\n{"id": "B-A-2242a175fb66b8ac", "claim": "Theo đó, Công ty cam kết đồng hành cùng các trường để hỗ trợ sinh viên trên các phương diện như nghiên cứu học thuật, tiếp nhận thực tập, tài trợ học bổng, tiếp nhận tham quan và trải nghiệm thực tế các công trường và xây dựng nhận thức về ESG.", "evidence": "TABLE Phần I Phần II Phần III Phần IV Phần V Phần VI Con đường của Tổng quan Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG tương lai xanh về Coteccons về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons HỖ TRỢ ĐÀO TẠO THẾ HỆ TRẺ NEXT GEN Next Gen là một trong những chủ trương lớn của Coteccons trong việc góp phần Với chủ trương trên, trong năm qua, Coteccons tiếp tục hướng tới chương trình tiếp nhận thực tập sinh đào tạo nguồn nhân lực thế hệ mới, không chỉ cho Công ty, mà cho toàn ngành xây và hợp tác đào tạo, hỗ trợ với các trường đại học. Theo đó, Coteccons đều đặn tổ chức các đợt thực dựng và môi trường kinh doanh. Trong điều kiện và ưu thế hiện có của mình, "}\n{"id": "B-A-a99ac8666ba79f5c", "claim": "được đào tạo bởi các chuyên gia đến từ những đơn vị hàng đầu như Dale Carnegie Việt Nam, Arup, Hội đồng công trình xanh Việt Nam...", "evidence": "Ở hình thức trực tiếp, Công ty có các hoạt động điển hình là chương trình CTD Talk, khóa đào tạo về kỹ năng thuyết trình, ERP... được đào tạo bởi các chuyên gia đến từ những đơn vị hàng đầu như Dale Carnergie Việt Nam, Arup, Hội đồng công trình xanh Việt Nam... TABLE | | Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons COTECCONS 159 HOẠT ĐỘNG ĐÀO TẠO SINH ĐỘNG 26.326 2.164 1.807 giờ đào tạo trong năm giờ đào tạo về ESG người tham gia các khóa đào tạo ESG Hoạt động đào tạo được xem là xương sống để phát triển đội ngũ tại Coteccons. Công ty có đơn vị chuyên trách là Trung tâm đào tạo cùng các bộ phận, phòng ban xây dựng và tổ chức các chương trình đào tạo phù hợp với yêu cầu bổ sung, phát t"}\n{"id": "B-A-437b5da91ab076c2", "claim": "CTD Talk : Chương trình chia sẻ của các chuyên gia hàng đầu về các lĩnh vực mới hoặc chủ đề lớn mà người Coteccons cần nắm bắt như Công trình xanh, Vật liệu của tương lai, ESG trong ngành xây dựng...", "evidence": "TABLE | | Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons COTECCONS 159 HOẠT ĐỘNG ĐÀO TẠO SINH ĐỘNG 26.326 2.164 1.807 giờ đào tạo trong năm giờ đào tạo về ESG người tham gia các khóa đào tạo ESG Hoạt động đào tạo được xem là xương sống để phát triển đội ngũ tại Coteccons. Công ty có đơn vị chuyên trách là Trung tâm đào tạo cùng các bộ phận, phòng ban xây dựng và tổ chức các chương trình đào tạo phù hợp với yêu cầu bổ sung, phát triển năng lực cho nhân sự. Phương diện này ngày càng được đầu tư về chiều sâu để tăng hiệu quả cho người học và mở rộng hình thức, cơ hội học tập để nhiều người có thể tiếp cận. Tổng thời lượng của hoạt động đào tạo trong năm qua tại Coteccons đạt 26.326 giờ. • "}\n{"id": "B-A-2caf96eca30ad08e", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 05 SDG11.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 17 16 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-94b440f53014ab4a", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Bằng tinh thần ấy, Công ty đã và đang dành nhiều nỗ lực để góp phần nâng cao vai trò của người công nhân trong nhận thức của các bên liên quan, cải thiện môi trường làm việc và nâng cao đời sống của họ.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhằm nắm bắt thực tế về điều kiện sống, nguyện vọng và nhu cầu của công nhân để hỗ trợ họ tốt hơn, Coteccons đã tiến hành cuộc khảo sát nhận diện người lao động trên các công trường Công ty đang triển khai. KHẢO SÁT NHẬN DIỆN NGƯỜI LAO ĐỘNG TRÊN CÁC CÔNG TRƯỜNG Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons PHÁT TRIỂN MÔI TRƯỜNG LÀM VIỆC NHÂN VĂN nơi làm việc tốt nhất ngành BĐS và xây dựng."}\n{"id": "B-A-5088eb0dece97749", "claim": "Ông Vương cũng nhắc tới nguyên tắc tìm kiếm đối tác mà Trưởng tiểu ban ESG của Coteccons – ông Herwig Guido H.van Hove - từng nói: “Coteccons sẽ chỉ hợp tác với các đối tác chia sẻ những giá trị phát triển bền vững, không phân biệt giới tính, dân tộc, tín ngưỡng, không bóc lột lao động và tuyệt đối không sử dụng lao động trẻ em...”", "evidence": "Cơ cấu này được xác lập bao gồm các chức năng Định hướng - Quản lý – Tham vấn - Triển khai. Trưởng Tiểu Ban: Ông HERWIG GUIDO H.VAN HOVE Thúc đẩy và kết nối quốc tế, và Hội đồng quản trị, tài trợ cho các chiến lược ESG phù hợp với mục tiêu và giá trị của công ty. 1 Ông Herwig Guido H. Van Hove Thành viên HĐQT không điều hành Trưởng Tiểu ban 7 Nguyễn Trình Thùy Trang Thành viên phụ trách Khối Nội chính 2 Ông Võ Hoàng Lâm"}\n{"id": "B-A-c7c247747dadcf46", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Nhằm nắm bắt thực tế về điều kiện sống, nguyện vọng và nhu cầu của công nhân để hỗ trợ họ tốt hơn, Coteccons đã tiến hành cuộc khảo sát nhận diện người lao động trên các công trường Công ty đang triển khai.", "evidence": "Nhằm nắm bắt thực tế về điều kiện sống, nguyện vọng và nhu cầu của công nhân để hỗ trợ họ tốt hơn, Coteccons đã tiến hành cuộc khảo sát nhận diện người lao động, nhà thầu phụ trên các công trường Công ty đang triển khai. Qua khảo sát này, có thể thấy rõ với ngành xây dựng, sự chuyển dịch địa bàn làm việc và nhu cầu lao động ngày càng đa dạng, linh động. Tiêu biểu như sự dịch chuyển của công nhân xây dựng từ TABLE | Phần I Phần II Phần III Phần IV Phần V Phần VI Con đường của Tổng quan Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG tương lai xanh về Coteccons về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons | | Nhằm nắm bắt thực tế về điều kiện sống, nguyện vọng"}\n{"id": "B-A-9b0bdada32935915", "claim": "Việc xen canh cây cà gai leo không ảnh hưởng đến quá trình sinh trưởng của rừng trong 3 năm kế tiếp, vừa tạo nguồn thu nhập cho các hộ dân trồng rừng vừa bảo tồn môi sinh của đàn voọc.", "evidence": "Đến với xã Thạch Hoá, có thể cảm nhận rõ hạnh phúc của bà con khi được trao tay mô hình sinh kế mới. Việc xen canh cây cà gai leo không ảnh hưởng đến quá trình sinh trưởng của rừng trong 3 năm kế tiếp, vừa tạo nguồn thu nhập cho các hộ dân trồng rừng vừa bảo tồn môi sinh của đàn voọc. Để góp phần thúc đẩy chính sách trên đi vào thực tế hiệu quả, từ năm 2023, tiểu ban ESG của Coteccons đã cùng đại diện chính quyền tỉnh và lãnh đạo địa phương xã Thạch Đây là chia sẻ từ ông Trần Tiến, một người dân tham gia trồng cà gai leo tại thôn Đồng Tâm. Tiếp theo, tiểu ban ESG tiếp tục thực hiện nhiều khảo sát và xây dựng kế hoạch khả thi cho chương trình trồng cây cà gai leo. Theo đó, Coteccons hỗ trợ gi"}\n{"id": "B-A-e1bf5c11dcfe06e5", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Cuối tháng 03/2024, tiểu ban ESG Coteccons đã có chuyến đi khảo sát và đánh giá kết quả trồng rừng kết hợp theo dõi sự tăng trưởng của cây cà gai leo.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TRỒNG RỪNG VÀ BẢO VỆ QUẦN THỂ VOỌC GÁY TRẮNG 21.285 cây bản địa Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons NHỮNG CON SỐ TIÊU BIỂU TRONG NĂM Về thực thi ESG Về phát triển kinh doanh"}\n{"id": "B-A-bab547db3dbad0ad", "claim": "Theo đó, Coteccons hỗ trợ giống cây cà gai leo xen canh tại vùng đệm – nơi bà con đã hiến đất canh tác để trồng rừng tặng voọc.", "evidence": "Bình, có thể cảm nhận rõ hạnh phúc của bà con khi được trao tay mô hình sinh kế mới. Việc xen canh cây cà gai leo không ảnh hưởng đến quá trình sinh trưởng của rừng trong 3 năm kế tiếp, vừa tạo nguồn thu nhập cho các hộ dân trồng rừng vừa bảo tồn môi sinh của đàn voọc. 167 166 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS Đến với xã Thạch Hoá, có thể cảm nhận rõ hạnh phúc của bà con khi được trao tay mô hình sinh kế mới. Việc xen canh cây cà gai leo không ảnh hưởng đến quá trình sinh trưởng của rừng trong 3 năm kế tiếp, vừa tạo nguồn thu nhập cho các hộ dân trồng rừng vừa bảo tồn môi sinh của đàn voọc. Để góp phần thúc đẩy chính sách trên đi vào thực tế hiệu quả, từ năm 2023, tiểu ban ESG của Cotecc"}\n{"id": "B-A-bacb1aaf7a18df42", "claim": "Việc đổ thải nguồn sơn này cũng góp phần tăng phát thải vào môi trường.", "evidence": "Phần mềm này bao gồm cả nội dung giảm thiểu chất thải trong các giai đoạn thiết kế, tăng khả năng định lượng và cho phép thử nghiệm nhiều phương án thiết kế để tìm hiệu suất giảm chất thải khác nhau, cung cấp cơ sở chuyên môn cao nhất cho việc ra quyết định giảm thiểu chất thải xây dựng. 115 114 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS CO2 ra môi trường trong năm tài chính vừa qua. Cùng với việc ưu tiên sử dụng vật liệu gạch không nung cũng góp phần tiết giảm được 926 tấn CO2, tiết kiệm được 171 tấn dầu tương đương. Song song, 26% dòng vật liệu bền Đặc biệt, hệ thống dữ liệu trên phần mềm được hiển thị trực quan hóa bằng các biểu đồ, từ đó, dễ dàng so sánh, đánh giá các xu hướng phát thải để đư"}\n{"id": "B-A-8451fd108ba391d0", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons “Khi ESG đi vào đời sống ở Coteccons, những công việc hậu cần vốn quen thuộc trở nên có tính sứ mệnh hơn khi chúng tôi có cách nhìn và cách hiểu mới về giá trị mà công việc của mình tạo ra.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Khi chúng tôi xây dựng một công trình, chúng tôi hiểu rằng Coteccons đang cùng với khách hàng kiến tạo những giá trị trường tồn, không chỉ giới hạn ở lợi ích kinh tế mà còn là biểu tượng của bảo vệ môi trường và phát triển cộng đồng bền vững. Với vai trò là tổng thầu xây dựng hàng đầu Việt Nam, Coteccons xác định phải trở thành trung tâm Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Tại Coteccons, tất yếu chúng tôi đang thay đổi để tốt hơn. Sau hàng ngàn giờ suy nghĩ, chúng tôi không hề nghi ngờ gì về sự thành công của con đường mình đã chọn."}\n{"id": "B-A-58d6f0aba229f38b", "claim": "Việc ưu tiên sử dụng các vật liệu dễ phân hủy và tái sử dụng, cùng với việc lựa chọn nhà cung cấp bền vững không chỉ giảm thiểu tác động đến môi trường mà còn tạo ra một không gian làm việc thân thiện và thoải mái hơn.", "evidence": "Đặc biệt, các loại vải này sau một thời gian sử dụng có thể tái chế và tiếp tục sử dụng để may các thiết kế mới, tạo thành một vòng lặp khép kín, giảm thiểu rác thải xả ra môi trường. Với số lượng nhân sự lớn, Coteccons có nhu cầu đồng phục hằng năm rất nhiều. Việc may đồng phục từ nguyên liệu tái chế giúp giảm một phần tác động đến môi trường từ việc sử dụng nguyên liệu sản xuất và rác thải khó phân hủy từ đồng phục cũ. TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt k"}\n{"id": "B-A-6105a3d1c7ecd9b7", "claim": "Khi thực hiện các hoạt động xã hội hoặc phối hợp kiểm kê phát thải từ những ngày đầu tiên, tôi có cơ hội gặp gỡ với nhiều người từ các cấp chính quyền, các tổ chức lớn trong và ngoài nước...", "evidence": "TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Coteccons cũng ưu tiên nguyên vật liệu địa phương, vật liệu có độ bền cao, vật liệu xanh từ công nghệ mới... Chúng tôi đã có cơ hội gặp gỡ những đối tác chuyên nghiệp với những sản phẩm, giải pháp t"}\n{"id": "B-A-cfbedf489f50af40", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons “Tôi có cơ hội được quản lý dự án LEGO Manufactory Viet Nam do Coteccons làm tổng thầu.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons QUẢN LÝ RỦI RO LIÊN QUAN ĐẾN ESG RỦI RO VỀ AN TOÀN LAO ĐỘNG Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons KẾT QUẢ TĂNG TRƯỞNG KINH TẾ Năm tài chính 2024 của Coteccons xứng đáng được xem là một bức tranh rực rỡ."}\n{"id": "B-A-49cbee2aff36c64b", "claim": "Mục tiêu cần đạt được đầu tiên là xây dựng nhà máy trung hòa carbon đầu tiên trên thế giới của Tập đoàn LEGO, với các tiêu chuẩn Leed Platinium, Energy Center.", "evidence": "Trong số đó, Pandora là nhà máy có diện tích lớn nhất ở Đông Nam Á đáp ứng tiêu chuẩn LEED, LEGO là nhà máy trung hoà carbon đầu tiên của Tập đoàn LEGO trên toàn thế giới, Suntory PepsiCo là nhà máy áp dụng công nghệ xanh hiện đại nhất khu vực Châu Á - Thái Bình Dương... Các dự án này đều đòi hỏi hồ sơ dự thầu phải chứng minh được khả năng thực thi ESG của doanh nghiệp - bao gồm cả kiểm tra và phỏng vấn trực tiếp. Những sắc thái cảm xúc công chúng liên tưởng khi nghe nói hoặc nhìn thấy thương hiệu Do đó, tiêu chuẩn và mục tiêu về sức khỏe và an toàn tại Coteccons được liên tục nâng cấp để vượt trên mức tuân thủ. Quy trình thực thi được đặt dưới hệ thống giám sát tập trung và hiệu suất chính "}\n{"id": "B-A-e5bb8cb7c5a58421", "claim": "Đây sẽ là nhà máy bền vững nhất của LEGO trên thế giới về mặt thiết kế và xây dựng với trang thiết bị hiện đại, sử dụng hoàn toàn năng lượng mặt trời, mang đến 4.000 cơ hội việc làm trong vòng 15 năm tới...", "evidence": "Thống kê số công trường của Coteccons sử dụng năng lượng mặt trời: Chỉ sử dụng đèn năng lượng mặt trời: LEGO, Nova World Phan Thiết. Bên cạnh đó, trong mô hình phát triển kinh doanh của mình, Coteccons trên khắp Việt Nam. CAM KẾT SỬ DỤNG NĂNG LƯỢNG VÀ QUẢN LÝ NƯỚC BỀN VỮNG Trên cơ sở nhận thức sâu sắc rằng các công ty xây dựng có thể giảm thiểu tác động đến môi trường bằng cách sử dụng các nguồn năng lượng thay thế như năng lượng gió, mặt trời, thủy điện và pin hydro... Coteccons hướng tới hợp tác với các đối tác năng lượng tái tạo để mua sắm năng lượng cần thiết cho các dự án."}\n{"id": "B-A-15ce24e8a90c7ac7", "claim": "Điều này không chỉ giúp giảm lượng khí thải carbon mà còn cải thiện sức khỏe của mọi người.", "evidence": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững Mỗi sáng, mọi người cùng nhau thưởng thức bữa sáng trong không khí thân thiện và ấm cúng, tạo nên một tinh thần đoàn kết và gắn bó, cùng khởi đầu một ngày mới bằng chia sẻ câu chuyện của mỗi người. Việc sử dụng bình nước cá nhân thay vì chai nhựa dùng một lần cũng là một bước tiến quan trọng. Mỗi nhân viên đều được cấp một bình nước cá nhân, giúp giảm thiểu lượng rác thải nhựa đáng kể."}\n{"id": "B-A-b1d29d511570c30e", "claim": "Việc này giúp giảm thiểu lượng túi nhựa và hộp xốp sử dụng một lần.", "evidence": "Tiêu biểu như: Đối với khối văn phòng, lượng nước, giấy, điện hay hộp xốp nhựa được sử dụng luôn được ghi nhận về số lượng cắt giảm, tiết kiệm hay tái sử dụng rõ ràng và thống kê lại theo quý. Tòa nhà trụ sở chính của Coteccons ở TP.HCM cũng thiết lập hệ thống tái sử dụng nước thải từ hệ thống làm lạnh cho việc tưới cây và tối đa hóa việc sử dụng năng lượng mặt trời. Mỗi sáng, mọi người cùng nhau thưởng thức bữa sáng trong không khí thân thiện và ấm cúng, tạo nên một tinh thần đoàn kết và gắn bó, cùng khởi đầu một ngày mới bằng chia sẻ câu chuyện của mỗi người. Việc sử dụng bình nước cá nhân thay vì chai nhựa dùng một lần cũng là một bước tiến quan trọng. Mỗi nhân viên đều được cấp một bình "}\n{"id": "B-A-c0e1dd25a1084064", "claim": "Việc sử dụng bình nước cá nhân thay vì chai nhựa dùng một lần cũng là một bước tiến quan trọng.", "evidence": "Hình ảnh những chiếc xe đạp xếp hàng ngay ngắn trước công trường mỗi sáng là minh chứng rõ ràng cho sự thay đổi tích cực, khi tất cả cùng hướng tới một môi trường sạch hơn và cho mình sức khỏe tốt hơn. Chúng tôi cũng khuyến khích nhân viên ăn sáng tại chỗ thay vì mua mang đi. Việc này giúp giảm thiểu lượng túi nhựa và hộp xốp sử dụng một lần. Coteccons, từ văn phòng đến công trường không còn sử dụng nước đóng chai hay cốc nhựa, mà thay thế bằng những bình nước cỡ lớn và cốc thuỷ tinh dùng nhiều lần. Dự án “Nhà mình” tận dụng những vật liệu xây dựng như sơn, xi măng dư thừa từ các công trình đã hoàn thiện để sử dụng cho nhu cầu chỉnh trang, làm đẹp các công trình công cộng hoặc nhà dân ở nhữn"}\n{"id": "B-A-71ee5d6c15012e5a", "claim": "Thật vui khi tất cả cùng xây dựng ý thức và tạo sự thay đổi từ những việc nhỏ nhất như phân loại rác tại chỗ, giảm thiểu ly nhựa, tái chế rác thải và vật liệu thừa, dọn dẹp vệ sinh công trường để tăng khả năng phân loại rác thải...", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Người Coteccons đã và đang thực hiện giảm thiểu sử dụng chai, hộp nhựa, phân loại rác tại nguồn và xử lý rác thải phù hợp. Phòng Hành chính triển khai hoạt động phân loại rác thải theo 03 nhóm: giấy khô, nhựa khô, rác sinh hoạt hữu cơ từ 03/2023. Coteccons Building. Khối công trường cũng áp dụng tối đa nguyên tắc 3R trong quá trình xây dựng, qua tất cả công đoạn, từ chọn giải pháp, kết cấu, hoàn thiện, cơ điện... Một số sáng kiến nổi bật thực hiện chỉ tiêu 3R của các công trường như tái chế vật liệu từ chất thải cơ điện, kim loại từ dây điện, sử dụng triệt để sơn nước thừa từ các công trình hoàn thiện cho dự án t"}\n{"id": "B-A-39c7ca65069d3c79", "claim": "Đây là hoạt động nằm trong chuỗi chương trình hướng đến mục tiêu phát triển cộng đồng bền vững của Coteccons.", "evidence": "Quản lý, điều phối triển khai các hoạt động ESG trên 3 lĩnh vực hoạt động: công trình, môi trường và cộng đồng bền vững. Chuyên gia tư vấn ESG - Thành viên độc lập tiểu ban: Bà ĐINH THỊ NGỌC BÍCH Tư vấn chiến lược cho các dự án phát triển bền vững nhằm: Cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng những cải thiện trong chuỗi cung ứng, và sử dụng nguồn nhân lực. Tại Coteccons, Tiến sĩ Bích đã đóng góp quan trọng vào các dự án phát triển bền vững và tư vấn chiến lược ESG cho giai đoạn 2023 – 2027, dựa trên các tiêu chuẩn quốc tế. Những nỗ lực của bà đã giúp Coteccons đạt được nhiều thành tựu trong cải thiện môi trường làm việc, thúc đẩy các sáng kiến xanh, định hướng "}\n{"id": "B-A-892d3915f1ac9dc1", "claim": "Chiến lược ESG của Công ty đã gieo vào chúng tôi phương châm “Nơi nào Coteccons đến, nơi đó sẽ xanh và tốt đẹp hơn”.", "evidence": "TABLE Phần I Phần II Phần III Phần IV Phần V Phần VI Con đường của Tổng quan Giới thiệu Quản trị phát triển Kết quả tăng trưởng Thực hành ESG tương lai xanh về Coteccons về báo cáo ESG bền vững Kinh tế theo 5 mục tiêu của Coteccons Tại Coteccons, tất yếu chúng tôi đang thay đổi để tốt hơn. Sau hàng ngàn giờ suy nghĩ, chúng tôi không hề nghi ngờ gì về sự thành công của con đường mình đã chọn. Bây giờ chỉ là vấn đề thực thi, chậm hay nhanh chúng tôi sẽ thực hiện được. Chúng tôi đã tập trung vào các nền tảng và đã đạt được sự ổn định. Bây giờ chúng tôi đang chuyển sang giai đoạn tiếp theo của cuộc hành trình, nơi chúng tôi có thể chấp nhận rủi ro để hướng tới kết quả cao hơn. | | | Tại Coteccon"}\n{"id": "B-A-1534a1d473202821", "claim": "Mỗi bước chạy không chỉ vì sức khỏe và niềm vui của chính mình mà còn có thể đóng góp cây xanh cho các dự án trồng rừng.", "evidence": "Dự án thể thao này còn mang sứ mệnh lớn là lan tỏa rộng rãi ra cộng đồng tinh thần sống hướng về môi trường, thông điệp về phát triển bền vững và gây quỹ cho các chương trình trồng rừng hưởng ứng dự án Việt Nam Xanh. Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh. Mục tiêu là góp 11.000 cây xanh vào dự án trồng rừng nguyên sinh tỉnh Quảng Bình, Quảng Trị. Chương trình đã diễn ra rất thành công. Những món quà được trao đi trọn vẹn, mang lại niềm vui vượt ngoài mong đợi. Coteccons là đơn vị đầu tiên thực hiện một chương trình lớn như vậy tại địa phương nên mang đến niềm vui cho cả người lớn, chứ không chỉ các em nhỏ. Công trường cũ"}\n{"id": "B-A-fa760ae938d11786", "claim": "Công trường cũng thực hiện tốt việc cung cấp các hoạt động chăm sóc người lao động như nhà nghỉ công nhân, phòng và giường chăm sóc y tế, khu vực nước uống miễn phí, thăm hỏi và động viên lao động nữ có con nhỏ, thăm hỏi nhà trọ định kỳ, phát thưởng động viên các cá nhân đóng góp tích cực cho môi trường xanh sạch tại dự án.", "evidence": "TABLE KÊNH TRUYỀN THÔNG/GẶP GỠ | TẦN SUẤT ĐỐI THOẠI | NHỮNG VẤN ĐỀ QUAN TÂM | NỖ LỰC CỦA COTECCONS TRONG VIỆC ĐÁP ỨNG KỲ VỌNG CỦA CÁC BÊN LIÊN QUAN NGƯỜI LAO ĐỘNG | | | • Khảo sát với người lao động về môi trường làm việc, chính sách lương, thưởng, phúc lợi. • Hội nghị người lao động hàng năm. • Các sự kiện định kỳ của Công ty. • Các chương trình đào tạo, huấn luyện. • Các chương trình, ấn phẩm truyền thông nội bộ. • Cơ chế trao đổi trực tiếp, giải quyết khiếu nại, tham vấn hai chiều. | • Thường niên • Thường niên • Hàng năm/quý • Liên tục • Định kỳ hàng tháng • Liên tục | • Sự ổn định trong công việc, môi trường làm việc chuyên nghiệp, công bằng, không phân biệt đối xử. • Chính sách phúc lợ"}\n{"id": "B-A-a0099069ce495fb5", "claim": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Coteccons nhất quán với định hướng phát triển bền vững, phát triển có trách nhiệm bằng cách thực thi cụ thể các nhiệm vụ với môi trường, xã hội và con người xuyên suốt tiến trình thực hiện hoạt động kinh doanh.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons THỰC HÀNH ESG THEO 5 MỤC TIÊU CỦA COTECCONS Coteccons triển khai thực hành ESG một cách quyết liệt trong nội bộ, đồng thời tìm kiếm, xây dựng quan hệ với các đối tác chung mục tiêu phát triển bền vững. Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons 21 20 BÁO CÁO PHÁT TRIỂN BỀN VỮNG COTECCONS"}\n{"id": "B-A-15842539948b52aa", "claim": "Để tạo nên hiệu quả sâu rộng, có giá trị và tác động thực tế, Công ty đang nỗ lực làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả nhân viên và các bên liên quan.", "evidence": "Trong đó, trách nhiệm điển hình là đưa ESG đến mọi lĩnh vực kinh doanh của công ty, làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả nhân viên và các bên liên quan. Các vị trí quản lý cấp cao đã đảm nhiệm 2024 - Hiện tại: Phó Tổng giám đốc, Giám đốc vận hành 2023 - Hiện tại: Thành viên Tiểu ban ESG Hành tinh của chúng ta Nhằm thực hiện nhiệm vụ làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả bên liên quan, đặc biệt là nhân viên - người lao động của Coteccons, Công ty đã xây dựng triết lý và thông điệp truyền thông Future Inside You. Tương lai phụ thuộc vào cách chúng ta tư duy. Phát triển bền vững là kết quả của một lối tư duy hướng về tương lai, đặt con người và môi trườn"}\n{"id": "B-A-1201b2bebde778dc", "claim": "SUSTAINABILITY REPORT 2025 In 2025, Vietnamese enterprises in general, and The PAN Group in particular, faced a convergence of multifaceted challenges: volatility in U.S.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 2 | | In 2025, Vietnamese enterprises in general, and The PAN Group in particular, faced a convergence of multifaceted challenges: volatility in U.S. tariff policies, the increasingly extreme impacts of natural disasters and floods, and escalating global demands for quality, environmental stewardship, and social responsibility. These “giant waves” have become an unavoidable reality. With the theme “Steadfast Heart – Bold Step,” The PAN Group’s 2025 Sustainability Report demonstrates a long-term vision - remaining resolute in our commitment to sustainable development despite short-term fluctuations. Our “Go Global – Go Green” strategy continues to serve as our"}\n{"id": "B-A-4d457423de661f01", "claim": "With the theme “Steadfast Heart – Bold Step,” The PAN Group’s 2025 Sustainability Report demonstrates a long-term vision - remaining resolute in our commitment to sustainable development despite short-term fluctuations.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 2 | | In 2025, Vietnamese enterprises in general, and The PAN Group in particular, faced a convergence of multifaceted challenges: volatility in U.S. tariff policies, the increasingly extreme impacts of natural disasters and floods, and escalating global demands for quality, environmental stewardship, and social responsibility. These “giant waves” have become an unavoidable reality. With the theme “Steadfast Heart – Bold Step,” The PAN Group’s 2025 Sustainability Report demonstrates a long-term vision - remaining resolute in our commitment to sustainable development despite short-term fluctuations. Our “Go Global – Go Green” strategy continues to serve as our"}\n{"id": "B-A-3d18ba36eab98f5f", "claim": "and Khang An Foods established a roadmap toward the Net Zero 2050 goal.", "evidence": "Khang An Foods developed a roadmap toward achieving Net Zero by 2050; Lafooco expanded modern production facilities to meet increasingly stringent international standards; and Aquatex Bentre (ABT) and Sao Ta (FMC) promoted circular economy models in farming areas and processing plants. TABLE SUSTAINABILITY REPORT 2025 2 | | In 2025, Vietnamese enterprises in general, and The PAN Group in particular, faced a convergence of multifaceted challenges: volatility in U.S. tariff policies, the increasingly extreme impacts of natural disasters and floods, and escalating global demands for quality, environmental stewardship, and social responsibility. These “giant waves” have become an unavoidable rea"}\n{"id": "B-A-99fb02e81fa06f2c", "claim": "VN promoted circular economy models across their farming ponds and processing plants.", "evidence": "Khang An Foods developed a roadmap toward achieving Net Zero by 2050; Lafooco expanded modern production facilities to meet increasingly stringent international standards; and Aquatex Bentre (ABT) and Sao Ta (FMC) promoted circular economy models in farming areas and processing plants. Ï Promoted five key themes of Board on strategies and plans agreement with Standard Chartered to access ESG-based financial solutions and services. “Green Economy – Circular Economy” across the Group."}\n{"id": "B-A-864d53d4b34ffc3e", "claim": "– Go Green” strategy continues to serve as our North Star, aimed at elevating the value and position of Vietnamese agricultural products in the global market, embodying the spirit of “embracing difficulties, not avoiding them.”", "evidence": "Our “Go Global – Go Green” strategy continues to serve as our North Star, aimed at elevating the value and position of Vietnamese agricultural products in the global market, embodying the spirit of “embracing difficulties, not avoiding them.” 2025 also marked a year where PAN focused on consolidating internal strength and adapting our models to the new landscape. Our member companies proactively implemented strategies tailored to this new reality: Vinaseed restructured its organization to enhance governance efficiency; The Group contributed more than 7.48 billion VND to community activities, affirming a consistent, long-term commitment to social responsibility. The “Go Global – Go Green” str"}\n{"id": "B-A-dd8d077de85e4ce6", "claim": "36 Regenerative agriculture 122 SOCIAL 130 General Information 14 Sustainability governance and Organizational Structure 36 Journey to building a sustainable future", "evidence": "SUSTAINABILITY REPORT 2025 OVERVIEW OF PAN GROUP JOURNEY TO BUILDING A SUSTAINABLE FUTURE Ï Pan Pacific Joint Stock Company — the predecessor to SUSTAINABILITY PERFORMANCE REPORT ENVIRONMENT 86 130 SOCIAL 2025 SUSTAINABILITY PERFORMANCE INDEX 156 TOWARDS THE FUTURE 158"}\n{"id": "B-A-cb133e2118ab4f9d", "claim": "16 Business areas and distribution channels 18 Principles of Sustainable governance 44 Human resources 132 Supply chain 24 Sustainable development context", "evidence": "OVERVIEW OF PAN GROUP GENERAL INFORMATION SUPPLY CHAIN 14 24 18 26 BUSINESS AREAS AND DISTRIBUTION CHANNELS ASSOCIATION MEMBERSHIPS SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT CONTEXT SDGs Performance Results in 2025 Global Context The Sustainable Development Goals Report 2025 by the"}\n{"id": "B-A-a9ab68caf56381eb", "claim": "50 Attracting talent 138 Association membership 26 Sustainable development strategy 56 Occupational health and safety 140 Creating sustainable value 60 Education & Training", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT SOCIAL Human resources 132 Attracting talent 138 Occupational health and safety 140 Education and Training Worker participation, consultation, and communication on occupational health and safety 120 - 121, 140 - 141 411-1 Incidents of violations involving rights of indigenous peoples 145 403-5 Worker training on occupational health and safety"}\n{"id": "B-A-5011f4b9d9cff96a", "claim": "142 Sustainability goal framework 64 Diversity and equal opportunity 144 Stakeholder engagement 66 Product safety and quality 146 Risk management 72 Community value sharing", "evidence": "142 Diversity and Equal opportunity 144 Sản phẩm an toàn và chất lượng 146 Community value sharing 150 Innovation 154 130 131 SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Workplace Practices DIVERSITY AND EQUAL OPPORTUNITY Ï Career Development Opportunities: A unified performance appraisal system is applied across the"}\n{"id": "B-A-34b447189f476968", "claim": "With the spirit of “Steadfast Heart – Bold Step”, the PAN Group Group remains committed to pursuing economic growth in parallel with sustainable development, contributing to elevating", "evidence": "Group remains committed to pursuing economic growth in parallel with sustainable development, contributing to elevating Vietnamese agriculture on the global map. We believe that only by maintaining resilience in the face of volatility and staying true to core values can a business create long-term impacts for society. TO A LONG-TERM STRATEGY The past year was also one in which PAN focused on strengthening internal capabilities and improving governance quality. With the theme “Steadfast Heart – Bold Step,” The PAN Group’s 2025 Sustainability Report demonstrates a long-term vision - remaining resolute in our commitment to sustainable development despite short-term fluctuations."}\n{"id": "B-A-f6bbf2e89e74d34f", "claim": "Alongside strengthening internal capabilities, PAN continued to expand value chain linkages by collaborating with 50,700 farmers and 221 cooperatives under product off-take models, contributing to improved incomes and sustainable livelihoods in local communities.", "evidence": "The Group focuses on developing a transparent, stable, and sustainable supply chain by strengthening collaboration with farmers, cooperatives, and local partners. Production linkage programs, offtake agreements, and technical support initiatives continue to be implemented, contributing to improved raw material quality, stable market access, and increased income for producers within the agriculture–food value chain. Promoting a green economy Job creation and income improvement for communities In addition to its business activities, PAN continues to maintain and create stable employment opportunities for local workers, while indirectly supporting the livelihoods of thousands of farmers, aquacu"}\n{"id": "B-A-a5063abc23d497ba", "claim": "The “Go Global – Go Green” strategy continues to serve as the guiding principle for all our activities.", "evidence": "Our “Go Global – Go Green” strategy continues to serve as our North Star, aimed at elevating the value and position of Vietnamese agricultural products in the global market, embodying the spirit of “embracing difficulties, not avoiding them.” 2025 also marked a year where PAN focused on consolidating internal strength and adapting our models to the new landscape. Our member companies proactively implemented strategies tailored to this new reality: Vinaseed restructured its organization to enhance governance efficiency; STOCK EXCHANGE Since 2013, when PAN Group restructured and shifted its focus to the agriculture and food sectors, we have used international market expansion as a benchmark fo"}\n{"id": "B-A-8d32f39bcc5f2770", "claim": "PAN remains committed to enhancing the value of Vietnamese agricultural products in international markets, while promoting green transformation and transparent governance, aiming for sustainable growth rather than growth at any cost.", "evidence": "Sustainable governance is the foundation for PAN’s long-term growth and stakeholder trust. The Group is committed to building a transparent governance system that complies with legal requirements and enhances corporate accountability. income for farmers and workers through training, technical support, and the development of sustainable supply chains. Group remains committed to pursuing economic growth in parallel with sustainable development, contributing to elevating Vietnamese agriculture on the global map. We believe that only by maintaining resilience in the face of volatility and staying true to core values can a business create long-term impacts for society. TO A LONG-TERM STRATEGY The"}\n{"id": "B-A-d0ea566ace8a8ae2", "claim": "The most valuable achievement of the Group is the breakthrough in cooperation and cohesion among the units within the agricultural ecosystem.", "evidence": "The “Breakthrough to New Horizons” Annual Report 2024 vividly encapsulates the dynamic production and business activities of the Group while re昀氀ecting the spirit of a new generatio n of resilient, pioneering agricultural and food business - a journey that PAN is proudly pursuing. 3 2 THE PAN GROUP activities at member companies, optimize ef昀椀ciency and enhance breakthrough development capabilities through international cooperation with leading research organizations. The Group will continue to invest heavily in research and development (R&D) to ensure sustainable development and long-term competitiveness. IN R&D, LEADING IN THE DEVELOPMENT OF NEW"}\n{"id": "B-A-be023327976ed7cd", "claim": "100 Sustainable Companies in Vietnam for the 6th consecutive year, the Top 20 listed companies with the best ESG practices, and the Top 50 Pioneering Enterprises committed to enhancing corporate governance standards in Vietnam (VNCG50).", "evidence": "80 100 700 In 2025, PAN Group was honored by the Vietnam Institute of Directors (VIOD) as one of the Top 50 pioneering companies committed to improving corporate governance standards in Vietnam (VNCG50). 2023 2024 2025 The VNCG50 recognizes listed companies with outstanding corporate governance scores in compliance with Vietnamese regulations, while also meeting good governance practices across the Asian region. HOSE PAN GROUP PAN GROUP PAN GROUP PAN GROUP ASOCIO 2025 ENTERPRISES WITH BREAKTHROUGH GOVERNANCE PIONEERING ENTERPRISES COMMITTED TO ENHANCING"}\n{"id": "B-A-115b6f5c5fc17a37", "claim": "Group remains committed to pursuing economic growth in parallel with sustainable development, contributing to elevating Vietnamese agriculture on the global map.", "evidence": "1.45 times higher than the regional minimum wage. With the spirit of “Steadfast Heart – Bold Step”, the PAN Group Group remains committed to pursuing economic growth in parallel with sustainable development, contributing to elevating SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT STRATEGY PAN Group has established its Sustainable Development Strategy 2030, encompassing all three pillars of sustainability: ECONOMIC SOCIAL “ The Group remains committed to its mission"}\n{"id": "B-A-4a7060a550f56b56", "claim": "Green transformation efforts continued to be accelerated, with total rooftop solar power capacity increasing to 5.53 MWp, while circular economy models and biological production continued to be expanded.", "evidence": "MWP rooftop solar power capacity, bringing the total installed capacity to 5.53 MWp. 94 95 As of the end of 2025, PAN Group’s total installed solar power capacity reached 5.53 MWp, with 100% coming from rooftop solar systems. The total renewable electricity generated and consumed across the Group in 2025 was approximately 4,393 MWh. Ï Investment in modern equipment and productivity improvement"}\n{"id": "B-A-8b0052a04f786bd5", "claim": "We are committed to ongoing innovation, transparency, and leadership in building sustainable values for today and future generations.", "evidence": "36 Regenerative agriculture 122 SOCIAL 130 General Information 14 Sustainability governance and Organizational Structure 36 Journey to building a sustainable future We collaborate to We value diverse perspectives and We are committed to creating positive, sustainable impacts We maximize value for all stakeholders 80"}\n{"id": "B-A-1ea42dfb2dd3a67b", "claim": "With a passion to nurture the world, we provide food security, food safety, and nutrition, creating sustainable values on behalf of farmers, families and society.", "evidence": "Email info@thepangroup.vn A pioneering agriculture and food Group offering trusted products and innovative solutions within an integrated value chain from Vietnam, for the world - Farm, Food, Family. Every day, with a passion to nurture the world, we set out to provide food security, food safety, and nutrition, creating sustainable values on behalf of farmers, families and society. TABLE ANNUAL REPORT 2024 CHAPTER 1 THE PAN GROUP OVERVIEW GUIDING PRINCIPLES We appreciate that the presence of a world- class agriculture and food product company increases the food security of Vietnam and the region. We appreciate that agriculture, as well as aquaculture, lies at the core of our country’s econom"}\n{"id": "B-A-83e443a0eeafda40", "claim": "Telephone +84-24 3760 6190 We seek sustainable and profitable growth, maximizing value for all key stakeholders.", "evidence": "We seek sustainable and pro昀椀table growth, maximizing value for all key stakeholders. Website www.thepangroup.vn 12 THE PAN GROUP 13 We promote business practices supporting Vietnam’s sustainability. VISION MISSION Tel +84-24 3760 6190 We offer a fair, professional, inspiring work environment, with opportunities that bene昀椀t our people and their families."}\n{"id": "B-A-2f693a5fe952135d", "claim": "Food Safety Initiative (GFSi) assessment guidelines.", "evidence": "Ï It applies stringent food safety management standards, including ISO, FSSC, BRCGS, Ï Substandard raw materials leading to food safety incidents. Product quality, food safety, and community responsibility risks - GRI 416: CUSTOMER HEALTH AND SAFETY 416-1 Assessment of the health and safety impacts of product and service categories 116 - 148 416-2 Incidents of non-compliance concerning the health and safety impacts of products and services"}\n{"id": "B-A-83ab1892dc3f23aa", "claim": "Initiative (GRI) Bank Safeguard Policy Statement on environmental and social issues.", "evidence": "Ï Legal violations. Pollution and environmental incident risks Ï Complaints from local communities. ENVIRONMENTAL AND SOCIAL RISKS Ï The Group issues context-appropriate policies, with particular focus on income, Ï Commit to “zero” community complaints or lawsuits regarding environmental issues."}\n{"id": "B-A-d89cb8a258392aa2", "claim": "“Green Economy – Circular Economy” across the Group.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT CIRCULAR ECONOMY GRI 306 Promoting a circular economy by designing products for reuse, reducing resource consumption, and repurposing waste are key approaches to minimizing environmental impact, lowering greenhouse gas emissions, and ensuring long-term sustainability. KG EPR Opens the Future for Green Recycling and Promotes a Circular Economy Project “Cosmetic Product Line (Soap, Scrub) from coffee by-products”"}\n{"id": "B-A-ac16e78e45dd1e11", "claim": "The Group has developed and implemented a set of criteria for evaluating suppliers and partners, serving as a basis for selection, collaboration, and periodic monitoring, with priority given to partners that comply with legal regulations and demonstrate stronger adherence to sustainable development standards.", "evidence": "Developed a system for collecting, recording, and reporting ESG data from subsidiaries, with periodic reports to the Sustainable Development Committee. On June 17, 2024, the BoD approved renaming these committees as: Human Resources Committee, Sustainable Development In the seafood sector, PAN Group collaborates with six agents and approximately 170 shrimp farming households, supplying key export markets such as the United States, Europe, and Japan. The Group integrates environmental, social, and human rights criteria into supplier selection and evaluation through its Code of Conduct and periodic assessment system."}\n{"id": "B-A-ec44a6858bb55d58", "claim": "In addition to evaluation activities, PAN emphasizes accompanying and enhancing the capacity of partners within the value chain, aiming to build a responsible, transparent, and resilient supply ecosystem with long-term adaptability.", "evidence": "The company also safeguarded shareholder rights by ensuring access to information, responding to shareholder inquiries, and transparently engaging with investors. an inclusive, safe, and development- driven work environment. The Group also actively collaborates with cooperatives and farmers across the country to build a resilient and sustainable agricultural supply chain, delivering long-term value to both the industry and the wider community. In addition to its business activities, PAN continues to maintain and create stable employment opportunities for local workers, while indirectly supporting the livelihoods of thousands of farmers, aquaculture producers, and partners within the supply c"}\n{"id": "B-A-5127a91c8c4cf7a4", "claim": "The Group also maintained its policy of reinvesting in production, research and development, and human resource capacity building, thereby laying a solid foundation for long-term and sustainable growth.", "evidence": "This not only increases product value but also conserves resources and minimizes environmental impact. PAN is expanding collaborations with partners in sustainable development, laying the foundation for long-term economic cooperation and growth. We have established strategic partnerships with organizations and enterprises both domestically and internationally to promote sustainable development initiatives and to share expertise and knowledge in this 昀椀eld. PAN is progressively integrating environmental and sustainability factors into its investment and production activities, aiming to minimize environmental impacts and promote green growth. Through efficient resource utilization, emissions r"}\n{"id": "B-A-faa8a92ecd732caf", "claim": "With sustainable development as its foundation, PAN focused on enhancing governance capacity, optimizing operational efficiency, and strengthening its core business areas.", "evidence": "ASSESSEMENTS OF THE BOARD OF DIRECTORS ON BUSINESS OPERATIONS CHAPTER 4 ANNUAL REPORT 2024 CORPORATE GOVERNANCE In 2025, the BOD will continue focusing on improving governance capacity and strengthening the internal management system, speci昀椀cally: Ï VFC: Profit before tax decreased by 23% due to The business performance in 2025 demonstrates that PAN has been gradually strengthening its sustainable business model, balancing financial efficiency with responsibilities to stakeholders, and moving toward stable, long-term development."}\n{"id": "B-A-02e831a89cd8e748", "claim": "Amid ongoing economic fluctuations, PAN remained committed to a development strategy that integrates business growth with social responsibility, recognizing human capital as the foundation for sustainable growth.", "evidence": "Through strict legal compliance, PAN continues to affirm its role as a responsible enterprise toward the community and society. Amid ongoing economic fluctuations, PAN remained committed to a development strategy that integrates business growth with social responsibility, recognizing human capital as the foundation for sustainable growth. 32 33 State, ensuring compliance with tax and financial regulations. The Group also maintained its policy of reinvesting in production, research and development, and human resource capacity building, thereby laying a solid foundation for long-term and sustainable growth. ECONOMIC PERFORMANCE GRI 201 In 2025, The PAN Group continued to maintain stable econom"}\n{"id": "B-A-6bf13f3daac5795b", "claim": "The Group places strong emphasis on building a safe, stable, and equitable working environment, while also creating opportunities for employees to enhance their capabilities through appropriate training programs.", "evidence": "PAN places strong emphasis on ensuring stable income and a safe working environment, while enhancing training programs, skill development, and improving the material and spiritual well-being of its employees. 21.8% 70.9% In 2025, PAN Group maintained a stable workforce of EMPLOYEES Agriculture Aquaculture High school and others Ï Ensure a safe, equitable working environment and provide career development opportunities for employees."}\n{"id": "B-A-d7f20cc86bee51cb", "claim": "Cost optimization and value enhancement Development of a sustainable supply chain Across its core business segments, PAN has continued to promote the application of technology and process improvements to optimize production costs, enhance resource efficiency, and increase product value.", "evidence": "Allocation to Sustainability and R&D Fund 2 BILLION VND Enhancing operational efficiency and continuing cost optimization across the entire system through production improvements and tight resource management. Operating expenses & remuneration for BOD SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE ECONOMIC DEVELOPMENT Contributing to the Group’s overall performance was the positive growth of its subsidiaries in key business areas. The seafood and processed food segments continued to deliver strong results, driven by effective expansion in export markets, supply chain optimization, and enhanced product value."}\n{"id": "B-A-4d36a7f3c232a1e5", "claim": "The Group focuses on developing a transparent, stable, and sustainable supply chain by strengthening collaboration with farmers, cooperatives, and local partners.", "evidence": "In addition to its business activities, PAN continues to maintain and create stable employment opportunities for local workers, while indirectly supporting the livelihoods of thousands of farmers, aquaculture producers, and partners within the supply chain. By expanding production, strengthening value chain capacity, and promoting regional linkages, the Group contributes to improving income levels and the quality of life in local communities. In addition, the Group maintains supplementary welfare policies, with a focus on working conditions, employee health, and long-term engagement. SUSTAINABILITY REPORT 2025 OVERVIEW OF PAN GROUP SUPPLY CHAIN AQUACULTURE PAN Group continuously promotes the"}\n{"id": "B-A-899e5b60ffa21573", "claim": "Through efficient resource utilization, emissions reduction, and participation in sustainable finance initiatives, the Group contributes to generating positive indirect economic impacts aligned with long-term sustainable development goals.", "evidence": "100% of official employees are covered by mandatory insurance schemes, contributing to social security and stable livelihoods. In 2025, PAN Group continued to implement its business operations with the objective of generating indirect economic value for communities and stakeholders across the agriculture–food value chain. Through investments in technology, supply chain development, and the promotion of sustainable production models, PAN contributes to improving economic efficiency, enhancing livelihoods, and supporting local economic development in the areas where it operates. This not only increases product value but also conserves resources and minimizes environmental impact. PAN is expand"}\n{"id": "B-A-7b9ad48a1ac52b0e", "claim": "Through strict legal compliance, PAN continues to affirm its role as a responsible enterprise toward the community and society.", "evidence": "TABLE Full legal compliance and support for state policies. Contributions to government and society. SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT COMMUNITY VALUE SHARING Spreading the Value of Sharing PAN Group consistently places strong emphasis on community activities, demonstrating its corporate social responsibility toward society and the environment."}\n{"id": "B-A-d045a8456ea6a8fe", "claim": "Maintaining a transparent, ethical, and compliant business environment not only helps mitigate risks but also serves as a foundation for building trust with stakeholders and achieving long-term sustainable growth.", "evidence": "This not only increases product value but also conserves resources and minimizes environmental impact. PAN is expanding collaborations with partners in sustainable development, laying the foundation for long-term economic cooperation and growth. We have established strategic partnerships with organizations and enterprises both domestically and internationally to promote sustainable development initiatives and to share expertise and knowledge in this 昀椀eld. Sustainable governance is the foundation for PAN’s long-term growth and stakeholder trust. The Group is committed to building a transparent governance system that complies with legal requirements and enhances corporate accountability. inco"}\n{"id": "B-A-2b4fc0651062d959", "claim": "Through these efforts, the Group aims to reinforce a culture of integrity, enhance governance effectiveness, and contribute positively to the sustainable development of both the business and society.", "evidence": "The PAN Group has developed a prudent business plan and a sustainable development strategy to overcome these dif昀椀culties. To achieve these objectives, PAN focuses on sustainable development, addressing environmental, social, and governance issues. The sustainable agriculture model has enabled PAN to af昀椀rm its leadership position both domestically and internationally. With the principle of “sustainable development and community orientation,” PAN not only focuses on business operations but also actively participates in programs and projects that deliver long-term benefits to society. These activities not only contribute to addressing social issues but also help build a brand image associated"}\n{"id": "B-A-3ba1bf273df29477", "claim": "Accordingly, PAN’s sustainable development activities are governed under a three-tier model: (i) The Sustainability Committee under the Board of Directors, responsible for advising the Board on strategies and plans aligned with sustainable development orientation;", "evidence": "PAN’s sustainability agenda is governed by a comprehensive three-tier structure: a Sustainability Committee under the Board of Directors responsible for strategic direction and policymaking; SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Sustainability Committee Established in 2021, the Sustainable Development Committee under the Board of Directors serves as the highest governing body of the Group responsible for overseeing the implementation of its sustainable development strategy, while driving efforts to keep pace with rapid global changes."}\n{"id": "B-A-9c11b8e99fed21d6", "claim": "Member Head of Agriculture Segment Head of Packaged Head of Seafood Segment Foods Segment LEVEL The Pan Group’s Sustainability Department", "evidence": "The agriculture segment achieved revenue of VND 6,040 billion, accounting for 37.3% of the Group’s total revenue – a share comparable to 2022. This segment also contributed 56% of the Group’s overall pre-tax pro昀椀t. AGRICULTURE AQUACULTURE The packaged food segment contributed more positively to pro昀椀t, achieving VND BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Sustainability Department – PAN Group Member in charge of Sustainable Development"}\n{"id": "B-A-3facafa138f58546", "claim": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Sustainability Committee Established in 2021, the Sustainable Development Committee under the Board of Directors serves as the highest governing body of the Group responsible for overseeing the implementation of its sustainable development strategy, while driving efforts to keep pace with rapid global changes.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT STRATEGY PAN Group has established its Sustainable Development Strategy 2030, encompassing all three pillars of sustainability: ECONOMIC SOCIAL “ The Group remains committed to its mission SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT CONTEXT SDGs Performance Results in 2025 Global Context The Sustainable Development Goals Report 2025 by the"}\n{"id": "B-A-6f30062042e755e2", "claim": "Members of the Steering Committee bring deep expertise in their respective fields, forming a strong foundation for managing the Group’s sustainability initiatives.", "evidence": "The establishment of the Group’s Sustainability Steering Committee demonstrated its leadership role in setting and driving sustainability initiatives. PAN and its subsidiaries have developed a comprehensive sustainability data collection system, laying the groundwork for future ESG goals. Additionally, member companies have begun setting ESG (ii) The Sustainability Steering Committee, which plays a leading role in overseeing and ensuring effective management of sustainability across the Group. Its members include the Chief Executive Officer and three representatives from the Group’s key business sectors: Agriculture, Aquaculture, and Food Processing;"}\n{"id": "B-A-88d978d05f209a9a", "claim": "Its roles and responsibilities include: Development Committee Fimex VN Ma Ich Hung – CEO Head of Sustainable Committee Executive Assistant, Workshop", "evidence": "TABLE Ma Ich Hung – CEO | Head of Sustainable Committee | Executive Assistant, Workshop Manager, Chief Financial Officer, Administration Director Nguyen Van Cu – Assistant CEO | Deputy Head of Sustainable Committee | Huynh Thi Ngoc My – Deputy CEO | Head of Sustainable Committee | Deputy QA, Technical Director, Deputy Chief Accountant, Vice Chairman of Trade Union, Workshop Manager, QC Department, Industrial Engineering Department Nguyen Thi Thuy Hang – Deputy QA Manager | Deputy Head of Sustainable Committee | Tran Van Dung – Deputy CEO | Head of Sustainable Committee | Deputy CEO, Head of Administration & HR, Chief Accountant, Plant Director, Fumigation Branch Director, HSE Department, Hea"}\n{"id": "B-A-d43089d1d275eadb", "claim": "Director, Deputy Director of Sales Nguyen Duy Duan – Secretary of Sustainable Committee Head of QC Sustainability Departments at Subsidiaries", "evidence": "TABLE Ma Ich Hung – CEO | Head of Sustainable Committee | Executive Assistant, Workshop Manager, Chief Financial Officer, Administration Director Nguyen Van Cu – Assistant CEO | Deputy Head of Sustainable Committee | Huynh Thi Ngoc My – Deputy CEO | Head of Sustainable Committee | Deputy QA, Technical Director, Deputy Chief Accountant, Vice Chairman of Trade Union, Workshop Manager, QC Department, Industrial Engineering Department Nguyen Thi Thuy Hang – Deputy QA Manager | Deputy Head of Sustainable Committee | Tran Van Dung – Deputy CEO | Head of Sustainable Committee | Deputy CEO, Head of Administration & HR, Chief Accountant, Plant Director, Fumigation Branch Director, HSE Department, Hea"}\n{"id": "B-A-1abda3099c4fe743", "claim": "Vinaseed Nguyen Van Si – Head of Sustainable General Oversight of Sustainability A network of Sustainable Development Committees and designated officers at member companies has been established to ensure that the Group’s policies are widely communicated and effectively implemented across all subsidiaries, while progressively enhancing awareness and aligning behaviors with the Group’s development orientation.", "evidence": "Nguyen Thi Thuy Hang – Deputy Head of Sustainable Committee Deputy QA Manager Ï Monitoring and reporting the implementation of VFC Tran Van Dung – Head of Sustainable Committee sustainability initiatives at member companies to the TABLE REPORT PREPARATION TEAM | Ï Mr. Nguyen Van Si – Head of Sustainability. Ï Ms. Luong Nguyen Khanh Linh – Project Specialist. With support from Sustainability Coordinators across PAN Group’s member companies: Ï Report Reviewer: Mr. Nguyen Hong Hiep – Director of External Affairs and Communications. Ï Report Approver: Ms. Nguyen Thi Tra My – Chief Executive Officer and Head of the Sustainability Steering Committee of PAN Group."}\n{"id": "B-A-16fd481ff7f731e9", "claim": "584 Nha Trang Nguyen Anh Duc – Technical Director General Oversight of short-, medium-, and long-term sustainability objectives to the management of member companies.", "evidence": "Assistant CEO medium- and long-term sustainability objectives. Lafooco Huynh Thi Ngoc My – Head of Sustainable Committee Deputy QA, Technical Director, Deputy CEO TABLE Ma Ich Hung – CEO | Head of Sustainable Committee | Executive Assistant, Workshop Manager, Chief Financial Officer, Administration Director Nguyen Van Cu – Assistant CEO | Deputy Head of Sustainable Committee | Huynh Thi Ngoc My – Deputy CEO | Head of Sustainable Committee | Deputy QA, Technical Director, Deputy Chief Accountant, Vice Chairman of Trade Union, Workshop Manager, QC Department, Industrial Engineering Department Nguyen Thi Thuy Hang – Deputy QA Manager | Deputy Head of Sustainable Committee | Tran Van Dung – Depu"}\n{"id": "B-A-e19a0ec89e7c80a1", "claim": "It was established to ensure that the activities of all stakeholders align with the Group’s sustainable development orientation and its commitment to: “Providing high-quality products with reliable origins within a fully integrated value chain.” management, and environmental protection to address climate change.", "evidence": "To achieve this, PAN has strengthened a flexible and streamlined sustainable governance system, implementing activities from policy planning to execution, measurement, and reporting in a systematic and effective manner, ensuring alignment with social development and environmental protection goals. Accordingly, PAN’s sustainable development activities are governed under a three-tier model: (i) The Sustainability Committee under the Board of Directors, responsible for advising the Board on strategies and plans aligned with sustainable development orientation; The PAN Group is committed to strictly complying with state regulations on the environment, labor, and social affairs. We have establish"}\n{"id": "B-A-617db49a46a399db", "claim": "PAN Group is committed to practicing responsible corporate governance principles.", "evidence": "TABLE BSUÁSOT CAÁINOA PBHILÁITT YT RRIEỂPNO BRỀTN 2 0V2Ữ5NG 2025 PRINCIPLES OF SUSTAINABLE GOVERNANCE PAN Group is committed to practicing responsible corporate governance principles. By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders. Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization. Our commitments and statements on sustainable governance are also clearly presented in the 2025 Annual Report. 44 The Group is committed to building a strong and resil"}\n{"id": "B-A-98768411ac71324a", "claim": "By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders.", "evidence": "TABLE BSUÁSOT CAÁINOA PBHILÁITT YT RRIEỂPNO BRỀTN 2 0V2Ữ5NG 2025 PRINCIPLES OF SUSTAINABLE GOVERNANCE PAN Group is committed to practicing responsible corporate governance principles. By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders. Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization. Our commitments and statements on sustainable governance are also clearly presented in the 2025 Annual Report. 44 Vietnamese agriculture on the global map. We believe "}\n{"id": "B-A-0c6c04638a287faf", "claim": "Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization.", "evidence": "TABLE BSUÁSOT CAÁINOA PBHILÁITT YT RRIEỂPNO BRỀTN 2 0V2Ữ5NG 2025 PRINCIPLES OF SUSTAINABLE GOVERNANCE PAN Group is committed to practicing responsible corporate governance principles. By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders. Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization. Our commitments and statements on sustainable governance are also clearly presented in the 2025 Annual Report. 44 International Finance Corporation (IFC) and Standard "}\n{"id": "B-A-a0c83d5fe6798533", "claim": "The Practice Principles focuses on realizing the vision of a sustainable farm-to-table value chain, while ensuring food security and nutritional safety.", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Practice Principles PRINCIPLES OF SUSTAINABLE GOVERNANCE First issued in 2017 as the Product Principles, the principles was revised and updated in 2019 into the Email info@thepangroup.vn A pioneering agriculture and food Group offering trusted products and innovative solutions within an integrated value chain from Vietnam, for the world - Farm, Food, Family. Every day, with a passion to nurture the world, we set out to provide food security, food safety, and nutrition, creating sustainable values on behalf of farmers, families and society."}\n{"id": "B-A-fbed2f7bb9a2d576", "claim": "It also emphasizes requirements on traceability, waste", "evidence": "Ï Product technical specifications Ï Equipment and machinery Ï Product quality Ï General requirements Ï Facilities and factories Ï Product traceability Traceability PAN Group places strong emphasis on product traceability. Accordingly, we have implemented an integrated traceability system covering the entire supply chain, starting from input raw materials. The Group requires that all supplied inputs originate from sources not associated with biodiversity violations."}\n{"id": "B-A-53bcc219d9c3aeac", "claim": "Our commitments and statements on sustainable governance are also clearly presented in the 2025 Annual Report.", "evidence": "TABLE BSUÁSOT CAÁINOA PBHILÁITT YT RRIEỂPNO BRỀTN 2 0V2Ữ5NG 2025 PRINCIPLES OF SUSTAINABLE GOVERNANCE PAN Group is committed to practicing responsible corporate governance principles. By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders. Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization. Our commitments and statements on sustainable governance are also clearly presented in the 2025 Annual Report. 44 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE "}\n{"id": "B-A-6df343cdfaf05860", "claim": "By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long-", "evidence": "By complying with the law, safeguarding core values, and strengthening the company’s reputation, we aim to ensure long- term success and foster a high level of trust among all stakeholders. Our efforts in this area are further reinforced by the integration of sustainability considerations into all processes and at every level of the organization. Vietnamese agriculture on the global map. We believe that only by maintaining resilience in the face of volatility and staying true to core values can a business create long-term impacts for society. Alongside strengthening internal capabilities, PAN continued to expand value chain linkages by collaborating with 50,700 farmers and 221 cooperatives u"}\n{"id": "B-A-706cf56cbd50853a", "claim": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Code of Conduct Recognized Sustainability Standards Issued in 2017 together with the Practice Principles, these two components form a comprehensive framework outlining PAN Group’s perspectives, requirements, and internal as well as external measures in its sustainable development strategy.", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Practice Principles PRINCIPLES OF SUSTAINABLE GOVERNANCE First issued in 2017 as the Product Principles, the principles was revised and updated in 2019 into the SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY GOAL FRAMEWORK With a vision to become a leading agriculture and food group in ESG practices in the region by 2030, PAN Group has been systematically implementing initiatives to integrate sustainability into the strategies and governance practices of its member companies. In 2024, the Group launched the “Sustainable Business Assessment and"}\n{"id": "B-A-0cc8bf0b623f3f4d", "claim": "Board for the 5th term (2025–2028), demonstrating PAN’s ongoing commitment to fostering a responsible business community.", "evidence": "Through strict legal compliance, PAN continues to affirm its role as a responsible enterprise toward the community and society. Amid ongoing economic fluctuations, PAN remained committed to a development strategy that integrates business growth with social responsibility, recognizing human capital as the foundation for sustainable growth. 32 33 ANNUAL REPORT 2024 CHAPTER 2 2024 BUSINESS RESULTS ENVIRONMENTAL & SOCIAL RESPONSIBILITY REPORT The PAN Group is committed to sustainable development, pursuing strategies that deliver long-term value for both the business and the community."}\n{"id": "B-A-344ade1d1a5f0bc6", "claim": "PANSI is the Group’s internal sustainability assessment framework, implemented since 2023 to guide and promote the sustainable transformation of its member companies.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY GOAL FRAMEWORK With a vision to become a leading agriculture and food group in ESG practices in the region by 2030, PAN Group has been systematically implementing initiatives to integrate sustainability into the strategies and governance practices of its member companies. In 2024, the Group launched the “Sustainable Business Assessment and SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Standardized and Transparent ESG Data Outcomes after two years of implementation Since October 2024, PAN Group has deployed an internal sustainability data collection system across all member companies."}\n{"id": "B-A-4452382a6525bbda", "claim": "PAN’s performance showed in the “Top 100 Sustainable Companies in Vietnam” assessment and announcement program for the period 2023–2025, organized by VBCSD – VCCI.", "evidence": "Total Assessment Score Overall Rank Source: Announcement of the results of the CSI100 Program by VCCI. 46 47 SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT WATER AND WASTEWATER MANAGEMENT Water consumption during the 2023–2025 period Unit 2023 2024 2025 GRI 303 Water supply by municipal companies m3"}\n{"id": "B-A-924981a2ea97af36", "claim": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Corporate Culture Corporate culture is the core foundation that creates intrinsic strength and is closely tied to the Group’s development.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE REPORT INFORMATION REPORT OVERVIEW PAN Group’s 2025 Sustainability Report is prepared and disclosed in accordance with the Global Reporting Initiative (GRI) Standards. SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE GOVERNANCE GRI 2 STANDALONE SUSTAINABILITY REPORTS PREPARED IN ACCORDANCE SUSTAINABILITY GOVERNANCE"}\n{"id": "B-A-0e2ae045f1a9585d", "claim": "It serves as a key driver for PAN to achieve its ambition of becoming a leading agriculture and food group, realizing the aspiration of “Born to nourish the world.” With a mindset of innovation, responsibility, and long-term commitment, PAN not only delivers high-quality products but also spreads positive values, contributing to the development of a sustainable and prosperous agriculture–food ecosystem.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Collaboration and Development The acceleration of modern science and technology across the agriculture and aquaculture value chain is seen as a key driver enabling PAN Group’s ecosystem to move both faster and further. We aim not only for speed but also for long-term, sustainable growth. This cultural fabric serves as a potent internal power, driving the Group to grow stronger and ultimately ful昀椀ll the shared dream - to become the leading agriculture and food corporation with the aspiration “Born to nourish the world”. Every member of PAN believes that true value is built over time through unwavering dedication. We are committed to our mi"}\n{"id": "B-A-e42f960cc6c97f29", "claim": "PAN Group has also been included in the Top 20 of the Vietnam Sustainability Index (VNSI) assessment program for five consecutive years.", "evidence": "SUSTAINABLE COMPANIES VIETNAM SUSTAINABILITY SUSTAINABLE COMPANIES IN VIETNAM INDEX (VNSI) IN VIETNAM (CSA) VCCI FORTUNE INVESTMENT BRIDGE 20 2023 2024 2025 PAN Industry Average VN100 Index Average Source: HOSE’s Sustainable Development Assessment Scorecard In addition, PAN Group complies with the IFC Sustainability"}\n{"id": "B-A-7e222a514a659222", "claim": "The VNSI, initiated by the Ho Chi Minh City Stock Exchange (HOSE) in 2017, evaluates the level of compliance of listed companies with sustainability criteria across environmental, social, and corporate governance (ESG) dimensions.", "evidence": "The Company’s shares have been of昀椀cially listed and traded on the Ho Chi Minh Stock Exchange (HOSE) with the stock symbol PAN since 18 November 2010. 3 Sao Ta Foods Joint Stock Company (ii) Soc Trang 47.21% 50.12% 80 100 700 In 2025, PAN Group was honored by the Vietnam Institute of Directors (VIOD) as one of the Top 50 pioneering companies committed to improving corporate governance standards in Vietnam (VNCG50). 2023 2024 2025 The VNCG50 recognizes listed companies with outstanding corporate governance scores in compliance with Vietnamese regulations, while also meeting good governance practices across the Asian region."}\n{"id": "B-A-f98913f2854926aa", "claim": "We collaborate to We value diverse perspectives and We are committed to creating positive, sustainable impacts We maximize value for all stakeholders 80", "evidence": "We take responsibility for our impact, acting voluntarily and sincerely to uphold PAN’s corporate social responsibility commitments. We maximize value for all stakeholders within our ecosystem and proactively seek feedback to drive continuous improvement. As a responsible organization, we continuously assess our activities and their impacts, and develop appropriate plans aligned with stakeholders’ expectations and interests— minimizing negative impacts while enhancing positive contributions arising from our operations. Through regular dialogue mechanisms to understand stakeholder concerns, PAN Group strives to address legitimate expectations and deliver value to all stakeholders."}\n{"id": "B-A-63684537f7c478d2", "claim": "20 2023 2024 2025 PAN Industry Average VN100 Index Average Source: HOSE’s Sustainable Development Assessment Scorecard In addition, PAN Group complies with the IFC Sustainability", "evidence": "SOLIDARITY CREATIVITY DEDICATION EFFICIENCY PAN’S ASSESSMENT SCORES HIGHER THAN THE INDUSTRY AVERAGE VNSI Scorecard: PAN Group (2023 – 2025) DO THE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Sustainability Department – PAN Group Member in charge of Sustainable Development"}\n{"id": "B-A-f17654bd5d045b60", "claim": "The IFC ESG Performance Indicators provide a globally recognized sustainability framework based on the Environmental and Social Performance Standards and Corporate Governance Methodology of IFC.", "evidence": "Ï Report Approver: Ms. Nguyen Thi Tra My – Chief Executive Officer This report is an essential component of the performance monitoring and reporting system, aligned with the IFC Performance Standards and the PAN Sustainability Ï Water conservation Ï Safe and friendly working environment Farmers PAN Group’s Sustainable Development Target Framework is designed to integrate environmental, social, and governance (ESG) factors into its long-term development strategy, in alignment with international reporting standards such as the GRI Standards and"}\n{"id": "B-A-ccbb001781aebd84", "claim": "Out of 169 SDG (Sustainable Development Goals) targets, 139 can be assessed using global trend data from the 2015 baseline to the most recent year, supplemented by monitoring agency analyses.", "evidence": "In contrast, 48% of the targets show insufficient progress, including 31% with only marginal improvement and 17% with no progress at all. Most concerning, 18% of the targets have regressed compared to the 2015 baseline. General Overview In 2025, the trend of sustainable development has shifted significantly from management ambition to operational expectation. SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT CONTEXT SDGs Performance Results in 2025 Global Context The Sustainable Development Goals Report 2025 by the"}\n{"id": "B-A-55bbe623a5e4b88b", "claim": "General Overview In 2025, the trend of sustainable development has shifted significantly from management ambition to operational expectation.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE RISK MANAGEMENT Risk Management Process Risk Management Policies and Procedures PAN Group has established a multi-layered risk management system structured vertically, spanning from operational units to the executive level of member companies and up to the management level of PAN Group. ANNUAL REPORT 2024 CHAPTER 1 General Information of The PAN Group RISK MANAGEMENT PLAN IN 2025 OBJECTIVES Ensure stable business operations and sustainable growth amid market"}\n{"id": "B-A-210a6c1c10e27434", "claim": "The increase in new regulations, stricter investor scrutiny, and rising stakeholder demands have required organizations to integrate ESG compliance directly into their business strategies and daily operations.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE STAKEHOLDER ENGAGEMENT Stakeholder Influence Chart 7 6 Stakeholders are those who are directly or indirectly affected by PAN Group’s operations. Compliance In 2025, PAN Group continued to commit to full compliance with tax regulations in all countries and localities where it operates. Tax compliance is carried out on the principles of transparency, accuracy, and responsibility, contributing to state budget revenues and reinforcing the Group’s business ethics. Integrating anti-corruption into operations"}\n{"id": "B-A-1389a5731e004035", "claim": "also requires large companies to assess environmental and social impacts across their value chains in accordance with the Corporate Sustainability Due Diligence Directive (CSDD).", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY STRATEGY We are committed to minimizing negative environmental impacts and continuously improving products and technologies toward greater environmental friendliness. The Group has established environmental management systems in line with international standards such as ISO SUSTAINABLE COMPANIES SUSTAINABILITY REPORTS IN ACCORDANCE WITH GRI STANDARDS SUSTAINABILITY SCORE - HOSE IN VIETNAM (CSI) VBCSD, VCCI 8 9"}\n{"id": "B-A-3165ca45598889bf", "claim": "31% On track or target met European market requirements, such as the EU Green Deal, are becoming increasingly stringent.", "evidence": "We position ourselves as an active and proactive player in the green transition across all the countries and territories where we operate. Sustainable development and the green economy are increasingly becoming mandatory standards in international markets, while the rise of trade protectionism and technical barriers is placing higher demands on exporting enterprises. Although these requirements present challenges, they also open up significant opportunities for Vietnamese businesses. requirement clients across major markets, such as the European Union, the United States, and Japan. Fairtrade Certification 6,439 From 30 to 50 Achieved by Fimex VN"}\n{"id": "B-A-5a6c50024733daf9", "claim": "The EU is also expected to expand the scope of the Carbon Border Adjustment Mechanism (CBAM) to more sectors in the future, alongside the implementation of the Regulation on Deforestation-free Products (EUDR).", "evidence": "Ï The report focuses on the agriculture and food sectors within Vietnam, REPORTING SCOPE as the Group’s operations outside Vietnam are currently not significant. The report complies with the requirements of the Amended and supplemented on June 17, 2024 At the same time, to protect the interests of the Company and its shareholders, the Company also issued a new Regulation on the Company’s Capital Representatives at companies in which PAN"}\n{"id": "B-A-38320fc5860a82fe", "claim": "Marginal progress Stagnation 18% 18% Regression According to the report by the Sustainable Development Solutions Network (SDSN), Vietnam ranked 61st out of", "evidence": "In contrast, 48% of the targets show insufficient progress, including 31% with only marginal improvement and 17% with no progress at all. Most concerning, 18% of the targets have regressed compared to the 2015 baseline. General Overview In 2025, the trend of sustainable development has shifted significantly from management ambition to operational expectation. an organization representing Vietnam in the global sustainable development network of the World Business Council for Sustainable Development. Participation in VBCSD demonstrates PAN’s consistent orientation toward adopting sustainable development practices in line with international standards, while strengthening the alignment between i"}\n{"id": "B-A-40597d6cb7f3d288", "claim": "The SDG Index is a composite measure that evaluates each country’s performance across all 17 goals—from poverty reduction, education, and healthcare to clean energy and climate action—and ranks countries accordingly.", "evidence": "167 countries in the SDG Index 2025, which measures progress toward the 17 Sustainable Development Goals (SDGs). Vietnam’s SDG Index score reached 73.35 out of 100, significantly higher than the regional average. Within ASEAN, Vietnam currently ranks second after Ï Execute community support programs focusing on education, healthcare, and local economic development. 158 159"}\n{"id": "B-A-8df693eb8c7f2aa9", "claim": "This commitment was to be updated in 2022 to align with the ambition of achieving net-zero emissions by 2050.", "evidence": "Ï Ensure that sourcing areas are Ï Commit to achieving net-zero emissions by 2050. 5.09 4.97 Reduced 73,027 77,015 53,538 TONS CO2/ BILLION VND 20,456 9,415 10,475 CO2 emissions intensity per revenue compared PAN Group, with its ambition to become a leading agriculture and food corporation, has committed to achieving net zero emissions by 2050."}\n{"id": "B-A-e0636010b5cd8bf0", "claim": "5, 2022, Vietnam aims to reduce total methane emissions by at least 30% by 2030 compared to 2020 levels across sectors including crop production, livestock, solid waste management, wastewater treatment, oil and gas extraction, coal mining, and fossil fuel consumption.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. targets for the 2025–2030 period. Water resource management: The PAN Group has several water-intensive businesses, particularly seafood processing accounting for over 90% of PAN’s total annual water consumption. Recognizing the importance of water resources, the Group has adopted sustainable water management solutions, includi"}\n{"id": "B-A-5dfc42060ed393ed", "claim": "In 2025, ministries and agencies also piloted a carbon trading exchange through 2028, with official operation expected from 2029.", "evidence": "The EU is also expected to expand the scope of the Carbon Border Adjustment Mechanism (CBAM) to more sectors in the future, alongside the implementation of the Regulation on Deforestation-free Products (EUDR). In 2025, the EU Moderate progress 17% 17% Exchange rate risks Exchange rate risk is the risk that the fair value of the planned future cash 昀氀ows 昀氀uctuate due to changes in exchange rates. PAN may be exposed to this risk through its subsidiaries and af昀椀liates engaged in import and export and whenever they raise foreign capital. 57 56 THE PAN GROUP"}\n{"id": "B-A-449afa6764db8628", "claim": "To achieve these results, the Group’s leadership has clearly defined a development strategy closely linked to sustainability, aiming to meet its five-year target of becoming a leading agriculture and food group in Vietnam with revenue of USD 1 billion and profit of USD 100 million.", "evidence": "The PAN Group has developed a prudent business plan and a sustainable development strategy to overcome these dif昀椀culties. To achieve these objectives, PAN focuses on sustainable development, addressing environmental, social, and governance issues. The sustainable agriculture model has enabled PAN to af昀椀rm its leadership position both domestically and internationally. It serves as a key driver for PAN to achieve its ambition of becoming a leading agriculture and food group, realizing the aspiration of “Born to nourish the world.” With a mindset of innovation, responsibility, and long-term commitment, PAN not only delivers high-quality products but also spreads positive values, contributing "}\n{"id": "B-A-1c038dc20d50d9b3", "claim": "The Group pursues a sustainable agriculture and aquaculture orientation, completing a closed-loop value chain from farm to table.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE CLOSED-LOOP VALUE CHAIN PAN focuses on building a sustainable raw material production foundation across its three key sectors—agriculture, aquaculture, and packaged foods—through investments in seed research and development, the application of biotechnology, and the implementation of internationally certified farming and cultivation models. A pioneering agriculture and food Group offering trusted products and innovative solutions within an integrated value chain from Vietnam, for the world - Farm, Food, Family. By 2030, become a leading ESG Agriculture and Food corporation in the region. CONTINUING TO SEEK M&A OPPORTUNITIES IN AGRICULTURE,"}\n{"id": "B-A-cd882882fa4b23fb", "claim": "Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024).", "evidence": "Member companies proactively restructured, innovated products, invested in technology, and advanced green transformation. The area of high-quality, low-emission rice models in the Mekong Delta was expanded to 47,520 hectares—an increase of 47 times compared to 2024. Organic EU Organic JAS HECTARES of high-quality, low-emission rice cultivation area Organic USDA HECTARES of linked coffee raw material areas certified organic under USDA, EU, and JAS standards At the same time, VinaRice – Vinaseed was granted a Certificate of Trademark Usage"}\n{"id": "B-A-fc81f15c0abc2a7a", "claim": "Delta by 2030,” aiming to bring Vietnam’s green, low- emission rice to global markets.", "evidence": "Rights for the label “Green Low-Emission Vietnamese Rice” on April 24, 2025, by the Vietnam Rice Industry Association, for rice products sourced from the company’s project implementation areas. program across 10 member companies. Ï Increased the cultivated area of Ï Initiated the “Life form Forestry” Development Subcommittee (now representatives of three key business high-quality, low-emission rice forty-sevenfold to 47,520 hectares in the Mekong Delta region. Project, aiming to plant 1 million trees by 2030."}\n{"id": "B-A-3f04b927703dd5e8", "claim": "We position ourselves as an active and proactive player in the green transition across all the countries and territories where we operate.", "evidence": "TARGET ACHIEVEMENT) The plan remains aligned with the Group’s strategic direction, aiming to position the Company as a leading player in the Agriculture and Food sector across the region, providing safe, high-quality, and fully traceable products for both domestic and international markets. Cash dividend MINIMUM 5% Key directions for 2025 include: Compliance In 2025, PAN Group continued to commit to full compliance with tax regulations in all countries and localities where it operates. Tax compliance is carried out on the principles of transparency, accuracy, and responsibility, contributing to state budget revenues and reinforcing the Group’s business ethics. Integrating anti-corruption int"}\n{"id": "B-A-602985462923b1e5", "claim": "Sustainable development and the green economy are increasingly becoming mandatory standards in international markets, while the rise of trade protectionism and technical barriers is placing higher demands on exporting enterprises.", "evidence": "an organization representing Vietnam in the global sustainable development network of the World Business Council for Sustainable Development. Participation in VBCSD demonstrates PAN’s consistent orientation toward adopting sustainable development practices in line with international standards, while strengthening the alignment between its corporate strategy and Vietnam’s sustainable development priorities. LEAN methodologies, helping to minimize waste throughout the manufacturing process. PAN prioritizes investment in high-efficiency motors and the installation of inverters (VFDs) to reduce energy consumption during operations. The transition to green energy is an inevitable trend and a mand"}\n{"id": "B-A-4e487defd73fa27b", "claim": "Among them, Khang An Foods – a member company of PAN Group – is also subject to regulations on greenhouse gas emission reduction, GHG inventory, and disclosure of environmental and social information.", "evidence": "the effectiveness of implementation plans. layer protection. *Currently, PAN Group has only one entity, Khang An Foods, that is required to conduct greenhouse gas inventory under Decision No. have set sustainable development targets for has prepared a greenhouse gas inventory report in accordance with Government Decree No. 06/2022/NĐ-CP on greenhouse gas emission reduction and ozone have established monthly data sustainability goals and annual action plans. the 2025–2030 period. collection systems."}\n{"id": "B-A-8270b0dc691e0e23", "claim": "This requires us to continuously improve production processes and reduce energy-related emissions to effectively meet emission reduction requirements while ensuring transparency in disclosures.", "evidence": "enhance the Ï Direct dialogues. Protect the Ï Continuously improve production processes to optimize value chain. Ï Communications via PAN’s natural resource efficiency, reduce emissions, and adopt website, suggestion letters, environment and eco-friendly technologies such as renewable energy, This requires a daily commitment to raising awareness and ensuring compliance with laws, regulations, and ethical standards. Ï Environmental Policy: Focus on pollution prevention, efficient use of resources (energy, water, land), reduction of greenhouse gas emissions, and promotion of sustainable production models. Our Code of Conduct serves as a guiding compass, helping us stay on the right path and fu"}\n{"id": "B-A-08268d2e9f5ce6ea", "claim": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Collaboration and Development The acceleration of modern science and technology across the agriculture and aquaculture value chain is seen as a key driver enabling PAN Group’s ecosystem to move both faster and further.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT REGENERATIVE AGRICULTURE Aquaculture > Shrimp farming success rate GRI 304 PAN Group’s vision is to scale regenerative agriculture across its entire value chains. SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE CLOSED-LOOP VALUE CHAIN PAN focuses on building a sustainable raw material production foundation across its three key sectors—agriculture, aquaculture, and packaged foods—through investments in seed research and development, the application of biotechnology, and the implementation of internationally certified farming and cultivation models."}\n{"id": "B-A-93313aea8b3b5c52", "claim": "Scientific Council, marking a significant milestone in its journey to advance research and apply science and technology to improve seed quality and promote sustainable development.", "evidence": "research and development in agricultural science; high-tech research and development activities: industrial-scale fermentation technology for recombinant microbial strains; Ï Member of the Vietnam Food Safety Science PAN Group has been a member of the Vietnam Business Council for Sustainable Development since 2019—"}\n{"id": "B-A-6c35025c332db11b", "claim": "Khang An Foods developed a roadmap toward achieving Net Zero by 2050;", "evidence": "VFC shifted its focus toward developing organic and biological crop protection products; and Khang An Foods established a roadmap toward the Net Zero 2050 goal. Meanwhile, Lafooco expanded its modern facilities to meet rising international standards, while Aquatex Bentre and Fimex Ï Vietnam Business Council for Sustainable Development. Ï Honored as one of the Best Companies to Work for in Asia by HR Asia2019. Ï Committed alongside the Government of Vietnam to achieve the Net Zero 2050 target."}\n{"id": "B-A-323fe0acb08b48c0", "claim": "and Aquatex Bentre (ABT) and Sao Ta (FMC) promoted circular economy models in farming areas and processing plants.", "evidence": "VN promoted circular economy models across their farming ponds and processing plants. Throughout more than a decade of pursuing sustainable development, The PAN Group continues to transform unavoidable challenges into drivers for innovation, steadily approaching and conquering higher global benchmarks. 2 3 Ï Promoted five key themes of Board on strategies and plans agreement with Standard Chartered to access ESG-based financial solutions and services. “Green Economy – Circular Economy” across the Group."}\n{"id": "B-A-dda98dabd56c1771", "claim": "In 2025, the Group implemented a program to set grassroots-level sustainability targets for its member companies.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY GOAL FRAMEWORK With a vision to become a leading agriculture and food group in ESG practices in the region by 2030, PAN Group has been systematically implementing initiatives to integrate sustainability into the strategies and governance practices of its member companies. In 2024, the Group launched the “Sustainable Business Assessment and Disclosure” program across its entire ecosystem, while officially adopting the PANSI framework as a tool to assess the level of implementation and effectiveness of sustainability practices at its subsidiaries. In 2025, the Group focused on advancing the sustainability target- setting phase"}\n{"id": "B-A-9b8026ee0e7db1a7", "claim": "We prioritize building awareness of the importance of target setting at the operational level before establishing Group-wide sustainability goals.", "evidence": "The establishment of the Group’s Sustainability Steering Committee demonstrated its leadership role in setting and driving sustainability initiatives. PAN and its subsidiaries have developed a comprehensive sustainability data collection system, laying the groundwork for future ESG goals. Additionally, member companies have begun setting ESG Disclosure” program across its entire ecosystem, while officially adopting the PANSI framework as a tool to assess the level of implementation and effectiveness of sustainability practices at its subsidiaries. In 2025, the Group focused on advancing the sustainability target- setting phase through targeted advisory and training programs, aimed at strengt"}\n{"id": "B-A-32949f0289ddf89a", "claim": "ENVIRONMENTAL The Group remains committed to its mission of contributing to food security while providing safe and nutritious products.", "evidence": "In terms of products, the Group will prioritize deep- processed products to build competitive advantages. safe and nutritious products. These strategies and objectives serve as the primary framework guiding the Group and its member companies in continuously creating shared value for stakeholders. In 2025, the Group implemented a program to set grassroots-level sustainability targets for its member companies. We prioritize building awareness of the importance of target setting at the operational level before establishing Group-wide sustainability goals. of contributing to food security while providing"}\n{"id": "B-A-94fabe24404fabef", "claim": "It plays a role in enhancing value by transforming raw materials into high-quality, differentiated, and competitive products for both domestic and international markets—particularly those with stringent requirements on quality, food safety, and sustainability such as the United States, Europe, and Japan.", "evidence": "Ï It applies stringent food safety management standards, including ISO, FSSC, BRCGS, Ï Substandard raw materials leading to food safety incidents. Product quality, food safety, and community responsibility risks SUSTAINABILITY REPORT 2025 OVERVIEW OF PAN GROUP DISTRIBUTION NETWORK The Group’s products are widely distributed in both domestic and international markets through an extensive distribution network of more than 450 distributors reaching 145,000 retail outlets across all 63 provinces and cities nationwide. Products such as rice, seafood, fresh flowers, cashews, and nuts have also expanded into many key international markets (nearly 45 countries), including Japan, Europe, the United S"}\n{"id": "B-A-7ff4c364bdd1f422", "claim": "Through technical support programs, offtake commitments, capacity development initiatives, and the assurance of safe working conditions,", "evidence": "Ï Implementing production linkage models that provide program, distributing a total of technical support and offtake commitments for farmers and cooperatives. The Group focuses on developing a transparent, stable, and sustainable supply chain by strengthening collaboration with farmers, cooperatives, and local partners. Production linkage programs, offtake agreements, and technical support initiatives continue to be implemented, contributing to improved raw material quality, stable market access, and increased income for producers within the agriculture–food value chain. Promoting a green economy Job creation and income improvement for communities"}\n{"id": "B-A-7a9c0fe5c5e441d7", "claim": "PAN Group develops a closed-loop value chain model—Farm – Food – Family—as a core component of its business model and long-term sustainable development strategy, aiming to ensure the creation of sustainable value for the", "evidence": "To achieve these results, the Group’s leadership has clearly defined a development strategy closely linked to sustainability, aiming to meet its five-year target of becoming a leading agriculture and food group in Vietnam with revenue of USD 1 billion and profit of USD 100 million. The Group pursues a sustainable agriculture and aquaculture orientation, completing a closed-loop value chain from farm to table. LONG-TERM AND MEDIUM-TERM DEVELOPMENT STRATEGY SUSTAINABLE DEVELOPMENT GOALS VISION ESG GOALS The Group continues to implement its sustainable development strategy and increase shareholder value in the medium and long term through the following strategic objectives:"}\n{"id": "B-A-6e6c9efc09ab1cff", "claim": "These efforts help improve resource efficiency, control environmental risks, ensure biosafety, and enhance traceability across the supply chain—key factors in adapting to climate change and meeting increasingly stringent market requirements.", "evidence": "Cost optimization and value enhancement Development of a sustainable supply chain Across its core business segments, PAN has continued to promote the application of technology and process improvements to optimize production costs, enhance resource efficiency, and increase product value. Technical innovation initiatives, along with the optimization of farming areas and raw material zones, help reduce input costs, improve productivity, and enhance economic efficiency for both the Group and its supply chain partners. In the context of increasingly frequent and unpredictable extreme weather events, PAN approaches climate change in a proactive and systematic manner, integrating the mindset of “cl"}\n{"id": "B-A-76b754b327428dfc", "claim": "Focusing on high-value segments and demanding markets enables PAN to effectively manage risks related to market fluctuations, policies, and climate, while also creating long- term growth opportunities aligned with sustainable consumption trends.", "evidence": "Amid ongoing economic fluctuations, PAN remained committed to a development strategy that integrates business growth with social responsibility, recognizing human capital as the foundation for sustainable growth. The Group places strong emphasis on building a safe, stable, and equitable working environment, while also creating opportunities for employees to enhance their capabilities through appropriate training programs. PAN and its subsidiaries fully comply with legal obligations regarding wages, taxes, and insurance; Ï Organizing annual company trips for all employees to The Group regularly monitors key human resource indicators such as employee turnover rate, recruitment rate, and employ"}\n{"id": "B-A-3d84ed0440094365", "claim": "In 2024, the Group launched the “Sustainable Business Assessment and", "evidence": "Discussed and published the results of the PAN Sustainable Business Assessment – PANSI 2023 at PAN Gala 2024. Developed and executed plans for monitoring the 20 2023 2024 2025 PAN Industry Average VN100 Index Average Source: HOSE’s Sustainable Development Assessment Scorecard In addition, PAN Group complies with the IFC Sustainability"}\n{"id": "B-A-26d65b6f647c1690", "claim": "Disclosure” program across its entire ecosystem, while officially adopting the PANSI framework as a tool to assess the level of implementation and effectiveness of sustainability practices at its subsidiaries.", "evidence": "VBCSD as a benchmark for building its internal PANSI framework. PANSI is the Group’s internal sustainability assessment framework, implemented since 2023 to guide and promote the sustainable transformation of its member companies. Ï Human Resources Policy: Commitment to building a professional and fair working environment; SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Solutions to achieve the Net Zero 2050 target Energy efficiency At PAN Group’s factories and production facilities, priority is given to adopting modern technologies and increasing the level of automation to optimize energy use, thereby reducing greenhouse gas emissions across the entire value chain."}\n{"id": "B-A-4d3ff17b4019f29b", "claim": "In 2025, the Group focused on advancing the sustainability target- setting phase through targeted advisory and training programs, aimed at strengthening the capabilities of leadership teams within member companies in identifying, integrating, and managing sustainability goals aligned with their respective business strategies and operational contexts.", "evidence": "In 2025, the Group implemented a program to set grassroots-level sustainability targets for its member companies. We prioritize building awareness of the importance of target setting at the operational level before establishing Group-wide sustainability goals. of contributing to food security while providing The establishment of the Group’s Sustainability Steering Committee demonstrated its leadership role in setting and driving sustainability initiatives. PAN and its subsidiaries have developed a comprehensive sustainability data collection system, laying the groundwork for future ESG goals. Additionally, member companies have begun setting ESG"}\n{"id": "B-A-1d84f47e0af9a6bd", "claim": "Communities Ï Giá trị đầu tư Ï Biodiversity conservation Ï Workforce development support Ï Reforestation: 1 million trees Ï Corporate Social Responsibility by 2030", "evidence": "Resource Regeneration and Conservation - Biodiversity - Tree planting: 1 million trees Community - Support human resource training - CSR - Emergency Support 3,50 Reforestation Number of trees planted; planting area Biodiversity Climate change Protection of aquaculture ecosystems Sustainable aquaculture area"}\n{"id": "B-A-c1502cb0a0926057", "claim": "As a responsible organization, we continuously assess our activities and their impacts, and develop appropriate plans aligned with stakeholders’ expectations and interests— minimizing negative impacts while enhancing positive contributions arising from our operations.", "evidence": "PAN is progressively integrating environmental and sustainability factors into its investment and production activities, aiming to minimize environmental impacts and promote green growth. Through efficient resource utilization, emissions reduction, and participation in sustainable finance initiatives, the Group contributes to generating positive indirect economic impacts aligned with long-term sustainable development goals. We collaborate to We value diverse perspectives and We are committed to creating positive, sustainable impacts We maximize value for all stakeholders 80"}\n{"id": "B-A-36c41595d594dc4b", "claim": "ADB safeguards, and PAN Code of Conduct.", "evidence": "CHANGES IN THE REPORT Ï Updated PAN Group’s Practice Principles and Code Ï Updated the assessment of environmental and social of Conduct. Ï Commitment to Stakeholders: Defining specific The Management Board is fully committed to complying with all applicable legal regulations. The Group will refuse to engage in any business transaction that violates the approved Code of Conduct."}\n{"id": "B-A-76fa4c5da65b767c", "claim": "Safe, professional, and friendly working environment.", "evidence": "Ï Ï Enhancing Organizational Awareness: Professional conduct, corporate culture, and workplace responsibilities are integrated into internal training programs, fostering a professional and collaborative environment. The Group will coordinate production and research among member companies, share experiences, save costs, and optimize operational ef昀椀ciency. Employees - Internal Training - Health, Prosperity - Friendly and safe working environment Management"}\n{"id": "B-A-6528d129a8b8fd6f", "claim": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Assessment of Key Risks Key Risk Categories Impacts Mitigation Measures AGRICULTURE SECTOR", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Key Risk Categories Impacts Mitigation Measures Ï The Group enhances production practices to adapt to adverse environmental SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT PRODUCT SAFETY AND QUALITY Growth of Key Product categories in 2025 Product Growth rate Pangasius fish -2% Cakes"}\n{"id": "B-A-b1dfe85de7d4d79f", "claim": "Product quality, food safety, and community responsibility risks", "evidence": "Product quality, food safety, community responsibility risks C. BRANDING RISKS Responsibilities for customers are taken seriously. Risks can arise from many factors: raw materials, harvesting, preservation, processing and distribution processes; TABLE SKSIR LAICOS DNA LATNEMNORIVNE | Pollution and environmental incident risks | Ï Incidents related to solid waste, wastewater, and emissions. Ï Legal violations. Ï Complaints from local communities. | Labor and employment risks | Ï Shortage of production labor. Ï Conflicts and grievances from employees. | Health and safety risks | Ï Shortage of production labor. Ï Conflicts and grievances from employees. | Product quality, food safety, and commu"}\n{"id": "B-A-6277e586cf3af6f9", "claim": "Accordingly, the Group’s approach is based on a double materiality principle, including: The initial list is compiled based on: the operational characteristics of PAN’s ecosystem in the agriculture, aquaculture, and food sectors;", "evidence": "(ii) The Sustainability Steering Committee, which plays a leading role in overseeing and ensuring effective management of sustainability across the Group. Its members include the Chief Executive Officer and three representatives from the Group’s key business sectors: Agriculture, Aquaculture, and Food Processing; AND ORGANIZATIONAL STRUCTURE WITH GRI STANDARDS The overarching operating philosophy of PAN Group— from business strategy to product development—is to build an agriculture–food ecosystem based on a closed-"}\n{"id": "B-A-862d244a283d73c9", "claim": "The results of this assessment will shape PAN Group’s long-term sustainable development strategy.", "evidence": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 TOWARDS THE FUTURE TOWARDS THE FUTURE STRATEGIC DIRECTION PAN Group is committed to sustainable development with a long-term strategy. ANNUAL REPORT 2024 CHAPTER 2 2024 BUSINESS RESULTS ENVIRONMENTAL & SOCIAL RESPONSIBILITY REPORT The PAN Group is committed to sustainable development, pursuing strategies that deliver long-term value for both the business and the community."}\n{"id": "B-A-b3df567ceb68cf24", "claim": "profit Customer and consumer satisfaction Number of food safety violations", "evidence": "TABLE Customer and consumer satisfaction Traceability Research and Development Product diversification No. Content Kết quả 1 Number of violations related to food safety and hygiene. 0 2 Number of product recalls due to quality non-compliance. 0 3 Total number of incidents of non-compliance with regulations and voluntary codes related to product and service information and labeling."}\n{"id": "B-A-7ac769b9cf1f1dc6", "claim": "High-Quality and Safe Products Economic performance Innovation in Business Operations Number of initiatives per year;", "evidence": "High-Quality and Safe Products Wastewater and waste control Wastewater treatment compliance rate; number of environmental incidents Economic performance Regenerative agriculture TABLE Sustainable economic growth Innovation in Business Operations Improvement Quality of life Creation Employment"}\n{"id": "B-A-7286ff597b95d4b8", "claim": "Materiality Topic Greenhouse gas emissions Total CO2e emissions;", "evidence": "16,570,365 26,294,114 23,810,747 Reduction Rate Compared to 2023 % -2% In 2025, coal data from Vinaseed units will be added GREENHOUSE GAS EMISSIONS Scope 1 CO2 Emissions kgCO2e 06/2022/NĐ-CP dated January 7, 2022, on greenhouse gas emission mitigation and ozone layer protection. Among them, Khang An Foods – a member company of PAN Group – is also subject to regulations on greenhouse gas emission reduction, GHG inventory, and disclosure of environmental and social information."}\n{"id": "B-A-740d631843100a97", "claim": "water intensity Water conservation Responsible usage Volume of water recycled, reused, and circulated", "evidence": "Resilience”, the Group is promoting responsible water management throughout its operations and value chain. Key efforts include: M3/PER VND BILLION Ï Reducing water consumption and improving water OF WATER CONSUMPTION INTENSITY (PER UNIT OF REVENUE) use efficiency. Natural Resources Ï 2.3% reduction in CO2 emission intensity Ï Energy consumption (tons CO2 per VND billion revenue) Ï Water usage Ï 6% reduction in water intensity (1,000 m3"}\n{"id": "B-A-9f033b1ca6e1539f", "claim": "High-Quality and Safe Products Wastewater and waste control Wastewater treatment compliance rate;", "evidence": "TABLE Greenhouse gas emissions Green energy Water resources and quality Responsible usage Sustainable raw material Sustainable waste management Wastewater and waste control Reforestation Protection of aquaculture ecosystems Responsible farming Soil regeneration Companies and factories within the PAN Group have established close partnerships with reputable waste collection and treatment service providers. We conduct thorough evaluations and screenings to select units with the necessary capacity and experience, responsible for the safe and efficient collection, transportation, and treatment of municipal waste, industrial waste, and hazardous waste. The operations of these partners are regularl"}\n{"id": "B-A-0246aa2b0ca27a50", "claim": "3,50 Reforestation Number of trees planted;", "evidence": "Communities Ï Giá trị đầu tư Ï Biodiversity conservation Ï Workforce development support Ï Reforestation: 1 million trees Ï Corporate Social Responsibility by 2030 Ï Plant one million trees by 2030. complaints or disputes."}\n{"id": "B-A-f6f90866e0938ef2", "claim": "planting area Biodiversity Climate change Protection of aquaculture ecosystems Sustainable aquaculture area", "evidence": "TABLE Greenhouse gas emissions Green energy Water resources and quality Responsible usage Sustainable raw material Sustainable waste management Wastewater and waste control Reforestation Protection of aquaculture ecosystems Responsible farming Soil regeneration Biodiversity Risk Assessment For each aquaculture pond construction project (shrimp, pangasius), we conduct surveys of geology, geography, and baseline environmental conditions as a basis for assessing the level of impact of production activities on surrounding communities. Environmental Protection Requirements Conservation and Restoration of Biodiversity in"}\n{"id": "B-A-71a79ef751ba34fd", "claim": "3,00 Responsible farming Sustainable farming and aquaculture area Farmers Regenerative agriculture Water conservation Product R&D Workers Circular economy Soil regeneration", "evidence": "TABLE Greenhouse gas emissions Green energy Water resources and quality Responsible usage Sustainable raw material Sustainable waste management Wastewater and waste control Reforestation Protection of aquaculture ecosystems Responsible farming Soil regeneration SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT REGENERATIVE AGRICULTURE Aquaculture > Shrimp farming success rate GRI 304 PAN Group’s vision is to scale regenerative agriculture across its entire value chains."}\n{"id": "B-A-a6fd0e6cea361cfe", "claim": "Biodiversity Working conditions Number of incidents and accidents;", "evidence": "TABLE Number of Serious Accidents Involving Staff and Workers | Case | 0 | 0 | 0 Number of Injuries Requiring Medical Leave | Case | 10 | 0 | 1 Total Lost Days Due to Accidents | Days | 143 | 0 | 46 10 0 1 groups such as working at heights, working in high- temperature environments, operating mechanical equipment, and handling chemicals. Total number of days lost due to occupational accidents"}\n{"id": "B-A-0c5a46cc9efbe69d", "claim": "Number of farmers accessing sustainable farming technical support services Environmental Product Responsibility Social Economic Community development Number of beneficiaries;", "evidence": "amount of financial support Community Emergency support Number of beneficiaries; amount of financial support 80 81 CREATIVITY EFFICIENCY THINK DIFFERENTLY PURSUE EXCELLENCE To community & social responsibility: PAN employees actively support sustainability, balancing economic growth, environmental protection, and social welfare."}\n{"id": "B-A-20262b7549e18d39", "claim": "These are international standards and represent the latest version for sustainability reporting.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT BIODIVERSITY Environmental Management of Farming Areas in Accordance with International Standards GRI 304 Ï 01 pangasius farm has achieved the Aquaculture SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY STRATEGY We are committed to minimizing negative environmental impacts and continuously improving products and technologies toward greater environmental friendliness. The Group has established environmental management systems in line with international standards such as ISO"}\n{"id": "B-A-0096e88b269b8947", "claim": "Nguyen Thi Tra My – Chief Executive Officer This report is an essential component of the performance monitoring and reporting system, aligned with the IFC Performance Standards and the PAN Sustainability", "evidence": "TABLE REPORT PREPARATION TEAM | Ï Mr. Nguyen Van Si – Head of Sustainability. Ï Ms. Luong Nguyen Khanh Linh – Project Specialist. With support from Sustainability Coordinators across PAN Group’s member companies: Ï Report Reviewer: Mr. Nguyen Hong Hiep – Director of External Affairs and Communications. Ï Report Approver: Ms. Nguyen Thi Tra My – Chief Executive Officer and Head of the Sustainability Steering Committee of PAN Group. Members of the Steering Committee bring deep expertise in their respective fields, forming a strong foundation for managing the Group’s sustainability initiatives. Ï Ms. Nguyen Thi Tra My – Chairwoman, Ms. Tra My is a co-founder and Chief Executive Officer of PAN"}\n{"id": "B-A-73167dfa2a123dab", "claim": "Index (PANSI), which is based on the Corporate Sustainability Index (CSI) developed by the Vietnam Business Council for Sustainable Development (VBCSD", "evidence": "an organization representing Vietnam in the global sustainable development network of the World Business Council for Sustainable Development. Participation in VBCSD demonstrates PAN’s consistent orientation toward adopting sustainable development practices in line with international standards, while strengthening the alignment between its corporate strategy and Vietnam’s sustainable development priorities. MS. NGUYEN THI TRA MY COUNCIL FOR SUSTAINABLE DEVELOPMENT CEO OF THE PAN GROUP (VBCSD) 36 37"}\n{"id": "B-A-5573d9e0e44c0c11", "claim": "Its publication ensures consistency and provides stakeholders— including investors, customers, and partners—with a clear view of PAN Group’s sustainability commitment.", "evidence": "PAN Group focuses on the agriculture and food sectors—Vietnam’s core competitive advantages—with a strategy centered on integration and value chain enhancement. We create and share economic value in a balanced manner with stakeholders, thereby fostering motivation for broader social development. ECONOMIC Trusted by Investors and Stakeholders Group’s enduring growth. To colleagues, we promote a workplace of equality and respect, embracing personal freedoms and 3 4 SOLIDARITY DEDICATION SERVE THE WORLD WORK TOGETHER We are committed to making a lasting, positive impact on our employees, customers, stakeholders, partners, communities, and the environment."}\n{"id": "B-A-b46cf1443a875063", "claim": "For all inquiries related to this report, please contact: Sustainability Committee, The PAN Group Joint Stock Company Ha Noi Office: 15th Floor, ICON4 Tower, 243A De La Thanh Street, Lang", "evidence": "TABLE REPORT CONTACT INFORMATION | For all inquiries related to this report, please contact: Sustainability Committee, The PAN Group Joint Stock Company Ha Noi Office: 15th Floor, ICON4 Tower, 243A De La Thanh Street, Lang Ward, Hanoi Phone: (84 - 024) 3760 6190 Email: Sustainability@thepangroup.vn Website: www.lafooco.vn HANOI BRANCH PAN-HULIC JOINT STOCK COMPANY Address: 15th 昀氀oor, ICON4 Tower, 243A De La Thanh Street, Lang Thuong Ward, Dong Da District, Hanoi."}\n{"id": "B-A-60fb3ceb847a612a", "claim": "Nguyen Thi Tra My – Chief Executive Officer and Head of the Sustainability Steering Committee of PAN Group.", "evidence": "Members of the Steering Committee bring deep expertise in their respective fields, forming a strong foundation for managing the Group’s sustainability initiatives. Ï Ms. Nguyen Thi Tra My – Chairwoman, Ms. Tra My is a co-founder and Chief Executive Officer of PAN Members of the Committee possess extensive experience across a wide range of products, value chains, and geographic markets. Sustainability Steering Committee Ï Ms. Nguyen Thi Tra My – Chairwoman, Ms. Tra My is a co-founder and Chief Executive Officer of PAN Group, and Chairwoman of the Board of Directors of Vietnam"}\n{"id": "B-A-a3a36ae38ee337a6", "claim": "The remaining shrimp supply is sourced from farming households under strict evaluation and control criteria aligned with sustainable aquaculture practices, ensuring a stable and sustainable input supply.", "evidence": "F um Ï 6 suppliers and 170 aquaculture Farm – Food – Family. Ï More than 60% of shrimp supplies I oc households ns come from local households O M es co Ï 50,453 farming households and 220 At the same time, PAN implements training programs and works closely with agents and farming households to enhance sustainable aquaculture practices, ensure product quality, and strengthen transparency across the supply chain. In addition to evaluation activities, PAN emphasizes accompanying and enhancing the capacity of partners within the value chain, aiming to build a responsible, transparent, and resilient supply ecosystem with long-term adaptability. A Q U A C U L E T R U U R T E L U AGRICULTURE"}\n{"id": "B-A-0427ce3a440a920b", "claim": "PAN’s member companies, such as Fimex VN and Aquatex Bentre, actively participate in certification programs, research initiatives, and sustainable aquaculture practices, including:", "evidence": "Ï 02 shrimp farms have achieved the Best Aquaculture Practices (BAP) certification (Vita Farm, Hono Farm – Fimex VN). We have fully implemented responsible aquaculture practices, including stringent requirements on environmental protection, social responsibility, animal welfare, and food safety. The PAN Group focuses on building a common corporate culture with shared values, close connections between member companies, while respecting the differences and traditions of each company. between the parent company and member companies through environmental activities, sustainable development and events such as the CEO Summit. These programs help leaders and employees of member companies share expe"}\n{"id": "B-A-78993bc21d174741", "claim": "Seed products that do not meet quality standards are repurposed for food use, with no waste generated.", "evidence": "Treated wastewater must meet strict quality standards, ensuring no pollution of water sources, protecting ecosystems, and safeguarding public health. We commit to transparency in disclosing wastewater quality information and are ready to cooperate with regulatory authorities for monitoring and evaluation. OF FACTORIES AND PROCESSING FACILITIES have wastewater treatment systems that meet standards and comply with the issued Environmental Permits At the same time, the Group’s aquaculture farming sites must comply with consumer safety standards while taking animal welfare into consideration. Processing facilities are required to use high-quality input materials and operate production processes "}\n{"id": "B-A-5a164e3a95bc35cc", "claim": "In 2025, Vinaseed launched the “Carbon Footprint", "evidence": "Vinaseed strengthened international partnerships and launched several new high-quality crop varieties to combat climate change. 21 20 THE PAN GROUP Ï At Vinaseed, in 2025, the program “Promoting Innovation POLICY FOUNDATION FOR PROMOTING A CULTURE OF INNOVATION Ï At 584 Nha Trang, in 2025 the company developed and"}\n{"id": "B-A-5aa1f7e21b6cca8b", "claim": "During processing, the facilities maximize the reuse and recycling of discarded packaging for purposes such as packing raw materials, agricultural by-products, or selling them for recycling.", "evidence": "Ï Encouraging the reuse, recycling, and/or upcycling of 0.2 0.0 0 by-products and waste materials. (straw, rice stalks, etc.) Tons 85,964 Fertilizer production, soil nutrient restoration Recyclable discarded packaging Tons 1,242 Repackaging of raw materials Waste sludge"}\n{"id": "B-A-ae6141d1d04deb60", "claim": "PP BAGS Packaged Food equivalent to approximately 4 tons of packaging The main raw materials for packaged products such as green coffee beans and cashew kernels are developed by PAN Group through linkage models with cooperatives and local farmers, combined with training in sustainable farming practices to ensure quality, supply stability, and traceability.", "evidence": "Tracking” program to collect used seed packaging in the Mekong Delta region. Under this initiative, participating farmers and distributors receive VND 8,000 for each returned used package. From June to December 2025, the program collected 20,000 PP bags, equivalent to approximately 4 tons of packaging. The Group focuses on developing a transparent, stable, and sustainable supply chain by strengthening collaboration with farmers, cooperatives, and local partners. Production linkage programs, offtake agreements, and technical support initiatives continue to be implemented, contributing to improved raw material quality, stable market access, and increased income for producers within the agricul"}\n{"id": "B-A-46b36ca45d087a8a", "claim": "These solutions help optimize resource use and reduce waste generation.", "evidence": "- Waste management, reuse STRENGTHENING COOPERATION IN PRODUCTION AND BUSINESS BETWEEN The Group will also share 昀椀nancial resources among member companies to optimize capital utilization and minimize 昀椀nancial costs. Cross-selling and distribution of products between member companies will help expand the market and optimize sales costs. food value chain. Ï Waste management and recycling Resource regeneration and conservation Ï Income improvement Ï Số lượng sáng kiến/năm Ï High-quality and efficient solutions"}\n{"id": "B-A-93210a6841cdc92a", "claim": "For confectionery manufacturing plants, key raw materials—including wheat flour, sugar, and eggs—are sourced from domestic suppliers and comply with food safety standards such as ISO 22000.", "evidence": "Ï It applies stringent food safety management standards, including ISO, FSSC, BRCGS, Ï Substandard raw materials leading to food safety incidents. Product quality, food safety, and community responsibility risks RAW MATERIAL TRACEABILITY MANAGEMENT PROCESS Supervise and evaluate the performance of farming ponds and suppliers Screen and assess raw materials to ensure compliance with standards"}\n{"id": "B-A-5f304a64584f86e0", "claim": "PAN promotes a circular economy model by utilizing by- products generated during production.", "evidence": "KG EPR Opens the Future for Green Recycling and Promotes a Circular Economy Project “Cosmetic Product Line (Soap, Scrub) from coffee by-products” SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT CIRCULAR ECONOMY GRI 306 Promoting a circular economy by designing products for reuse, reducing resource consumption, and repurposing waste are key approaches to minimizing environmental impact, lowering greenhouse gas emissions, and ensuring long-term sustainability."}\n{"id": "B-A-2cdaad67dd9a2b0d", "claim": "Approximately 10% of shells are reused or recycled;", "evidence": "cashew shells and walnut shells are used as biomass fuel for boilers, while shells and coffee grounds are recycled into value-added products. These solutions help optimize resource use and reduce waste generation. For confectionery manufacturing plants, key raw materials—including wheat flour, sugar, and eggs—are sourced from domestic suppliers and comply with food safety standards such as ISO 22000. Ï Reuse 100% of treated wastewater Lafooco: Replace 10% of non- recyclable packaging 65 64"}\n{"id": "B-A-6e3268a9c74b34f3", "claim": "Sugar Tons 4,205 4,367 6,236 and optimization of manufacturing processes to minimize losses and reduce intermediate stages that generate waste.", "evidence": "TABLE Tons | 3,976 | 3,398 Tons | 4,205 | 4,367 Tons | 808 | 938 Tons | 2,968 | 2,462 Tons | 35 | 25 Tons | 163 | 295 Tons | 2,043 | 2,197 Tons | | 6,353 Solutions to minimize the risks: Market risks and competition pressure PAN consistently manages the quality of its production processes according to international standards of sustainable development like ASC, GLOBAL G.A.P., and MSC. Over the past few years, a series of policies on quality management and manufacturing improvements to increase ef昀椀ciency, improve quality, and lower cost has been applied."}\n{"id": "B-A-b671826384d210be", "claim": "The Group requires that all supplied inputs originate from sources not associated with biodiversity violations.", "evidence": "For PAN Group, where business activities are closely associated with land, water resources, biodiversity, and the livelihoods of millions of farmers, climate resilience and adaptation capacity are not only a requirement of risk management, but a prerequisite to ensure sustainable growth, food security, and long-term prosperity for future generations. Ï Internal Code of Conduct: Providing detailed guidance All employees are required to comply with the Code of Conduct and immediately report any activities or behaviors that may constitute serious violations."}\n{"id": "B-A-a4838f25c96e9d5f", "claim": "At the same time, the Group’s aquaculture farming sites must comply with consumer safety standards while taking animal welfare into consideration.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Animal Welfare Product Responsibility Indicators 2025 Increasingly stringent animal welfare requirements are driving changes across farming, harvesting, and processing practices. At the same time, they serve as a key foundation in the Group’s strategy to attract and retain talent, contributing to a stable workforce and strengthening the Group’s long- term competitiveness. Employee welfare activities include:"}\n{"id": "B-A-df7ec75827f6403f", "claim": "For PAN Group, where business activities are closely associated with land, water resources, biodiversity, and the livelihoods of millions of farmers, climate resilience and adaptation capacity are not only a requirement of risk management, but a prerequisite to ensure sustainable growth, food security, and long-term prosperity for future generations.", "evidence": "PAN aims to build stable livelihoods, improve quality of life, and strengthen long-term stakeholder trust—key factors for the continuity and resilience of its business model. PAN Group develops a closed-loop value chain model—Farm – Food – Family—as a core component of its business model and long-term sustainable development strategy, aiming to ensure the creation of sustainable value for the “climate change adaptation and nature-based orientation” into its strategy, operations, and investment decisions. Climate change adaptation is no longer a long-term or abstract risk, but has become a factor reshaping the entire value chain of businesses today."}\n{"id": "B-A-72f5ead1af52330c", "claim": "During the 2020–2025 period, PAN has achieved significant internal transformations, gradually building a foundation for its long-term emissions reduction pathway.", "evidence": "VBCSD as a benchmark for building its internal PANSI framework. PANSI is the Group’s internal sustainability assessment framework, implemented since 2023 to guide and promote the sustainable transformation of its member companies. Ï Human Resources Policy: Commitment to building a professional and fair working environment; During the Winter– Spring crop 2024–2025, rice cultivated by 5,473 farmers within Vinaseed’s value chain achieved lower emissions compared to conventional practices. In terms of environmental performance, total emission reductions reached 78,485.62 tons of CO2 equivalent, as independently verified by Regrow Ag through the MRV system platform. In terms of economic performan"}\n{"id": "B-A-87e723a33c38e532", "claim": "Accordingly, the Group has been deploying a range of synchronized solutions, from adopting sustainable agricultural practices and reducing dependence on fossil fuels in production, to investing in renewable energy and improving energy efficiency across the entire value chain.", "evidence": "To realize this goal, the Group has implemented a range of comprehensive solutions, including sustainable agricultural practices, reducing fossil fuel consumption in production, investing in renewable energy, and enhancing energy ef昀椀ciency. In 2024, the Group eliminated 100% of coal use in confectionery production facilities and rolled out a series of energy ef昀椀ciency initiatives to optimize production. 91 THE PAN GROUP 90 SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Solutions to achieve the Net Zero 2050 target Energy efficiency At PAN Group’s factories and production facilities, priority is given to adopting modern technologies and increasing the level of automation to op"}\n{"id": "B-A-bbebe7e1d9e596f2", "claim": "Scope 2 Indirect emissions Electricity consumption from the national grid and the purchase of steam from external suppliers.", "evidence": "Not Applicable 305-2 Energy indirect (Scope 2) GHG emissions 94 - 96 GRI 300: ENVIRONMENTAL STANDARDS 305-3 Other indirect (Scope 3) GHG emissions SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT CLIMATE CHANGE The Group’s CO2 emission sources mainly arise from the following activities: GRI 302, 305 Scope 1 Direct emissions Fuel consumption for boilers and dryers;"}\n{"id": "B-A-68a7aa812e38c7ea", "claim": "The significant decrease in Scope 1 CO2 emissions in 2024 was mainly due to the complete elimination of 100% coal usage in Bibica’s confectionery manufacturing plants.", "evidence": "REDUCTION IN RIVER WATER USAGE COMPARED TO 2024 Groundwater usage increased by 27%, mainly due to increased production at shrimp processing plants. Shrimp production in 2025 is projected to increase by 16,570,365 26,294,114 23,810,747 Reduction Rate Compared to 2023 % -2% In 2025, coal data from Vinaseed units will be added GREENHOUSE GAS EMISSIONS Scope 1 CO2 Emissions kgCO2e"}\n{"id": "B-A-24abe0f98166b276", "claim": "In 2025, the Group’s total CO2 emissions increased by 6.1%, primarily driven by the expansion of production scale at two factories of Lafooco and Bibica, as well as the stabilization of operations at Fimex VN’s shrimp farms.", "evidence": "Bibica Western Confectionery Plant, the Quang Nam Seed Drying, Processing and Storage Plant, and the Lafooco Long An Factory, while aquaculture operations have been maintained steadily with increased output, the Group’s total energy consumption in 2025 increased by 6.5% compared to 2024. REDUCTION IN RIVER WATER USAGE COMPARED TO 2024 Groundwater usage increased by 27%, mainly due to increased production at shrimp processing plants. Shrimp production in 2025 is projected to increase by"}\n{"id": "B-A-787765f5b5d2bab8", "claim": "CO2 Emission Intensity (ton CO2e/ Revenue (VND bn) Increased Emission factors applied: - Emission factors for gasoline, diesel, and gas are based on: (1) Decision No.", "evidence": "Natural Resources Ï 2.3% reduction in CO2 emission intensity Ï Energy consumption (tons CO2 per VND billion revenue) Ï Water usage Ï 6% reduction in water intensity (1,000 m3 (M3/VND BILLION REVENUE) HECTARES MWP CO2 EMISSION INTENSITY WATER USAGE INTENSITY ROOFTOP SOLAR POWER CAPACITY (IN- ACROSS THE GROUP OF HIGH-QUALITY, LOW-EMISSION"}\n{"id": "B-A-1cda94273b10551a", "claim": "2626/2022-BTNMT, which promulgates the list of emission factors for greenhouse gas inventories, including four categories covering energy;", "evidence": "06/2022/NĐ-CP dated January 7, 2022, on greenhouse gas emission mitigation and ozone layer protection. Among them, Khang An Foods – a member company of PAN Group – is also subject to regulations on greenhouse gas emission reduction, GHG inventory, and disclosure of environmental and social information. Materiality Topic Greenhouse gas emissions Total CO2e emissions; CO2e emission intensity Climate change Green energy Percentage of renewable electricity;"}\n{"id": "B-A-1df71d8b22346cd1", "claim": "(2) The IPCC Emission Factor Database (EFDB) available at: https://www.ipcc-nggip.iges.", "evidence": "CO2 Emission Intensity (ton CO2e/ Revenue (VND bn) Increased Emission factors applied: - Emission factors for gasoline, diesel, and gas are based on: (1) Decision No. 2626/2022-BTNMT, which promulgates the list of emission factors for greenhouse gas inventories, including four categories covering energy; Website: https://khanganfoods.com Address: 584 Le Hong Phong Street, Phuoc Long Ward, Nha Trang City, Khanh Hoa Province. Tel: 0258 3881176 – 3882677 Fax: (84-258) 3884 442"}\n{"id": "B-A-8deefec0b61e9649", "claim": "- The grid electricity emission factor of 0.6592 tCO2/MWh, as pub-", "evidence": "Materiality Topic Greenhouse gas emissions Total CO2e emissions; CO2e emission intensity Climate change Green energy Percentage of renewable electricity; In August 2025, Khang An Foods – a member company of PAN Group – completed the installation and commissioning of its first rooftop solar power system with a capacity of 1.38 MWp at the Tam An facility, helping to reduce grid electricity consumption by 10% at the plant."}\n{"id": "B-A-b32a10f3311df460", "claim": "MWP rooftop solar power capacity, bringing the total installed capacity to 5.53 MWp.", "evidence": "As of the end of 2025, PAN Group’s total installed solar power capacity reached 5.53 MWp, with 100% coming from rooftop solar systems. The total renewable electricity generated and consumed across the Group in 2025 was approximately 4,393 MWh. Ï Investment in modern equipment and productivity improvement (M3/VND BILLION REVENUE) HECTARES MWP CO2 EMISSION INTENSITY WATER USAGE INTENSITY ROOFTOP SOLAR POWER CAPACITY (IN- ACROSS THE GROUP OF HIGH-QUALITY, LOW-EMISSION"}\n{"id": "B-A-ec5ff45b665f2bfa", "claim": "Meanwhile, companies in the agriculture segment generate lower CO2 emissions, as their operations mainly involve trading and product distribution.", "evidence": "The significant decrease in Scope 1 CO2 emissions in 2024 was mainly due to the complete elimination of 100% coal usage in Bibica’s confectionery manufacturing plants. In 2025, the Group’s total CO2 emissions increased by 6.1%, primarily driven by the expansion of production scale at two factories of Lafooco and Bibica, as well as the stabilization of operations at Fimex VN’s shrimp farms. In the same year, we also incorporated comprehensive energy consumption data from Vinaseed’s affiliated units. 16.0 538,575 200,000 11.0 398,330 69.3% 80.4% 76.4% 100,000 6.0 Agriculture Aquaculture Packaged Foods - 1.0 Our seafood segment companies account for a larger share of CO2 emissions, primarily du"}\n{"id": "B-A-145c1e609d458e1f", "claim": "Within this system, key information such as facility details, production volumes, energy consumption data, water use and wastewater data, occupational health and safety data, human resources data, and community engagement data is required to be collected.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-c992bb02fa69d5bf", "claim": "These inputs enable the calculation of carbon emissions and water consumption, as well as the reporting of environmental and social performance over a defined period.", "evidence": "targets for the 2025–2030 period. Water resource management: The PAN Group has several water-intensive businesses, particularly seafood processing accounting for over 90% of PAN’s total annual water consumption. Recognizing the importance of water resources, the Group has adopted sustainable water management solutions, including optimizing water use in production, recycling treated wastewater, and raising employee awareness about water conservation. As a result, total water consumption water consumption intensity (1,000m3/billion VND revenue) high-quality rice cultivation SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT WATER AND WASTEWATER MANAGEMENT Water consumption during the"}\n{"id": "B-A-dfca90465090e94f", "claim": "*Currently, PAN Group has only one entity, Khang An Foods, that is required to conduct greenhouse gas inventory under Decision No.", "evidence": "06/2022/NĐ-CP dated January 7, 2022, on greenhouse gas emission mitigation and ozone layer protection. Among them, Khang An Foods – a member company of PAN Group – is also subject to regulations on greenhouse gas emission reduction, GHG inventory, and disclosure of environmental and social information. have set sustainable development targets for has prepared a greenhouse gas inventory report in accordance with Government Decree No. 06/2022/NĐ-CP on greenhouse gas emission reduction and ozone have established monthly data sustainability goals and annual action plans. the 2025–2030 period. collection systems."}\n{"id": "B-A-fcc1043f0147361a", "claim": "LEAN methodologies, helping to minimize waste throughout the manufacturing process.", "evidence": "We are committed to building a culture of learning and development through continuous training activities based on a structured training framework that includes goal setting, needs analysis, current-state evaluation, modern and technology-integrated methodologies, strategic planning, and impact measurement for re昀椀nement and enhancement. 2024 Age structure 7000 6579 Practical training activities throughout the year: Aquaculture Packaged food Agriculture 6000 Sugar Tons 4,205 4,367 6,236 and optimization of manufacturing processes to minimize losses and reduce intermediate stages that generate waste. Eggs Tons 808 938 1,326 Raw cashew nuts Tons 2,968 2,462 2,978 Ï Continuous improvement of pr"}\n{"id": "B-A-f82f2577ecc4347c", "claim": "The transition to green energy is an inevitable trend and a mandatory requirement to meet international standards, while also serving as a key objective in PAN Group’s pursuit of sustainable development and green growth, and in realizing its Net Zero emissions target by 2050.", "evidence": "We position ourselves as an active and proactive player in the green transition across all the countries and territories where we operate. Sustainable development and the green economy are increasingly becoming mandatory standards in international markets, while the rise of trade protectionism and technical barriers is placing higher demands on exporting enterprises. Although these requirements present challenges, they also open up significant opportunities for Vietnamese businesses. Regulations; along with a series of operational guidelines, covering areas such as accounting, human resources, and administration. In 2024, PAN Group reaf昀椀rmed its long-term commitment to sustainable developme"}\n{"id": "B-A-99be12847b81cf5c", "claim": "As of the end of 2025, PAN Group’s total installed solar power capacity reached 5.53 MWp, with 100% coming from rooftop solar systems.", "evidence": "MWP rooftop solar power capacity, bringing the total installed capacity to 5.53 MWp. 94 95 PAN Group’s total installed solar power capacity reached The total renewable electricity generated and consumed across the Group in 2025 was approximately Ï Smart and environmentally friendly logistics"}\n{"id": "B-A-2cf925454d7cb3d4", "claim": "In August 2025, Khang An Foods – a member company of PAN Group – completed the installation and commissioning of its first rooftop solar power system with a capacity of 1.38 MWp at the Tam An facility, helping to reduce grid electricity consumption by 10% at the plant.", "evidence": "Rooftop solar power system at Khang An Foods 102 103 MWP rooftop solar power capacity, bringing the total installed capacity to 5.53 MWp. 94 95"}\n{"id": "B-A-50291eb8de216bc4", "claim": "The use of biomass-based steam (rice husk) also helps the plant cut CO2 emissions by", "evidence": "The significant decrease in Scope 1 CO2 emissions in 2024 was mainly due to the complete elimination of 100% coal usage in Bibica’s confectionery manufacturing plants. In 2025, the Group’s total CO2 emissions increased by 6.1%, primarily driven by the expansion of production scale at two factories of Lafooco and Bibica, as well as the stabilization of operations at Fimex VN’s shrimp farms. In the same year, we also incorporated comprehensive energy consumption data from Vinaseed’s affiliated units. Accordingly, the Group has been deploying a range of synchronized solutions, from adopting sustainable agricultural practices and reducing dependence on fossil fuels in production, to investing in"}\n{"id": "B-A-1bbc314537f7e516", "claim": "Emission management 6 out of 11 Vinaseed seed and rice processing plants have converted their drying systems to utilize biomass—such as rice husks, corn cobs, and compressed husk fuel—as substitutes for coal in boiler operations.", "evidence": "In addition, these systems are regularly inspected, maintained, and upgraded to ensure stable and efficient operation. The total biomass consumption (rice husks and corn cobs) at Vinaseed was approximately TONS Emission monitoring results at selected factories (mg/Nm3) 7,881,985 6,614,893 6,735,787 Coal Ton 5,673 148 245 Reused Water m3 NA 1,516 18,032 Biomass – Rice husks and seed shells Ton 1,231 5,643 3,601 Treated Wastewater m3"}\n{"id": "B-A-c90eb795169de8a8", "claim": "To achieve this, the Group has made strong investments in modern emission treatment systems, applying advanced technologies such as electrostatic precipitation, chemical absorption, and catalytic treatment.", "evidence": "Ï For seafood processing plants, wastewater treatment technology applies anaerobic biological processes combined with phosphorus and chlorine treatment, as well as sludge treatment. Bibica has proactively embraced advanced technologies by strongly applying artificial intelligence (AI) and Industry 4.0 solutions in both business operations and internal management. From advertising and data analytics to process automation, technology is becoming a key driver for accelerating growth and enhancing the company’s competitiveness."}\n{"id": "B-A-f760148f8f5a5bd2", "claim": "The total biomass consumption (rice husks and corn cobs) at Vinaseed was approximately TONS Emission monitoring results at selected factories (mg/Nm3)", "evidence": "60–70% compared to using diesel fuel. Emission management 6 out of 11 Vinaseed seed and rice processing plants have converted their drying systems to utilize biomass—such as rice husks, corn cobs, and compressed husk fuel—as substitutes for coal in boiler operations. In 2025, the total biomass consumption (rice husks and corn cobs) at Vinaseed was approximately 2,617 tons. PAN Group always places environmental protection as a top priority. We are committed to ensuring that all emission treatment systems at production facilities operate effectively and strictly comply with environmental standards and regulations. To achieve this, the Group has made strong investments in modern emission treatm"}\n{"id": "B-A-ffe6a2bccb8367ce", "claim": "The Group aims to minimize water consumption, promote water recycling, and optimize efficiency across all processes to address the challenges of global warming and the risk of future water scarcity.", "evidence": "Ï Treating wastewater to the highest standards to minimize impacts on receiving environments. Ï Recycling and reusing water in production processes and after wastewater treatment. Ï Collaborating with local stakeholders (farmers, From 2025 onwards, the Group plans to enhance its water-related initiatives by developing innovations and new technologies to improve water use efficiency, while expanding the scope of water risk assessments across the value chain. In this regard, we strive to ensure that the Group’s water management activities contribute to creating shared value for both society and the environment. Groundwater m3 510,518 461,999 586,504 River water m3"}\n{"id": "B-A-5a56ee5f68e87562", "claim": "Bacillus subtilis microorganisms to treat the environment and recycle water in the ponds, thereby reducing river water usage by more than 84%.", "evidence": "In 2025, saline intrusion occurred continuously in the Mekong Delta region. Therefore, in its pangasius farming operations, the subsidiary Aquatex Bentre applied improvements to its farming processes by using self- developed and self-produced Bacillus subtilis probiotics for environmental treatment and water recirculation in ponds, thereby reducing river water usage by more than 84%. The Bacillus subtilis probiotic production system at Aquatex Bentre 124 125"}\n{"id": "B-A-23d95ddfed620c9c", "claim": "cooperatives, authorities, etc.) to sustainably manage river basins and marine resources, thereby enhancing water resilience in natural ecosystems.", "evidence": "Unlike traditional agriculture models that tend to deplete natural resources, regenerative agriculture focuses on restoring and regenerating soil, biodiversity, and natural ecosystems. VinaRice, a subsidiary of Vinaseed, participated in Phase 3 (Winter–Spring crop 2024–2025) of the project quality seed and enhances resilience to environmental and water resource fluctuations. Organic Farming Areas for Cashew and Coffee To achieve regenerative circularity and a sustainable balance among natural environmental factors, PAN’s subsidiaries, Lafooco and SHIN Coffee, have consistently complied with standards and strived to deliver verified and certified results as evidence of their sustainability co"}\n{"id": "B-A-b2e8269ae2f3f1d3", "claim": "The wastewater treatment process at the aquaculture ponds follows the Environmental Permit approved and evaluated by the state management authorities.", "evidence": "Ï Water quality management: Ensure that wastewater from aquaculture ponds, after treatment, meets regulatory standards before being discharged into the environment and does not cause pollution. Ï Soil environmental protection: All shrimp ponds are of aquaculture ponds prepare Environmental Impact Assessment reports on biodiversity and on plant and animal species that need to be protected High-Quality and Safe Products Wastewater and waste control Wastewater treatment compliance rate; number of environmental incidents Economic performance Regenerative agriculture"}\n{"id": "B-A-af26105abc28d1b5", "claim": "The basic process is as follows: POND WASTEWATER M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-76fef4c7dbc04d73", "claim": "The wastewater after the RO filtration system is considered clean and reusable.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-d02bd05f20addbba", "claim": "This solution helps reduce freshwater intake while also lowering wastewater treatment costs, saving input water approximately", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-1f034f9ad99124ae", "claim": "Council), with plans to expand to 43 hectares by 2026 DISCHARGE CANAL (wastewater is treated to meet QCVN 40:2011/ At Lafooco’s air conditioner manufacturing facility, the", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-570ea6753a7243f1", "claim": "RO water filtration system used in the production process and for biological protection also utilizes the wastewater from the system for toilet and floor cleaning.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-722756eb4256730b", "claim": "Additionally, Vinaseed implemented over 47,520 hectares of high-quality, low-emission rice cultivation in the Mekong Delta region.", "evidence": "Ï At COP28, signed a cooperation on the initiative “The One Million Hectares of High-Quality and Low-Emission Rice Program” deploying over 1,035.3 hectares in the Mekong Delta region. Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024). This contributes to the project “Sustainable development of one million hectares of high-quality, low-emission rice cultivation associated with green growth in the Mekong"}\n{"id": "B-A-01d2bc805d46ae2a", "claim": "M3/YEAR At the Aquatex Bentre pangasius processing plant, we have improved and upgraded the wastewater treatment system to meet Class A standards, QCVN 11-MT:2015/ BTNMT.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-177307483eea44eb", "claim": "The treated wastewater is reused for factory cleaning, plant irrigation, and fire fighting.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 Efficient water use initiatives at processing and production plants At the Ca Na - Phan Ri branch of 584 Nha Trang company, water used to clean the salt separator is reused as the source water for mixing brine for the brine rods in the traditional fish sauce production process. This helps save approximately M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in"}\n{"id": "B-A-95fabbbaf5c8c266", "claim": "At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for", "evidence": "The basic process is as follows: POND WASTEWATER M3/YEAR At the BioSpring probiotics production plant, wastewater from the RO filtration system accounts for 70–80% of the input water in the production process. The wastewater after the RO filtration system is considered clean and reusable. The Engineering and Production departments collaborated to recover this water into the domestic water tank for use in domestic purposes, irrigation, sanitation, and firefighting. This solution helps reduce freshwater intake while also lowering wastewater treatment costs, saving input water approximately"}\n{"id": "B-A-d21068f5e098adea", "claim": "At the Aquatex Bentre pangasius processing plant, we have improved and upgraded the wastewater treatment system to meet Class A standards, QCVN 11-MT:2015/", "evidence": "M3/YEAR At the Aquatex Bentre pangasius processing plant, we have improved and upgraded the wastewater treatment system to meet Class A standards, QCVN 11-MT:2015/ BTNMT. The treated wastewater is reused for factory cleaning, plant irrigation, and fire fighting. The volume is approximately 50 m3/day, equivalent to Treated wastewater must meet strict quality standards, ensuring no pollution of water sources, protecting ecosystems, and safeguarding public health. We commit to transparency in disclosing wastewater quality information and are ready to cooperate with regulatory authorities for monitoring and evaluation. OF FACTORIES AND PROCESSING FACILITIES have wastewater treatment systems that"}\n{"id": "B-A-3f9ebe8990eb5ab8", "claim": "Treated wastewater must meet strict quality standards, ensuring no pollution of water sources, protecting ecosystems, and safeguarding public health.", "evidence": "TABLE Greenhouse gas emissions Green energy Water resources and quality Responsible usage Sustainable raw material Sustainable waste management Wastewater and waste control Reforestation Protection of aquaculture ecosystems Responsible farming Soil regeneration Ï Ensure products meet food safety Ï Promote sustainable agricultural production and food processing practices, protecting soil, water resources, and biodiversity."}\n{"id": "B-A-ef666e7db912bc75", "claim": "We commit to transparency in disclosing wastewater quality information and are ready to cooperate with regulatory authorities for monitoring and evaluation.", "evidence": "payment obligations in accordance with regulations. Tax risk and dispute management Ï Ensuring transparency in documentation, records, and tax-related information when working with regulatory authorities. The Group places strong emphasis on tax risk management by strengthening internal controls and ensuring compliance with prevailing legal regulations. All activities are conducted in accordance with approved procedures and standards. We work closely with relevant authorities to monitor and assess compliance with environmental regulations. Ï Standardized Waste Collection and Storage Facilities"}\n{"id": "B-A-86572fb8c0817743", "claim": "BTNMT (Column A) Rice QCVN 40:2011/ BTNMT (Column B) fish not used in the production process, only domestic wastewater is generated.", "evidence": "QCVN 11-MT-2015/BTNMT for seafood processing plants and Column B standards under QCVN 40:2011/ BTNMT for food processing plants. Total P mg/L 0.035 0.39 5.5 10 - Total N mg/L 6.5 Small habit changes Indicator Unit Pangasius processing Shrimp processing Fish sauce production Rice production QCVN 19:2009/ BTNMT (Column B) PAN Group launched a response to the “Earth Hour"}\n{"id": "B-A-a37d7787476a6635", "claim": "Domestic wastewater is mainly treated using biological methods (anaerobic and aerobic) to decompose organic matter, combined with mechanical steps (screening, sedimentation).", "evidence": "Ï For food processing plants, wastewater treatment Wastewater monitoring results at certain factories of the PAN Group technology applies a combination of anaerobic, anoxic, and aerobic biological processes to treat pollutant components. Ï For seafood processing plants, wastewater treatment technology applies anaerobic biological processes combined with phosphorus and chlorine treatment, as well as sludge treatment."}\n{"id": "B-A-54108cd230370870", "claim": "(*) : The fish sauce factory’s treated wastewater meets Column B of QCVN 11:2015/BTNMT.", "evidence": "BTNMT (Column A) Rice QCVN 40:2011/ BTNMT (Column B) fish not used in the production process, only domestic wastewater is generated. Domestic wastewater is mainly treated using biological methods (anaerobic and aerobic) to decompose organic matter, combined with mechanical steps (screening, sedimentation). TSS mg/L 8.64 - < 5 100 28 100 BOD5 mg/L M3/YEAR At the Aquatex Bentre pangasius processing plant, we have improved and upgraded the wastewater treatment system to meet Class A standards, QCVN 11-MT:2015/ BTNMT. The treated wastewater is reused for factory cleaning, plant irrigation, and fire fighting. The volume is approximately 50 m3/day, equivalent to"}\n{"id": "B-A-273fbfbc78072e65", "claim": "Accordingly, the PAN Group focuses on advancing the circular economy across all aspects by developing human resources, enhancing innovation, upgrading production processes for optimal efficiency and environmental friendliness, collaborating with relevant partners to drive sustainable packaging innovation, and strengthening the role and engagement of farmers and communities within the circular economy.", "evidence": "The Group focuses on developing a transparent, stable, and sustainable supply chain by strengthening collaboration with farmers, cooperatives, and local partners. Production linkage programs, offtake agreements, and technical support initiatives continue to be implemented, contributing to improved raw material quality, stable market access, and increased income for producers within the agriculture–food value chain. Promoting a green economy Job creation and income improvement for communities Ï Foster a corporate culture that PAN prioritizes environmental protection in all production and business activities, aiming to minimize adverse impacts and promote a green, circular economy model. Commu"}\n{"id": "B-A-41c960318b0842c5", "claim": "Of this, 85% of the waste consists of seafood and food by-products that will be reused.", "evidence": "Ï Encouraging the reuse, recycling, and/or upcycling of 0.2 0.0 0 by-products and waste materials. 2024 94.8% 1.2% 5.4% 2.3% 92.2% 2023 0.1% Recyclable waste Non-recyclable waste Domestic waste Hazardous waste TONS of sludge from the pangasius processing plant will be used to raise earthworms."}\n{"id": "B-A-3e6a0e7b22395c9f", "claim": "Waste Management performance data from 2023-2025 2.6% 0.5% 2025 96.8% 0.2% TONS 1.1% 2.9% of rice husks and various types of nut shells (cashew, walnut) will be reused as fuel in 2025.", "evidence": "Waste is categorized into recyclable waste (paper, plastic, metal, production by-products, and agricultural residues), domestic/municipal waste, other general waste, and hazardous waste. Nut Shells (Walnut, Cashew, Rice Husk) Tons cashew shells and walnut shells are used as biomass fuel for boilers, while shells and coffee grounds are recycled into value-added products. These solutions help optimize resource use and reduce waste generation. For confectionery manufacturing plants, key raw materials—including wheat flour, sugar, and eggs—are sourced from domestic suppliers and comply with food safety standards such as ISO 22000."}\n{"id": "B-A-10780d24f83c8ec6", "claim": "2024 94.8% 1.2% 5.4% 2.3% 92.2% 2023 0.1% Recyclable waste Non-recyclable waste Domestic waste Hazardous waste TONS of sludge from the pangasius processing plant will be used to raise earthworms.", "evidence": "Waste is categorized into recyclable waste (paper, plastic, metal, production by-products, and agricultural residues), domestic/municipal waste, other general waste, and hazardous waste. Nut Shells (Walnut, Cashew, Rice Husk) Tons (straw, rice stalks, etc.) Tons 85,964 Fertilizer production, soil nutrient restoration Recyclable discarded packaging Tons 1,242 Repackaging of raw materials Waste sludge"}\n{"id": "B-A-039aed9ea4db5dfb", "claim": "Waste generation volume per unit of revenue 30,000 1.8 1.6 1.6 25,000 1.4 The PAN Group prioritizes and promotes waste management across the entire value chain.", "evidence": "Ï Waste generation per VND billion revenue) Ï 96.8% waste recycling rate across the Group 60 61 1.4 1.3 1.2 20,000 1.0 15,000 26,294 0.8 Ï Promoting waste reduction at the source and 23,811 0.6 10,000 minimizing the use of input resources."}\n{"id": "B-A-daa3021a3564ad56", "claim": "Total solid waste (tons) Total solid waste per revenue (tons/billion VND)", "evidence": "In 2024, 93.9% of total waste generated across the Group was recycled. Of these, 76.6% of aquaculture by- products were repurposed into new products. CO2 emission intensity (tons/billion VND revenue) specialized 昀氀ooded melaleuca forest Ï Waste generation per VND billion revenue) Ï 96.8% waste recycling rate across the Group 60 61"}\n{"id": "B-A-dc1265773e521799", "claim": "Equipment, systems, and facilities for storing hazardous waste, general industrial solid waste, and municipal solid waste are fully compliant with the requirements specified in Circular No.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. Ï Product technical specifications Ï Equipment and machinery Ï Product quality Ï General requirements Ï Facilities and factories Ï Product traceability"}\n{"id": "B-A-529a541ae700270b", "claim": "Companies and factories within the PAN Group have established close partnerships with reputable waste collection and treatment service providers.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. In the coming time, in parallel with cooperating with strategic partners, the Group continues to invest in research to completely close the value chain, speci昀椀cally in the 昀椀ngerlings and aquafeed segments. Vietnam has great advantages in the agriculture and food sector; however, domestic consumers still do not have widesprea"}\n{"id": "B-A-bde7f5cbf89b34be", "claim": "We conduct thorough evaluations and screenings to select units with the necessary capacity and experience, responsible for the safe and efficient collection, transportation, and treatment of municipal waste, industrial waste, and hazardous waste.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. Equipment, systems, and facilities for storing hazardous waste, general industrial solid waste, and municipal solid waste are fully compliant with the requirements specified in Circular No. 02/2022/TT-BTNMT dated January 10, 2022. Ï Collaboration with Competent Waste Management Units"}\n{"id": "B-A-e3bd53e016f04a91", "claim": "In 2025, no legal violations were recorded among PAN Group members, and there were no waste-related incidents causing environmental pollution.", "evidence": "ANNUAL REPORT 2024 CHAPTER 1 GENERAL INFORMATION OF THE PAN GROUP B. ENVIRONMENTAL AND SOCIAL RISKS 3 Packaged food sector Pollution risks and environmental incidents Ï Committed to zero legal violations accidents."}\n{"id": "B-A-fb5510c49609917a", "claim": "Details of recyclable waste types in 2025", "evidence": "2024 94.8% 1.2% 5.4% 2.3% 92.2% 2023 0.1% Recyclable waste Non-recyclable waste Domestic waste Hazardous waste TONS of sludge from the pangasius processing plant will be used to raise earthworms. Waste is categorized into recyclable waste (paper, plastic, metal, production by-products, and agricultural residues), domestic/municipal waste, other general waste, and hazardous waste. Nut Shells (Walnut, Cashew, Rice Husk) Tons"}\n{"id": "B-A-0a8e4692a6149fab", "claim": "These activities have significantly enhanced environmental awareness, contributing to the development of a green corporate culture.", "evidence": "Ï Foster a corporate culture that PAN prioritizes environmental protection in all production and business activities, aiming to minimize adverse impacts and promote a green, circular economy model. Community engagement is a key component of PAN’s sustainable development strategy. The Group not only contributes financially but also directly participates in community support programs, helping improve the lives of employees, farmers, and local residents. PAN is progressively integrating environmental and sustainability factors into its investment and production activities, aiming to minimize environmental impacts and promote green growth. Through efficient resource utilization, emissions reduct"}\n{"id": "B-A-7394084d01eb4ac1", "claim": "Units operating in the food sector, including Bibica, Lafooco, Khang An Foods, Aquatex Bentre, 584 Nha Trang, and Vinaseed, strictly comply with environmental management requirements under recognized food safety management systems such as HACCP and BRCGS.", "evidence": "PAN-HULIC Bibica Lafooco 584 Nha Trang SHIN Coffee Fimex VN Khang An Foods Aquatex Bentre Scope Total Number of Units Collecting Data 11 Implemented by four subsidiaries: Lafooco, Khang An Foods, Aquatex Bentre, Fimex VN 47.8% Over 50 598 Furthermore, our business units strictly adhere to the Codes of Conduct (CoC) required by high-"}\n{"id": "B-A-b9885f0f00b1739c", "claim": "Recycled paper Tons 388,000 Materials for industrial production", "evidence": "Recycled plastic Tons 203,673 Materials for industrial production Recycled metal Tons 73,696 Materials for industrial production Ï Waste Segregation at Source All units within the PAN Group effectively implement waste segregation at the source. TABLE Tons | 388,000 | Materials for industrial production Tons | 203,673 | Materials for industrial production Tons | 73,696 | Materials for industrial production Tons | 2,229,331 | Fuel, industrial material production Tons | 11,043,673 | Chitin, Chitosan, Glucosamine Animal feed Tons | 8,315,485 | Animal feed Tons | 333,200 | Animal feed Tons | 256,010 | Animal feed Tons | 85,964 | Fertilizer production, soil nutrient restoration Tons | 1,242 | Repa"}\n{"id": "B-A-31124ecf259de4ed", "claim": "Waste is categorized into recyclable waste (paper, plastic, metal, production by-products, and agricultural residues), domestic/municipal waste, other general waste, and hazardous waste.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. 2024 94.8% 1.2% 5.4% 2.3% 92.2% 2023 0.1% Recyclable waste Non-recyclable waste Domestic waste Hazardous waste TONS of sludge from the pangasius processing plant will be used to raise earthworms."}\n{"id": "B-A-cac212c1a1acae80", "claim": "Depending on their business sector, each unit currently implements different recycling approaches.", "evidence": "216,294,580 216,294,580 The Group chooses business segments as the main reporting ones because the Group’s risks and pro昀椀tability are primarily impacted by the differences in products and services provided by the Group. Segments by geography are secondary reporting ones. The Group’s business activities are organized and managed according to the nature of the products and services provided by the Group with each division being a strategic business unit that provides different products and serves different markets. Enhancing decision-making and planning capabilities based on a thorough understanding of the business environment, opportunities, and challenges faced by the Group. LEVEL 3: EXTERN"}\n{"id": "B-A-bcb20fe3dd2892ed", "claim": "The units have gradually eliminated single-use plastic products, such as straws, plastic cups, and plastic bags, from their production processes and company activities.", "evidence": "Specifically, products such as Purple-black brown rice, Red brown rice, Red rice Phuc Tho, and ST25 ecological rice have switched to paper packaging and paper boxes, using coir rope handles instead of plastic bags. Currently, the weight of environmentally friendly packaging for these rice products accounts for approximately 8% of the total packaging weight across all product types in 2024. Recycled plastic Tons 203,673 Materials for industrial production Recycled metal Tons 73,696 Materials for industrial production Ï Waste Segregation at Source All units within the PAN Group effectively implement waste segregation at the source."}\n{"id": "B-A-1e1f507fdbeb57ce", "claim": "At the same time, the transition to environmentally friendly alternatives, such as glass cups and fabric bags, has helped these units build a positive image in the eyes of customers and partners.", "evidence": "Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions. At the same time, Bibica has implemented numerous packaging innovations to reduce weight, optimize packing, and use environmentally friendly packaging. For example, in 2025, Bibica eliminated the use of metal rivets on paper shopping bags, saving approximately VND Ï Encourage the use of environmentally friendly materials, reduce plastic waste, and promote circular economy practices."}\n{"id": "B-A-c604d4701976a722", "claim": "Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions.", "evidence": "Vietnam Environmental Protection Fund. In 2024, the company fulfilled this obligation with a contribution of over VND 600 million. Vinaseed fulfills its recycling responsibility through its subsidiary, Vietnam Agrifood Investment and Import- Export Company (Vina Agrifood). The company has opted for a combination of financial contributions to the Ï Regarding Recycling Responsibility Food products, such as Bibica’s confectionery or Vinaseed’s processed rice, are also subject to recycling responsibility. Depending on their business sector, each unit currently implements different recycling approaches."}\n{"id": "B-A-e98ec3c599229767", "claim": "In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly packaging groups and innovated the method for printing production and expiration dates, using laser printers instead of ink to minimize solvent emissions into the environment.", "evidence": "Specifically, products such as Purple-black brown rice, Red brown rice, Red rice Phuc Tho, and ST25 ecological rice have switched to paper packaging and paper boxes, using coir rope handles instead of plastic bags. Currently, the weight of environmentally friendly packaging for these rice products accounts for approximately 8% of the total packaging weight across all product types in 2024. Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions. At the same time, Bibica has implemented numerous packaging innovations to reduce weight, optimize packing, and use environmentally friendly pa"}\n{"id": "B-A-38e391e0427cde29", "claim": "KG EPR Opens the Future for Green Recycling and Promotes a Circular Economy Project “Cosmetic Product Line (Soap, Scrub) from coffee by-products”", "evidence": "PAN promotes a circular economy model by utilizing by- products generated during production. Approximately 10% of shells are reused or recycled; for example, Ï Encouraging the reuse, recycling, and/or upcycling of 0.2 0.0 0 by-products and waste materials."}\n{"id": "B-A-ee8034f8c4bc033f", "claim": "This initiative contributes to waste reduction and minimizes the “ecological footprint” on the environment.", "evidence": "To ful昀椀ll its vision and mission, the Group not only prioritizes economic growth but also places a strong emphasis on social responsibility and environmental protection. Through concrete actions, PAN actively minimizes the environmental footprint while implementing programs that bene昀椀t the community, employees, and stakeholders. ENVIRONMENT 35.8% 2024 marked a signi昀椀cant milestone in The PAN Group’s sustainability journey. PAN is progressively integrating environmental and sustainability factors into its investment and production activities, aiming to minimize environmental impacts and promote green growth. Through efficient resource utilization, emissions reduction, and participation in "}\n{"id": "B-A-185260a1c76411ed", "claim": "Group ecosystem have proactively implemented solutions to meet EPR requirements.", "evidence": "We take responsibility for our impact, acting voluntarily and sincerely to uphold PAN’s corporate social responsibility commitments. We maximize value for all stakeholders within our ecosystem and proactively seek feedback to drive continuous improvement. 77% achieve greater goals and deliver 69% 70 68% 65% embrace change in our ecosystem and proactively seek 60 64% operations and outstanding as a catalyst for for our employees,"}\n{"id": "B-A-917b7520a2ab114d", "claim": "In addition, Bibica has replaced PS material liners (environmentally unfriendly) with recycled PET material, with an approximate packaging weight of 14 tons.", "evidence": "Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions. At the same time, Bibica has implemented numerous packaging innovations to reduce weight, optimize packing, and use environmentally friendly packaging. For example, in 2025, Bibica eliminated the use of metal rivets on paper shopping bags, saving approximately VND 1.3 billion in packaging costs and reducing the amount of aluminum packaging waste released into the market by around 7.3 tons. In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly pack"}\n{"id": "B-A-0662cf5cacf24f98", "claim": "Plastic trays have also been replaced with recyclable paper trays to improve environmental friendliness, with an estimated weight of around 23 tons.", "evidence": "Group ecosystem have proactively implemented solutions to meet EPR requirements. In addition, Bibica has replaced PS material liners (environmentally unfriendly) with recycled PET material, with an approximate packaging weight of 14 tons. Specifically, products such as Purple-black brown rice, Red brown rice, Red rice Phuc Tho, and ST25 ecological rice have switched to paper packaging and paper boxes, using coir rope handles instead of plastic bags. Currently, the weight of environmentally friendly packaging for these rice products accounts for approximately 8% of the total packaging weight across all product types in 2024."}\n{"id": "B-A-61c80f7091d34381", "claim": "Vinaseed fulfills its recycling responsibility through its subsidiary, Vietnam Agrifood Investment and Import- Export Company (Vina Agrifood).", "evidence": "Hanoi 99.96% 99.96% within the within the Vietnam National Agrifood Export Import and Investment Company Limited group group Dong Thap 100.00% Ï Regarding Recycling Responsibility Food products, such as Bibica’s confectionery or Vinaseed’s processed rice, are also subject to recycling responsibility. Depending on their business sector, each unit currently implements different recycling approaches."}\n{"id": "B-A-5227162661a10dfd", "claim": "Vietnam Environmental Protection Fund and transitioning its product packaging toward environmentally friendly materials.", "evidence": "1.3 billion in packaging costs and reducing the amount of aluminum packaging waste released into the market by around 7.3 tons. In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly packaging groups and innovated the method for printing production and expiration dates, using laser printers instead of ink to minimize solvent emissions into the environment. Ï Encourage the use of environmentally friendly materials, reduce plastic waste, and promote circular economy practices."}\n{"id": "B-A-7f891d0b52015cbd", "claim": "Khang An Foods’ recycling fund program has been conducted over four seasons since 2022, aiming to raise employees’ awareness of environmental protection and encourage the practice of sorting and recycling plastic waste.", "evidence": "Waste Prevention Team to monitor and maintain the “Right and Enough” culture, helping to raise awareness and responsibility among employees in practicing savings and using resources efficiently. Responsibilities are divided into four levels, from employees to the Ï Encourage the use of environmentally friendly materials, reduce plastic waste, and promote circular economy practices."}\n{"id": "B-A-ad49b8bef95ba0d0", "claim": "Notably, the company avoids using glass and metal packaging, which helps lower disposal costs and ensures safety for farmers during use and disposal.", "evidence": "Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions. At the same time, Bibica has implemented numerous packaging innovations to reduce weight, optimize packing, and use environmentally friendly packaging. For example, in 2025, Bibica eliminated the use of metal rivets on paper shopping bags, saving approximately VND 1.3 billion in packaging costs and reducing the amount of aluminum packaging waste released into the market by around 7.3 tons. In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly pack"}\n{"id": "B-A-6a092a8b4d8078e2", "claim": "Currently, the weight of environmentally friendly packaging for these rice products accounts for approximately 8% of the total packaging weight across all product types in 2024.", "evidence": "1.3 billion in packaging costs and reducing the amount of aluminum packaging waste released into the market by around 7.3 tons. In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly packaging groups and innovated the method for printing production and expiration dates, using laser printers instead of ink to minimize solvent emissions into the environment. Bibica has chosen to collaborate with authorized recycling partners to fulfill this responsibility, which helps optimize costs compared to financial contributions. At the same time, Bibica has implemented numerous packaging innovations to reduce weight, opt"}\n{"id": "B-A-7be7ccdd28ee7423", "claim": "In 2025, the amount of plastic waste collected and recycled reached KG 118 119", "evidence": "1.3 billion in packaging costs and reducing the amount of aluminum packaging waste released into the market by around 7.3 tons. In 2025, the amount of recycled sludge at the plant reached Regarding paper packaging, the company has shifted to environmentally friendly packaging groups and innovated the method for printing production and expiration dates, using laser printers instead of ink to minimize solvent emissions into the environment. Recycled plastic Tons 203,673 Materials for industrial production Recycled metal Tons 73,696 Materials for industrial production Ï Waste Segregation at Source All units within the PAN Group effectively implement waste segregation at the source."}\n{"id": "B-A-4ed7100db3c610be", "claim": "Acknowledging this, the PAN Group approaches chemical management through proactive risk governance, closely integrated within the food safety management system and quality control across the entire value chain.", "evidence": "ENHANCING FINANCIAL MANAGEMENT, OPERATIONS, AND RISK MANAGEMENT SYSTEMS Improving the governance model: The PAN Group applies international standards for parent-subsidiary company governance, ensuring resource optimization and creating shared value for the entire system. SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE RISK MANAGEMENT Risk Management Process Risk Management Policies and Procedures PAN Group has established a multi-layered risk management system structured vertically, spanning from operational units to the executive level of member companies and up to the management level of PAN Group."}\n{"id": "B-A-9ccd9b6c94de5e5b", "claim": "mechanical–biological measures combined with low-odor, low-residue chemicals, safe for customers and employees.", "evidence": "United States, and Japan. Chemicals used for facility Chemical Safety Management Practices at the Plant: Ï Fully equip employees with appropriate personal protective equipment (PPE) to ensure safety and prevent incidents. Ï Chemicals/gases for disinfection are used according to international standards, with risk control procedures and full protective equipment provided for employees. 120 121"}\n{"id": "B-A-6c34d2f2e9033b64", "claim": "The food safety management systems at member units are established in accordance with international standards such as HACCP, ISO 22000, and BRC (depending on the sector), with a key focus on controlling chemical residues and preventing cross-contamination.", "evidence": "TABLE HACCP, BRC, IFS, HALAL | BRC, HACCP, HALAL, IFS | FSSC 22000, HACCP | FSSC 22000, ISO 22000 | BRCGS, HACCP | FSSC 22000, HALAL | FSSC 22000 TCVN 5107-2018 | QCVN 01- 188:2018/ BNNPTNT | ISO 22000 ISO 17025 | | | | | | | ISO 17025 | ISO 17025 Ï Certification of Food Safety Management System in accordance with international standards"}\n{"id": "B-A-7c9b6a3b0985b777", "claim": "Procedures for the storage, preparation, use, and disposal of chemicals are strictly controlled, and employees receive regular training on hazard identification and safe handling practices.", "evidence": "Ï Issue internal regulations, guidelines, and instructions on the safe use, storage, and disposal of chemicals. Disinfection Services for Warehouses, Containers, and Exported Agricultural Products: Ï The operational process is safe and strictly complies with the regulations of quarantine and customs authorities, as well as the requirements of export customers. Ï Develop incident prevention plans. Ï Conduct regular health check-ups for employees twice Specifically, at VFC – a PAN Group member engaged in the business and production of plant protection and disinfectant products – a rigorous chemical management system has been established, strictly complying with chemical safety and environmenta"}\n{"id": "B-A-15e4e92533c41cc2", "claim": "From warehouse receipt, storage, and usage to hazardous waste disposal, all stages are closely monitored using the ePACIFIC software.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Waste Management The PAN Group is committed to strictly complying with environmental protection regulations. We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste. All activities are conducted in accordance with approved procedures and standards. We work closely with relevant authorities to monitor and assess compliance with environmental regulations. Ï Standardized Waste Collection and Storage Facilities"}\n{"id": "B-A-655c5de9a9cf3a90", "claim": "In addition, VFC regularly conducts training sessions to raise chemical safety awareness among all employees, ensuring everyone fully understands the risks and preventive measures.", "evidence": "Ï The Group conducts regular assessments and implements preventive measures to address potential risks in the workplace. United States, and Japan. Chemicals used for facility Chemical Safety Management Practices at the Plant: Ï Fully equip employees with appropriate personal protective equipment (PPE) to ensure safety and prevent incidents."}\n{"id": "B-A-2e4c3115a42b1f4f", "claim": "The goal is to create an environmentally friendly agricultural system that maintains ecological balance.", "evidence": "Environment Social Governance PAN is working towards an environmentally friendly production model that efficiently utilizes natural resources and minimizes negative impacts on ecosystems. Ï Enhance the quality of life and Ï Strengthen transparency in corporate management, complying with governance and sustainability standards. Ï Encourage the use of environmentally friendly materials, reduce plastic waste, and promote circular economy practices."}\n{"id": "B-A-04359cd7b193148e", "claim": "To realize this objective, PAN Group has proactively integrated biodiversity protection solutions into its business operations.", "evidence": "PAN expands cooperation with capable partners in international market development to promote exports and build a global brand. THE PAN GROUP\'S KEY OBJECTIVES To realize the sustainable development strategy, The PAN Group focuses on the following core objectives: SUSTAINABLE DEVELOPMENT IN THREE CORE BUSINESS AREAS Bibica has proactively embraced advanced technologies by strongly applying artificial intelligence (AI) and Industry 4.0 solutions in both business operations and internal management. From advertising and data analytics to process automation, technology is becoming a key driver for accelerating growth and enhancing the company’s competitiveness."}\n{"id": "B-A-a99287aa86daf253", "claim": "We are committed to transforming our business model to harmonize with nature, contributing to the conservation and restoration of biodiversity, while ensuring sustainable development for both the company and the community.", "evidence": "ANNUAL REPORT 2024 CHAPTER 2 2024 BUSINESS RESULTS ENVIRONMENTAL & SOCIAL RESPONSIBILITY REPORT The PAN Group is committed to sustainable development, pursuing strategies that deliver long-term value for both the business and the community. Biodiversity Risk Assessment For each aquaculture pond construction project (shrimp, pangasius), we conduct surveys of geology, geography, and baseline environmental conditions as a basis for assessing the level of impact of production activities on surrounding communities. Environmental Protection Requirements Conservation and Restoration of Biodiversity in"}\n{"id": "B-A-79bd728df13e1d4f", "claim": "We have fully implemented responsible aquaculture practices, including stringent requirements on environmental protection, social responsibility, animal welfare, and food safety.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Animal Welfare Product Responsibility Indicators 2025 Increasingly stringent animal welfare requirements are driving changes across farming, harvesting, and processing practices. Ï It applies stringent food safety management standards, including ISO, FSSC, BRCGS, Ï Substandard raw materials leading to food safety incidents. Product quality, food safety, and community responsibility risks"}\n{"id": "B-A-74289d62853d546a", "claim": "Biodiversity Risk Assessment For each aquaculture pond construction project (shrimp, pangasius), we conduct surveys of geology, geography, and baseline environmental conditions as a basis for assessing the level of impact of production activities on surrounding communities.", "evidence": "Ï Water quality management: Ensure that wastewater from aquaculture ponds, after treatment, meets regulatory standards before being discharged into the environment and does not cause pollution. Ï Soil environmental protection: All shrimp ponds are of aquaculture ponds prepare Environmental Impact Assessment reports on biodiversity and on plant and animal species that need to be protected 14001, ASC, and BAP to closely control the environmental impacts of production activities. ENVIRONMENT Environmental Protection and Resource Conservation PAN Group is committed to creating the greatest possible value for employees, customers, partners, suppliers, and surrounding communities."}\n{"id": "B-A-471466d605b225a5", "claim": "Environmental Protection Requirements Conservation and Restoration of Biodiversity in", "evidence": "To realize this objective, PAN Group has proactively integrated biodiversity protection solutions into its business operations. We are committed to transforming our business model to harmonize with nature, contributing to the conservation and restoration of biodiversity, while ensuring sustainable development for both the company and the community. Stewardship Council (ASC) certification (Vita Farm, 14001, ASC, and BAP to closely control the environmental impacts of production activities. ENVIRONMENT Environmental Protection and Resource Conservation PAN Group is committed to creating the greatest possible value for employees, customers, partners, suppliers, and surrounding communities."}\n{"id": "B-A-3a63dfbca5c4b980", "claim": "In addition, native fish are cultured in drainage canals to absorb nutrients and organic matter from wastewater discharged from shrimp ponds.", "evidence": "Ï Water quality management: Ensure that wastewater from aquaculture ponds, after treatment, meets regulatory standards before being discharged into the environment and does not cause pollution. Ï Soil environmental protection: All shrimp ponds are of aquaculture ponds prepare Environmental Impact Assessment reports on biodiversity and on plant and animal species that need to be protected This helps save approximately Ï Aquaculture farms (shrimp, pangasius) We apply responsible aquaculture management solutions in the use of water for shrimp and pangasius farming, taking advantage of natural water sources from the sea and rivers to create a balanced and sustainable shrimp and pangasius farming"}\n{"id": "B-A-e72ee6d4acd64e29", "claim": "Increasing tree coverage also contributes to carbon sequestration—one of the main drivers of global warming—and helps mitigate climate change.", "evidence": "IFRS Sustainability Disclosure Standards (IFRS S1 and IFRS S2). The framework reflects PAN’s systematic approach to identifying, managing, and mitigating material risks, while also capturing sustainability- related opportunities arising from climate change, evolving market requirements, and the increasing expectations of stakeholders across the agriculture– Vinaseed strengthened international partnerships and launched several new high-quality crop varieties to combat climate change. 21 20 THE PAN GROUP"}\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: INPUTS=gsi_layer_b2.jsonl python -u multirun.py jobs_32b.jso', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('INPUTS=gsi_layer_b2.jsonl python -u multirun.py jobs_32b.json issue_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)